# AutoQSAR Benchmark Summary

This notebook reads the output from `run_autoqsar_ga_benchmarks.py` and summarizes model performance across datasets.

By default it loads the most recent run under `./benchmark_results/`, then builds:
- a model-performance summary across datasets
- a leaderboard-relative summary using cached top-10 reference artifacts from the run directory
- diagnostic visuals for model spread, predictions, and (if present) GA convergence
- feature-family selection representation analysis from selector outputs (coverage, enrichment, and drop/watchlist suggestions)



In [53]:
# @title 0. Choose a benchmark run { display-mode: "form" }
benchmark_run_dir = "AUTO"  # @param {type:"string"}
show_top_n_models = 10  # @param {type:"slider", min:3, max:20, step:1}
rmse_metric_column_preference = "AUTO"  # @param {type:"string"}
prediction_sample_cap = 5000  # @param {type:"integer"}


In [54]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


def display_note(text):
    display(Markdown(text))


def resolve_benchmark_dir(path_text="AUTO"):
    path_text = str(path_text).strip()
    if path_text and path_text.upper() != "AUTO":
        candidate = Path(path_text).expanduser().resolve()
        if not candidate.exists():
            raise FileNotFoundError(f"Benchmark directory not found: {candidate}")
        return candidate

    root = Path("./benchmark_results").resolve()
    if not root.exists():
        for parent in Path(".").resolve().parents:
            candidate = parent / "benchmark_results"
            if candidate.exists():
                root = candidate
                break
        else:
            raise FileNotFoundError("No ./benchmark_results directory exists yet.")
    candidates = sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError("No benchmark run directories were found under ./benchmark_results.")
    return candidates[0]


def read_optional_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return None


def read_optional_json(path):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return None


def pick_column(columns, candidates):
    cols = list(columns)
    for candidate in candidates:
        if candidate in cols:
            return candidate
    lowered = {str(col).strip().lower(): col for col in cols}
    for candidate in candidates:
        match = lowered.get(str(candidate).strip().lower())
        if match is not None:
            return match
    return None


def metric_column(metric_df, preferred="AUTO", metric_hint="rmse"):
    if metric_df is None or metric_df.empty:
        return None
    if str(preferred).strip() and str(preferred).strip().upper() != "AUTO":
        if preferred not in metric_df.columns:
            raise ValueError(f"Requested metric column not found: {preferred}")
        return preferred
    candidates = []
    hint = str(metric_hint).strip().lower()
    if hint == "rmse":
        candidates = [
            "test_rmse", "Test RMSE", "cv_rmse", "CV RMSE", "mean_test_rmse",
            "rmse", "RMSE"
        ]
    elif hint == "r2":
        candidates = ["test_r2", "Test R2", "cv_r2", "CV R2", "r2", "R2"]
    return pick_column(metric_df.columns, candidates)


CLASSIFICATION_METRIC_CANDIDATES = {
    "roc_auc": ["test_roc_auc", "Test ROC AUC", "roc_auc", "ROC AUC", "AUROC", "test_auroc", "cv_roc_auc", "CV ROC AUC"],
    "auprc": ["test_auprc", "Test AUPRC", "AUPRC", "auprc", "cv_auprc", "CV AUPRC", "average_precision"],
    "balanced_accuracy": ["test_balanced_accuracy", "Test Balanced Accuracy", "balanced_accuracy", "Balanced Accuracy", "cv_balanced_accuracy", "CV Balanced Accuracy"],
    "mcc": ["test_mcc", "MCC", "mcc", "cv_mcc"],
}

# TDC CYP benchmark tasks whose leaderboard-primary metric is AUPRC. Keep these
# row-specific so other classification datasets can continue to use AUROC or the
# metric declared in their benchmark metadata.
TDC_AUPRC_PRIMARY_DATASETS = {
    "tdc_cyp2c9_veith",
    "tdc_cyp2d6_veith",
    "tdc_cyp3a4_veith",
    "tdc_cyp2c9_substrate_carbonmangels",
    "tdc_cyp2d6_substrate_carbonmangels",
    "tdc_cyp3a4_substrate_carbonmangels",
}


def classification_metric_column(metric_df, preferred="AUTO", metric_kind=None):
    if metric_df is None or metric_df.empty:
        return None
    if str(preferred).strip() and str(preferred).strip().upper() != "AUTO":
        if preferred not in metric_df.columns:
            raise ValueError(f"Requested classification metric column not found: {preferred}")
        return preferred

    if metric_kind is not None:
        candidates = CLASSIFICATION_METRIC_CANDIDATES.get(str(metric_kind).strip().lower())
        if candidates:
            return pick_column(metric_df.columns, candidates)

    ordered_candidates = []
    for kind in ["roc_auc", "auprc", "balanced_accuracy", "mcc"]:
        ordered_candidates.extend(CLASSIFICATION_METRIC_CANDIDATES[kind])
    return pick_column(metric_df.columns, ordered_candidates)


def infer_classification_metric_kind(dataset_name="", primary_metric=""):
    dataset_key = str(dataset_name or "").strip().lower()
    metric_text = str(primary_metric or "").strip().lower()

    if dataset_key in TDC_AUPRC_PRIMARY_DATASETS:
        return "auprc"
    if "auprc" in metric_text or "pr_auc" in metric_text or "average_precision" in metric_text:
        return "auprc"
    if "balanced" in metric_text:
        return "balanced_accuracy"
    if "mcc" in metric_text or "matthews" in metric_text:
        return "mcc"
    if "roc" in metric_text or "auroc" in metric_text or metric_text == "auc":
        return "roc_auc"
    return "roc_auc"


def classification_metric_column_for_row(row, columns, fallback_col=None):
    metric_kind = infer_classification_metric_kind(row.get("dataset", ""), row.get("primary_metric", ""))
    selected = pick_column(columns, CLASSIFICATION_METRIC_CANDIDATES.get(metric_kind, []))
    if selected is not None:
        return selected, metric_kind
    if fallback_col in columns:
        return fallback_col, infer_classification_metric_kind("", fallback_col)
    fallback = classification_metric_column(pd.DataFrame(columns=columns))
    return fallback, infer_classification_metric_kind("", fallback)


def metric_is_lower_better(metric_name):
    text = str(metric_name or "").strip().lower()
    if not text:
        return True
    if any(key in text for key in ["rmse", "mae", "mse", "loss", "error"]):
        return True
    if any(key in text for key in ["balanced_accuracy", "accuracy", "roc_auc", "auroc", "auc", "auprc", "mcc", "f1", "r2", "spearman", "pearson"]):
        return False
    return True


def add_analysis_metric_columns(frame, rmse_col=None, classification_col=None):
    result = frame.copy()
    if result.empty:
        result["task_kind"] = []
        result["analysis_metric"] = []
        result["analysis_metric_kind"] = []
        result["analysis_metric_value"] = []
        result["analysis_metric_direction"] = []
        result["analysis_delta_from_best"] = []
        return result

    rmse_col = rmse_col if rmse_col in result.columns else metric_column(result, "AUTO", metric_hint="rmse")
    classification_col = classification_col if classification_col in result.columns else classification_metric_column(result)

    rmse_values = pd.to_numeric(result[rmse_col], errors="coerce") if rmse_col is not None else pd.Series(np.nan, index=result.index)
    class_probe_values = pd.Series(False, index=result.index)
    for candidates in CLASSIFICATION_METRIC_CANDIDATES.values():
        probe_col = pick_column(result.columns, candidates)
        if probe_col is not None:
            class_probe_values = class_probe_values | pd.to_numeric(result[probe_col], errors="coerce").notna()

    primary_text = result.get("primary_metric", pd.Series("", index=result.index)).fillna("").astype(str).str.lower()
    looks_classification = (
        primary_text.str.contains("auc|accuracy|mcc|f1|auprc|average_precision", regex=True)
        | class_probe_values
    )

    result["task_kind"] = np.where(rmse_values.notna() & ~looks_classification, "regression", np.where(looks_classification, "classification", "unknown"))
    result["analysis_metric"] = rmse_col or "rmse"
    result["analysis_metric_kind"] = "rmse"
    result["analysis_metric_value"] = rmse_values
    result["analysis_metric_direction"] = np.where(result["task_kind"].eq("classification"), "higher", "lower")

    class_mask = result["task_kind"].eq("classification")
    for row_index, row in result.loc[class_mask].iterrows():
        metric_col, metric_kind = classification_metric_column_for_row(row, result.columns, fallback_col=classification_col)
        result.loc[row_index, "analysis_metric"] = metric_col or "classification_metric"
        result.loc[row_index, "analysis_metric_kind"] = metric_kind
        if metric_col in result.columns:
            result.loc[row_index, "analysis_metric_value"] = pd.to_numeric(result.loc[row_index, metric_col], errors="coerce")
        else:
            result.loc[row_index, "analysis_metric_value"] = np.nan

    result["analysis_metric_value"] = pd.to_numeric(result["analysis_metric_value"], errors="coerce")
    result["analysis_delta_from_best"] = np.nan

    for dataset_name, idx in result.groupby("dataset", dropna=False).groups.items():
        local = result.loc[idx]
        values = pd.to_numeric(local["analysis_metric_value"], errors="coerce")
        valid = values.notna()
        if not valid.any():
            continue
        direction = str(local.loc[valid, "analysis_metric_direction"].mode().iloc[0])
        best_value = values[valid].min() if direction == "lower" else values[valid].max()
        if direction == "lower":
            result.loc[idx, "analysis_delta_from_best"] = values - best_value
        else:
            result.loc[idx, "analysis_delta_from_best"] = best_value - values

    return result


def best_rows_by_dataset(frame):
    if frame is None or frame.empty:
        return pd.DataFrame()
    work = frame.copy()
    if "analysis_metric_value" not in work.columns:
        work = add_analysis_metric_columns(work)
    work = work.loc[pd.to_numeric(work["analysis_metric_value"], errors="coerce").notna()].copy()
    if work.empty:
        return work
    work["_analysis_sort_value"] = np.where(
        work["analysis_metric_direction"].astype(str).eq("higher"),
        -pd.to_numeric(work["analysis_metric_value"], errors="coerce"),
        pd.to_numeric(work["analysis_metric_value"], errors="coerce"),
    )
    return (
        work.sort_values(["dataset", "_analysis_sort_value", "model"], ascending=[True, True, True])
        .groupby("dataset", as_index=False, dropna=False)
        .first()
        .drop(columns=["_analysis_sort_value"], errors="ignore")
    )


def standardize_metric_frame(metric_df):
    if metric_df is None:
        return None
    frame = metric_df.copy()
    rename_map = {}
    dataset_col = pick_column(frame.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(frame.columns, ["model", "Model"])
    workflow_col = pick_column(frame.columns, ["workflow", "Workflow"])
    if dataset_col is not None:
        rename_map[dataset_col] = "dataset"
    if model_col is not None:
        rename_map[model_col] = "model"
    if workflow_col is not None:
        rename_map[workflow_col] = "workflow"
    frame = frame.rename(columns=rename_map)
    return frame


def resolve_repo_file(relative_path):
    """Find a repo-relative file whether the notebook is run from repo root or bundle dir."""
    relative_path = Path(relative_path)
    for base in [Path(".").resolve(), *Path(".").resolve().parents]:
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    return relative_path


def normalize_reference_dataset_name(dataset_name, benchmark_suite="tdc"):
    text = str(dataset_name or "").strip()
    if not text:
        return ""
    lowered = text.lower()
    if lowered.startswith(("tdc_", "polaris_", "poduam_", "chemml_")):
        return lowered
    if lowered in {"esol_delaney", "freesolv_sampl", "lipophilicity"}:
        return lowered
    suite = str(benchmark_suite or "").strip().lower()
    if suite in {"tdc", "adme", "tox", "absorption", "distribution", "metabolism", "excretion", "toxicity"}:
        return f"tdc_{lowered}"
    return lowered


def clean_reference_text(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = str(value).strip()
    return text.replace("\\u00c2\\u00b1", "+/-").replace("\\u00b1", "+/-")


def format_reference_metric_value(mean_value, std_value=""):
    mean_text = clean_reference_text(mean_value)
    std_text = clean_reference_text(std_value)
    if std_text:
        return f"{mean_text} +/- {std_text}"
    return mean_text


def load_manual_benchmark_reference(path="data/benchmark_leaderboards/TDC_ADMET_Benchmark_Performance_by_Model.csv"):
    path = resolve_repo_file(path)
    frame = read_optional_csv(path)
    if frame is None or frame.empty:
        return pd.DataFrame()

    required = {"Dataset", "Metric", "Direction", "Model", "Score_Mean"}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"Manual benchmark reference is missing required columns: {missing}")

    work = frame.copy()
    work["Score_Mean"] = pd.to_numeric(work["Score_Mean"], errors="coerce")
    if "Score_StdDev" in work.columns:
        work["Score_StdDev"] = pd.to_numeric(work["Score_StdDev"], errors="coerce")
    else:
        work["Score_StdDev"] = np.nan

    rows = []
    captured_at = pd.Timestamp(path.stat().st_mtime, unit="s").strftime("%Y-%m-%d") if path.exists() else ""
    for _, item in work.iterrows():
        benchmark_id = clean_reference_text(item.get("Dataset"))
        task_category = clean_reference_text(item.get("Task_Category"))
        dataset_name = normalize_reference_dataset_name(benchmark_id, "tdc")
        score_mean = item.get("Score_Mean", np.nan)
        score_std = item.get("Score_StdDev", np.nan)
        if pd.isna(score_mean):
            continue
        rank_text = clean_reference_text(item.get("TDC_Rank"))
        rows.append(
            {
                "dataset": dataset_name,
                "dataset_source": f"Manual benchmark reference: {benchmark_id}",
                "benchmark_suite": "tdc",
                "benchmark_id": benchmark_id,
                "task_category": task_category,
                "leaderboard_url": clean_reference_text(item.get("DOI_or_URL")),
                "leaderboard_dataset_split": "reported in source literature/TDC reference table",
                "leaderboard_metric_name": clean_reference_text(item.get("Metric")),
                "leaderboard_metric_direction": clean_reference_text(item.get("Direction")),
                "rank": rank_text,
                "rank_numeric": pd.to_numeric(rank_text, errors="coerce"),
                "model": clean_reference_text(item.get("Model")),
                "metric_value": format_reference_metric_value(score_mean, score_std if pd.notna(score_std) else ""),
                "metric_value_numeric": float(score_mean),
                "is_top10_entry": True,
                "reference_source": "manual_tdc_admet_reference",
                "captured_at": captured_at,
                "paper_short": clean_reference_text(item.get("Paper_Short")),
                "year": clean_reference_text(item.get("Year")),
                "reference": clean_reference_text(item.get("Reference")),
                "doi_or_url": clean_reference_text(item.get("DOI_or_URL")),
                "manual_reference_source_file": str(path).replace("\\", "/"),
            }
        )

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    result["rank_numeric"] = pd.to_numeric(result["rank_numeric"], errors="coerce")
    for (_dataset, _metric), idx in result.groupby(["dataset", "leaderboard_metric_name"], dropna=False).groups.items():
        local = result.loc[idx].copy()
        direction = str(local["leaderboard_metric_direction"].dropna().astype(str).iloc[0]).strip().lower()
        ascending = not direction.startswith("higher")
        local = local.sort_values(["metric_value_numeric", "model"], ascending=[ascending, True])
        missing_rank = result.loc[local.index, "rank_numeric"].isna()
        if bool(missing_rank.any()):
            result.loc[local.index[missing_rank], "rank_numeric"] = np.arange(1, int(missing_rank.sum()) + 1)
            result.loc[local.index[missing_rank], "rank"] = result.loc[local.index[missing_rank], "rank_numeric"].astype(int).astype(str)
    return result


def combine_leaderboard_reference_frames(*frames):
    parts = []
    for frame in frames:
        if frame is None or frame.empty:
            continue
        work = frame.copy()
        if "dataset" in work.columns:
            work["dataset"] = work["dataset"].astype(str).str.strip()
        if "benchmark_id" in work.columns:
            work["benchmark_id"] = work["benchmark_id"].astype(str).str.strip()
        if "metric_value_numeric" in work.columns:
            work["metric_value_numeric"] = pd.to_numeric(work["metric_value_numeric"], errors="coerce")
        if "rank_numeric" in work.columns:
            work["rank_numeric"] = pd.to_numeric(work["rank_numeric"], errors="coerce")
        if "is_top10_entry" not in work.columns:
            work["is_top10_entry"] = True
        parts.append(work)
    if not parts:
        return pd.DataFrame()
    combined = pd.concat(parts, ignore_index=True, sort=False)
    subset_cols = [c for c in ["dataset", "benchmark_id", "leaderboard_metric_name", "model", "metric_value_numeric", "reference_source", "paper_short", "doi_or_url"] if c in combined.columns]
    if subset_cols:
        combined = combined.drop_duplicates(subset=subset_cols, keep="first")
    return combined.reset_index(drop=True)



In [55]:
RUN_DIR = resolve_benchmark_dir(benchmark_run_dir)
SUMMARY_METRICS_FILE = standardize_metric_frame(read_optional_csv(RUN_DIR / "summary_metrics.csv"))
TEST_RMSE_PIVOT = read_optional_csv(RUN_DIR / "test_rmse_pivot.csv")
PREDICTIONS = read_optional_csv(RUN_DIR / "predictions.csv")
GA_HISTORY = read_optional_csv(RUN_DIR / "ga_history.csv")
RUN_CONFIG = read_optional_json(RUN_DIR / "run_config.json")
LEADERBOARD_TOP10 = read_optional_csv(RUN_DIR / "leaderboard_top10_reference.csv")
LEADERBOARD_COMPARISON = read_optional_csv(RUN_DIR / "leaderboard_comparison_by_dataset.csv")
MANUAL_LEADERBOARD_REFERENCE = load_manual_benchmark_reference()
REFERENCE_LEADERBOARD_ROWS = combine_leaderboard_reference_frames(LEADERBOARD_TOP10, MANUAL_LEADERBOARD_REFERENCE)

run_status_rows = []
for status_path in sorted(RUN_DIR.glob("*/run_status.json")):
    payload = read_optional_json(status_path)
    if not isinstance(payload, dict):
        continue
    run_status_rows.append(
        {
            "dataset": status_path.parent.name,
            "status": payload.get("status", "unknown"),
            "checkpoint_stage": payload.get("checkpoint_stage", ""),
            "n_metrics_rows": payload.get("n_metrics_rows", np.nan),
            "started_at": payload.get("started_at", ""),
            "completed_at": payload.get("completed_at", ""),
            "elapsed_seconds": payload.get("elapsed_seconds", np.nan),
        }
    )
RUN_STATUS = pd.DataFrame(run_status_rows)

def _row_has_error_text_series(series):
    text = series.fillna("").astype(str).str.strip().str.lower()
    return text.ne("") & ~text.isin({"nan", "none"})

dataset_metric_tables = []
for metrics_path in sorted(RUN_DIR.glob("*/metrics.csv")):
    dataset_name = metrics_path.parent.name
    try:
        frame = pd.read_csv(metrics_path)
    except Exception:
        continue
    frame = standardize_metric_frame(frame)
    if frame is None or frame.empty:
        continue
    if "dataset" not in frame.columns:
        frame["dataset"] = dataset_name
    dataset_metric_tables.append(frame)

combined_metric_frames = []
if SUMMARY_METRICS_FILE is not None and not SUMMARY_METRICS_FILE.empty:
    summary_frame = SUMMARY_METRICS_FILE.copy()
    if "dataset" not in summary_frame.columns:
        raise ValueError("summary_metrics.csv is missing a dataset column.")
    summary_frame["_metrics_source"] = "summary_metrics_csv"
    combined_metric_frames.append(summary_frame)

if dataset_metric_tables:
    dataset_frame = pd.concat(dataset_metric_tables, ignore_index=True)
    dataset_frame["_metrics_source"] = "dataset_metrics"
    combined_metric_frames.append(dataset_frame)

if not combined_metric_frames:
    raise ValueError(f"No metrics tables were found under {RUN_DIR}")

combined_metrics = pd.concat(combined_metric_frames, ignore_index=True)
combined_metrics = standardize_metric_frame(combined_metrics)

if combined_metrics is None or combined_metrics.empty:
    raise ValueError(f"No metrics tables were found under {RUN_DIR}")
if "dataset" not in combined_metrics.columns:
    raise ValueError("Could not identify a dataset column in the metrics table.")
if "model" not in combined_metrics.columns:
    raise ValueError("Could not identify a model column in the metrics table.")

if "workflow" not in combined_metrics.columns:
    combined_metrics["workflow"] = ""

combined_metrics["_source_rank"] = combined_metrics.get("_metrics_source", "").map(
    {"dataset_metrics": 0, "summary_metrics_csv": 1}
).fillna(9)
combined_metrics["_error_rank"] = _row_has_error_text_series(combined_metrics.get("error", pd.Series(index=combined_metrics.index, dtype=object))).astype(int)
status_series = combined_metrics.get("status", pd.Series(index=combined_metrics.index, dtype=object)).fillna("").astype(str).str.strip().str.lower()
combined_metrics["_status_rank"] = np.where(status_series.str.startswith("skipped") | status_series.str.contains("error", regex=False), 1, 0)

_rmse_probe_col = metric_column(combined_metrics, rmse_metric_column_preference, metric_hint="rmse")
if _rmse_probe_col is not None:
    combined_metrics["_rmse_rank"] = pd.to_numeric(combined_metrics[_rmse_probe_col], errors="coerce")
else:
    combined_metrics["_rmse_rank"] = np.nan
combined_metrics["_rmse_rank"] = combined_metrics["_rmse_rank"].fillna(np.inf)

key_cols = ["dataset", "model", "workflow"]
combined_metrics = combined_metrics.sort_values(
    by=key_cols + ["_error_rank", "_status_rank", "_rmse_rank", "_source_rank"],
    ascending=[True, True, True, True, True, True, True],
)
SUMMARY_METRICS = combined_metrics.groupby(key_cols, as_index=False, dropna=False).first()
SUMMARY_METRICS = SUMMARY_METRICS.drop(columns=["_source_rank", "_error_rank", "_status_rank", "_rmse_rank", "_metrics_source"], errors="ignore")

if SUMMARY_METRICS is None or SUMMARY_METRICS.empty:
    raise ValueError(f"No metrics tables were found under {RUN_DIR}")

if "dataset" not in SUMMARY_METRICS.columns:
    raise ValueError("Could not identify a dataset column in the metrics table.")
if "model" not in SUMMARY_METRICS.columns:
    raise ValueError("Could not identify a model column in the metrics table.")

if {"model", "workflow"}.issubset(set(SUMMARY_METRICS.columns)):
    tabpfn_mask = SUMMARY_METRICS["model"].astype(str).str.contains("tabpfn", case=False, na=False)
    SUMMARY_METRICS.loc[tabpfn_mask, "workflow"] = "TabPFN deep learning"
    cfa_mask = SUMMARY_METRICS["model"].astype(str).str.startswith("CFA (", na=False)
    SUMMARY_METRICS.loc[cfa_mask, "workflow"] = "CFA fusion"

RMSE_COL = metric_column(SUMMARY_METRICS, rmse_metric_column_preference, metric_hint="rmse")
CLASSIFICATION_FALLBACK_COL = classification_metric_column(SUMMARY_METRICS)
# Backward-compatible name used by older display cells; row-specific primary
# classification metrics are stored in analysis_metric/analysis_metric_kind.
BALANCED_ACCURACY_COL = CLASSIFICATION_FALLBACK_COL
R2_COL = metric_column(SUMMARY_METRICS, "AUTO", metric_hint="r2")
SUMMARY_ANALYSIS_METRICS = add_analysis_metric_columns(SUMMARY_METRICS, rmse_col=RMSE_COL, classification_col=CLASSIFICATION_FALLBACK_COL)
SUMMARY_RMSE_PIVOT = (
    SUMMARY_METRICS.pivot_table(index="dataset", columns="model", values=RMSE_COL, aggfunc="mean").reset_index()
    if RMSE_COL is not None
    else pd.DataFrame()
)

print(f"Benchmark run directory: {RUN_DIR}")
print(f"Datasets: {SUMMARY_METRICS['dataset'].nunique()}")
print(f"Models: {SUMMARY_METRICS['model'].nunique()}")
print(f"Rows in merged summary table: {len(SUMMARY_METRICS):,}")
if SUMMARY_METRICS_FILE is not None:
    print(f"Rows in summary_metrics.csv: {len(SUMMARY_METRICS_FILE):,}")
print(f"Rows from dataset-level metrics after dedupe: {len(SUMMARY_METRICS):,}")
print(f"Regression metric column used: {RMSE_COL}")
print(f"Classification fallback metric column used: {CLASSIFICATION_FALLBACK_COL}")
if not SUMMARY_ANALYSIS_METRICS.empty and "analysis_metric_kind" in SUMMARY_ANALYSIS_METRICS.columns:
    classification_primary_summary = (
        SUMMARY_ANALYSIS_METRICS.loc[SUMMARY_ANALYSIS_METRICS["task_kind"].eq("classification")]
        .groupby(["analysis_metric_kind", "analysis_metric"], dropna=False)["dataset"]
        .nunique()
        .reset_index(name="datasets")
        .sort_values(["analysis_metric_kind", "analysis_metric"])
    )
    if not classification_primary_summary.empty:
        print("Classification primary metrics by dataset count:")
        display(classification_primary_summary.reset_index(drop=True))
if R2_COL is not None:
    print(f"R2 column used: {R2_COL}")
if not SUMMARY_ANALYSIS_METRICS.empty:
    task_count_source = SUMMARY_ANALYSIS_METRICS.copy()
    task_error_col = pick_column(task_count_source.columns, ["error", "Error"])
    if task_error_col is not None:
        task_count_source = task_count_source.loc[
            task_count_source[task_error_col].isna()
            | (task_count_source[task_error_col].astype(str).str.strip() == "")
        ].copy()
    task_count_source = task_count_source.loc[task_count_source["analysis_metric_value"].notna()].copy()
    task_counts = task_count_source.groupby("task_kind", dropna=False)["dataset"].nunique().to_dict()
    print(f"Datasets by inferred task kind with successful metrics: {task_counts}")

if "model" in SUMMARY_METRICS.columns:
    tabpfn_rows = SUMMARY_METRICS[SUMMARY_METRICS["model"].astype(str).str.contains("TabPFNRegressor", na=False)]
    if not tabpfn_rows.empty:
        print(
            "TabPFNRegressor rows retained for analysis: "
            f"{len(tabpfn_rows)} rows across {tabpfn_rows['dataset'].nunique()} datasets."
        )

if RUN_CONFIG is not None:
    display_note("Loaded `run_config.json` for this benchmark run.")

if not RUN_STATUS.empty:
    completed = int((RUN_STATUS["status"].astype(str).str.lower() == "completed").sum())
    print(f"Datasets with run_status.json: {len(RUN_STATUS)} (completed: {completed})")

if LEADERBOARD_TOP10 is not None:
    print(f"Leaderboard top-10 reference rows from run artifact: {len(LEADERBOARD_TOP10):,}")
if MANUAL_LEADERBOARD_REFERENCE is not None and not MANUAL_LEADERBOARD_REFERENCE.empty:
    print(
        "Manual benchmark reference rows loaded: "
        f"{len(MANUAL_LEADERBOARD_REFERENCE):,} across {MANUAL_LEADERBOARD_REFERENCE['dataset'].nunique()} datasets"
    )
if REFERENCE_LEADERBOARD_ROWS is not None and not REFERENCE_LEADERBOARD_ROWS.empty:
    print(
        "Combined leaderboard/reference rows available for comparison: "
        f"{len(REFERENCE_LEADERBOARD_ROWS):,} across {REFERENCE_LEADERBOARD_ROWS['dataset'].nunique()} datasets"
    )
if LEADERBOARD_COMPARISON is not None:
    print(f"Precomputed leaderboard comparison rows from run artifact: {len(LEADERBOARD_COMPARISON):,}")


Benchmark run directory: C:\Users\Scott.Coffin\OneDrive - California OEHHA\R_new\AutoQSAR\benchmark_results\all_benchmarks_no_unimol_20260425_223505
Datasets: 44
Models: 25
Rows in merged summary table: 858
Rows from dataset-level metrics after dedupe: 858
Regression metric column used: test_rmse
Classification fallback metric column used: test_roc_auc
Classification primary metrics by dataset count:


,analysis_metric_kind,analysis_metric,datasets
0,auprc,test_auprc,6
1,roc_auc,test_roc_auc,16


R2 column used: test_r2
Datasets by inferred task kind with successful metrics: {'classification': 22, 'regression': 22}
TabPFNRegressor rows retained for analysis: 34 rows across 34 datasets.


Loaded `run_config.json` for this benchmark run.

Datasets with run_status.json: 45 (completed: 44)
Leaderboard top-10 reference rows from run artifact: 164
Manual benchmark reference rows loaded: 190 across 28 datasets
Combined leaderboard/reference rows available for comparison: 354 across 37 datasets


In [56]:
config_summary = pd.DataFrame([RUN_CONFIG]) if isinstance(RUN_CONFIG, dict) else pd.DataFrame()
if not config_summary.empty:
    display(config_summary.T.rename(columns={0: "value"}))

analysis_metrics = SUMMARY_ANALYSIS_METRICS.copy()
metric_error_col = pick_column(analysis_metrics.columns, ["error", "Error"])
if metric_error_col is not None:
    analysis_metrics = analysis_metrics.loc[
        analysis_metrics[metric_error_col].isna()
        | (analysis_metrics[metric_error_col].astype(str).str.strip() == "")
    ].copy()
analysis_metrics = add_analysis_metric_columns(analysis_metrics, rmse_col=RMSE_COL, classification_col=BALANCED_ACCURACY_COL)
analysis_metrics = analysis_metrics.loc[analysis_metrics["analysis_metric_value"].notna()].copy()

best_by_dataset = best_rows_by_dataset(analysis_metrics)
best_display_cols = [
    col
    for col in [
        "dataset",
        "task_kind",
        "workflow",
        "model",
        "analysis_metric",
        "analysis_metric_kind",
        "analysis_metric_value",
        "analysis_delta_from_best",
        RMSE_COL,
        "test_roc_auc",
        "test_auprc",
        "test_balanced_accuracy",
        "test_mcc",
        R2_COL,
    ]
    if col is not None and col in best_by_dataset.columns
]
display_note("### Best model per dataset")
display(best_by_dataset[best_display_cols].sort_values(["task_kind", "dataset"]).reset_index(drop=True))

summary_rows = []
group_cols = [col for col in ["workflow", "model"] if col in analysis_metrics.columns]
for keys, group in analysis_metrics.groupby(group_cols, dropna=False):
    if not isinstance(keys, tuple):
        keys = (keys,)
    row = dict(zip(group_cols, keys))
    row.update(
        {
            "datasets": int(group["dataset"].nunique()),
            "regression_datasets": int(group.loc[group["task_kind"].eq("regression"), "dataset"].nunique()),
            "classification_datasets": int(group.loc[group["task_kind"].eq("classification"), "dataset"].nunique()),
            "mean_delta_from_dataset_best": float(group["analysis_delta_from_best"].mean(skipna=True)),
            "median_delta_from_dataset_best": float(group["analysis_delta_from_best"].median(skipna=True)),
            "mean_rmse": float(pd.to_numeric(group[RMSE_COL], errors="coerce").mean(skipna=True)) if RMSE_COL in group.columns else np.nan,
            "mean_classification_primary_metric": float(pd.to_numeric(group.loc[group["task_kind"].eq("classification"), "analysis_metric_value"], errors="coerce").mean(skipna=True)),
            "mean_test_roc_auc": float(pd.to_numeric(group["test_roc_auc"], errors="coerce").mean(skipna=True)) if "test_roc_auc" in group.columns else np.nan,
            "mean_test_auprc": float(pd.to_numeric(group["test_auprc"], errors="coerce").mean(skipna=True)) if "test_auprc" in group.columns else np.nan,
            "mean_balanced_accuracy": float(pd.to_numeric(group["test_balanced_accuracy"], errors="coerce").mean(skipna=True)) if "test_balanced_accuracy" in group.columns else np.nan,
            "mean_mcc": float(pd.to_numeric(group["test_mcc"], errors="coerce").mean(skipna=True)) if "test_mcc" in group.columns else np.nan,
        }
    )
    summary_rows.append(row)

model_summary = pd.DataFrame(summary_rows)
display_note("### Cross-dataset model summary")
if model_summary.empty:
    display_note("No successful model rows with usable regression/classification metrics were available.")
else:
    display(model_summary.sort_values("mean_delta_from_dataset_best", ascending=True).head(int(show_top_n_models)).reset_index(drop=True))

regression_best = best_by_dataset.loc[best_by_dataset["task_kind"].eq("regression")].copy()
if not regression_best.empty:
    display_note("### Regression winner diagnostics")
    display(
        regression_best.groupby(["model", "workflow"], dropna=False)
        .agg(
            best_dataset_count=("dataset", "nunique"),
            median_best_rmse=(RMSE_COL, "median") if RMSE_COL in regression_best.columns else ("analysis_metric_value", "median"),
            mean_best_rmse=(RMSE_COL, "mean") if RMSE_COL in regression_best.columns else ("analysis_metric_value", "mean"),
        )
        .reset_index()
        .sort_values(["best_dataset_count", "median_best_rmse", "model"], ascending=[False, True, True])
        .reset_index(drop=True)
    )

    regression_cols = [
        col
        for col in [
            "dataset",
            "model",
            "workflow",
            RMSE_COL,
            R2_COL,
            "analysis_delta_from_best",
            "elapsed_seconds",
        ]
        if col is not None and col in analysis_metrics.columns
    ]
    display_note("Regression dataset-level winners")
    display(regression_best[regression_cols].sort_values(["dataset", "model"]).reset_index(drop=True))

    regression_all = analysis_metrics.loc[analysis_metrics["task_kind"].eq("regression")].copy()
    if not regression_all.empty:
        regression_all["rmse_rank_within_dataset"] = regression_all.groupby("dataset")[RMSE_COL].rank(method="min", ascending=True) if RMSE_COL in regression_all.columns else regression_all.groupby("dataset")["analysis_metric_value"].rank(method="min", ascending=True)
        regression_detail_cols = [
            col
            for col in [
                "dataset",
                "model",
                "workflow",
                RMSE_COL,
                R2_COL,
                "rmse_rank_within_dataset",
                "analysis_delta_from_best",
                "elapsed_seconds",
            ]
            if col is not None and col in regression_all.columns
        ]
        display_note("Regression model rankings by dataset")
        display(regression_all[regression_detail_cols].sort_values(["dataset", "rmse_rank_within_dataset", "model"]).reset_index(drop=True))

classification_best = best_by_dataset.loc[best_by_dataset["task_kind"].eq("classification")].copy()
if not classification_best.empty:
    display_note("### Classification winner diagnostics")
    display(
        classification_best.groupby(["model", "workflow"], dropna=False)
        .agg(best_dataset_count=("dataset", "nunique"))
        .reset_index()
        .sort_values(["best_dataset_count", "model"], ascending=[False, True])
        .reset_index(drop=True)
    )

    adaboost_best_datasets = classification_best.loc[classification_best["model"].astype(str).str.lower().eq("adaboost"), "dataset"].tolist()
    if adaboost_best_datasets:
        ada_cols = [
            col
            for col in [
                "dataset",
                "model",
                "workflow",
                "analysis_metric",
                "analysis_metric_kind",
                "analysis_metric_value",
                "test_roc_auc",
                "cv_roc_auc",
                "test_auprc",
                "test_balanced_accuracy",
                "cv_balanced_accuracy",
                "test_mcc",
                "analysis_delta_from_best",
                "elapsed_seconds",
            ]
            if col in analysis_metrics.columns
        ]
        display_note(
            "AdaBoost is only counted as best where it has the highest held-out primary classification metric for that dataset. "
            "CYP leaderboard tasks use AUPRC where designated; other classification tasks use their declared primary metric or AUROC fallback."
        )
        display(
            analysis_metrics.loc[
                analysis_metrics["dataset"].isin(adaboost_best_datasets)
                & analysis_metrics["task_kind"].eq("classification"),
                ada_cols,
            ]
            .sort_values(["dataset", "analysis_delta_from_best", "model"])
            .reset_index(drop=True)
        )
    else:
        display_note(
            "AdaBoost is not the best model for any successful classification dataset after ranking by the dataset-specific primary classification metric; "
            "RMSE-only ranking would have been an artifact because classification RMSE values are missing."
        )


,value
dataset,None
refresh_leaderboards_only,False
dataset_name,None
include_local_csv,None
output_dir,benchmark_results\all_benchmarks_no_unimol_202...
...,...
ga_models_resolved,
ga_resolution,"{'mode': 'disabled', 'reason': 'empty_ga_models'}"
default_feature_families,"[morgan, ecfp6, fcfp6, layered, atom_pair, top..."
config_signature,"{""benchmark_profile"": ""cost_optimized"", ""cfa_i..."


### Best model per dataset

,dataset,task_kind,workflow,model,analysis_metric,analysis_metric_kind,analysis_metric_value,analysis_delta_from_best,test_rmse,test_roc_auc,test_auprc,test_balanced_accuracy,test_mcc,test_r2
0,tdc_ames,classification,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_roc_auc,roc_auc,0.876017,0.0,NaN,0.876017,0.906770,0.790616,0.596504,NaN
1,tdc_bbb_martins,classification,conventional,Random forest,test_roc_auc,roc_auc,0.932419,0.0,NaN,0.932419,0.982355,0.755824,0.584558,NaN
2,tdc_bioavailability_ma,classification,conventional,CatBoost,test_roc_auc,roc_auc,0.776688,0.0,NaN,0.776688,0.906613,0.652311,0.359420,NaN
3,tdc_carcinogens_lagunin,classification,conventional,AdaBoost,test_roc_auc,roc_auc,0.862626,0.0,NaN,0.862626,0.703605,0.681818,0.560968,NaN
4,tdc_clintox,classification,conventional,CatBoost,test_roc_auc,roc_auc,0.948884,0.0,NaN,0.948884,0.636451,0.798214,0.564337,NaN
5,tdc_cyp1a2_veith,classification,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_roc_auc,roc_auc,0.951752,0.0,NaN,0.951752,0.942873,0.886602,0.773028,NaN
6,tdc_cyp2c19_veith,classification,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_roc_auc,roc_auc,0.904288,0.0,NaN,0.904288,0.868276,0.837683,0.673282,NaN
7,tdc_cyp2c9_substrate_carbonmangels,classification,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_auprc,auprc,0.427480,0.0,NaN,0.638090,0.427480,0.526316,0.195922,NaN
8,tdc_cyp2c9_veith,classification,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_auprc,auprc,0.810239,0.0,NaN,0.901544,0.810239,0.819182,0.631261,NaN
9,tdc_cyp2d6_substrate_carbonmangels,classification,Chemprop v2,"Chemprop v2 (AttentiveFP, ensemble=1)",test_auprc,auprc,0.683938,0.0,NaN,0.800430,0.683938,0.684403,0.404049,NaN


### Cross-dataset model summary

,workflow,model,datasets,regression_datasets,classification_datasets,mean_delta_from_dataset_best,median_delta_from_dataset_best,mean_rmse,mean_classification_primary_metric,mean_test_roc_auc,mean_test_auprc,mean_balanced_accuracy,mean_mcc
0,TabPFN deep learning,TabPFNClassifier,10,0,10,0.042888,0.036254,NaN,0.750112,0.777952,0.770771,0.700566,0.429057
1,ensemble,Ensemble (Weighted average (inverse train RMSE)),22,9,13,0.045239,0.008326,4.616541,0.840897,0.863015,0.831817,0.551154,0.124795
2,conventional,"Voting Classifier (KNN, SVM)",22,0,22,0.046576,0.027351,NaN,0.786376,0.814536,0.759228,0.709263,0.463536
3,conventional,SVC,22,0,22,0.057313,0.033378,NaN,0.775639,0.805143,0.749421,0.723697,0.470192
4,conventional,LogisticRegression,22,0,22,0.060626,0.053451,NaN,0.772326,0.805353,0.751210,0.721473,0.458598
5,MapLight + GNN,"MapLight + GNN (CatBoost, Strict Parity)",44,22,22,0.204065,0.066499,5.883022,0.722139,0.756991,0.695248,0.740933,0.532115
6,conventional,CatBoost,44,22,22,0.215265,0.022903,5.991561,0.808278,0.835837,0.785302,0.737363,0.511884
7,conventional,MapLight CatBoost (Strict Parity),22,22,0,0.276120,0.043543,5.861825,NaN,NaN,NaN,NaN,NaN
8,conventional,XGBoost,44,22,22,0.315012,0.019382,6.190355,0.807578,0.836425,0.777877,0.740355,0.509545
9,conventional,Random forest,44,22,22,0.344958,0.032676,6.246482,0.803812,0.830316,0.780734,0.728452,0.506961


...

,dataset,model,workflow,test_rmse,test_r2,rmse_rank_within_dataset,analysis_delta_from_best,elapsed_seconds
0,chemml_cep_homo,TabPFNRegressor,TabPFN deep learning,0.079691,0.987207,1.0,0.000000,446.350
1,chemml_cep_homo,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,0.080214,0.987039,2.0,0.000523,942.245
2,chemml_cep_homo,Ensemble (Weighted average (inverse train RMSE)),ensemble,0.094496,0.982012,3.0,0.014805,942.433
3,chemml_cep_homo,HistGradientBoosting,conventional,0.103895,0.978256,4.0,0.024204,150.067
4,chemml_cep_homo,MapLight CatBoost (Strict Parity),conventional,0.107689,0.976639,5.0,0.027998,386.510
...,...,...,...,...,...,...,...,...
409,tdc_vdss_lombardo,HistGradientBoosting,conventional,8.051832,-1.204970,14.0,3.367701,612.648
410,tdc_vdss_lombardo,Tabular MLP,conventional,8.337577,-1.364248,15.0,3.653446,663.064
411,tdc_vdss_lombardo,CFA (Combinatorial Fusion),CFA fusion,8.662622,-1.552184,16.0,3.978491,1423.308
412,tdc_vdss_lombardo,Chemprop v2 (AttentiveFP + Selected descriptor...,Chemprop v2,10.265542,-2.584074,17.0,5.581411,1183.763


### Classification winner diagnostics

,model,workflow,best_dataset_count
0,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,7
1,Ensemble (Weighted average (inverse train RMSE)),ensemble,4
2,CatBoost,conventional,3
3,AdaBoost,conventional,2
4,CFA (Combinatorial Fusion),CFA fusion,1
5,"Chemprop v2 (AttentiveFP, ensemble=1)",Chemprop v2,1
6,HistGradientBoosting,conventional,1
7,LogisticRegression,conventional,1
8,Random forest,conventional,1
9,Tabular MLP,conventional,1


AdaBoost is only counted as best where it has the highest held-out primary classification metric for that dataset. CYP leaderboard tasks use AUPRC where designated; other classification tasks use their declared primary metric or AUROC fallback.

,dataset,model,workflow,analysis_metric,analysis_metric_kind,analysis_metric_value,test_roc_auc,cv_roc_auc,test_auprc,test_balanced_accuracy,cv_balanced_accuracy,test_mcc,analysis_delta_from_best,elapsed_seconds
0,tdc_carcinogens_lagunin,AdaBoost,conventional,test_roc_auc,roc_auc,0.862626,0.862626,0.914248,0.703605,0.681818,0.786341,0.560968,0.000000,66.059
1,tdc_carcinogens_lagunin,XGBoost,conventional,test_roc_auc,roc_auc,0.860606,0.860606,0.930251,0.636400,0.693939,0.820574,0.440386,0.002020,71.623
2,tdc_carcinogens_lagunin,Chemprop v2 (AttentiveFP + Selected descriptor...,Chemprop v2,test_roc_auc,roc_auc,0.832323,0.832323,NaN,0.708567,0.750505,NaN,0.568831,0.030303,410.341
3,tdc_carcinogens_lagunin,LogisticRegression,conventional,test_roc_auc,roc_auc,0.830303,0.830303,0.994993,0.643853,0.750505,0.904158,0.568831,0.032323,26.802
4,tdc_carcinogens_lagunin,CatBoost,conventional,test_roc_auc,roc_auc,0.822222,0.822222,0.948469,0.610155,0.716162,0.789118,0.555329,0.040404,83.694
5,tdc_carcinogens_lagunin,TabPFNClassifier,TabPFN deep learning,test_roc_auc,roc_auc,0.818182,0.818182,0.988253,0.622381,0.728283,0.901883,0.473617,0.044444,122.137
6,tdc_carcinogens_lagunin,HistGradientBoosting,conventional,test_roc_auc,roc_auc,0.806061,0.806061,0.924145,0.598551,0.648485,0.836434,0.356753,0.056566,53.321
7,tdc_carcinogens_lagunin,"Chemprop v2 (AttentiveFP, ensemble=1)",Chemprop v2,test_roc_auc,roc_auc,0.787879,0.787879,NaN,0.649915,0.727273,NaN,0.633300,0.074747,287.937
8,tdc_carcinogens_lagunin,Extra trees,conventional,test_roc_auc,roc_auc,0.777778,0.777778,0.949620,0.522942,0.716162,0.793713,0.555329,0.084848,37.740
9,tdc_carcinogens_lagunin,Tabular MLP,conventional,test_roc_auc,roc_auc,0.769697,0.769697,0.974935,0.607702,0.705051,0.946380,0.492659,0.092929,70.064


### GA value diagnostics across runs

This section checks whether GA tuning actually improved benchmark performance in prior runs.

- It compares each GA-tuned model (for example, `CatBoost GA`) against its non-GA counterpart on the same dataset.
- Positive `delta_vs_baseline` means GA improved performance (direction-aware by metric type).
- This is computed across all run folders under `benchmark_results/`.
        


In [57]:
# GA_IMPACT_DIAGNOSTICS_MARKER
# Evaluate GA uplift by dataset/model and aggregate across prior benchmark runs.


def ga_baseline_candidates(ga_model_name):
    ga_model_name = str(ga_model_name or "").strip()
    candidates = []
    if ga_model_name.endswith(" GA"):
        base = ga_model_name[: -len(" GA")].strip()
        if base:
            candidates.extend([base, f"Tuned {base}"])
    elif ga_model_name.startswith("Tuned "):
        base = ga_model_name[len("Tuned ") :].strip()
        if base:
            candidates.extend([base])
    return [c for c in candidates if c]


def metric_is_lower_better(metric_name):
    text = str(metric_name or "").strip().lower()
    if not text:
        return True
    if any(key in text for key in ["rmse", "mae", "mse", "loss", "error"]):
        return True
    if any(key in text for key in ["r2", "spearman", "pearson", "auc", "accuracy", "f1"]):
        return False
    return True


def load_run_metric_table(run_dir):
    summary = standardize_metric_frame(read_optional_csv(run_dir / "summary_metrics.csv"))
    if summary is not None and not summary.empty:
        return summary

    parts = []
    for metrics_path in sorted(run_dir.glob("*/metrics.csv")):
        dataset_name = metrics_path.parent.name
        frame = standardize_metric_frame(pd.read_csv(metrics_path))
        if frame is None or frame.empty:
            continue
        if "dataset" not in frame.columns:
            frame["dataset"] = dataset_name
        parts.append(frame)
    if not parts:
        return None
    return pd.concat(parts, ignore_index=True)


benchmark_root = RUN_DIR.parent if RUN_DIR.parent.name == "benchmark_results" else Path("./benchmark_results").resolve()
run_dirs = sorted([p for p in benchmark_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)

ga_rows = []
ga_attempt_rows = []
for run_dir in run_dirs:
    run_name = run_dir.name
    metrics = load_run_metric_table(run_dir)
    if metrics is None or metrics.empty:
        continue

    metrics = metrics.copy()
    rmse_col = metric_column(metrics, "AUTO", metric_hint="rmse")

    # Normalize key metric columns.
    for col in ["primary_metric_value", "primary_metric", rmse_col, "dataset", "model", "workflow"]:
        if col is not None and col in metrics.columns:
            if col in {"primary_metric_value", rmse_col}:
                metrics[col] = pd.to_numeric(metrics[col], errors="coerce")
            else:
                metrics[col] = metrics[col].astype(str)

    model_series = metrics["model"].astype(str) if "model" in metrics.columns else pd.Series(dtype=str)
    workflow_series = metrics["workflow"].astype(str).str.lower() if "workflow" in metrics.columns else pd.Series(dtype=str)
    ga_mask = model_series.str.endswith(" GA") | workflow_series.str.contains("ga", regex=False)
    ga_metrics = metrics.loc[ga_mask].copy()
    if ga_metrics.empty:
        continue

    for _, ga_row in ga_metrics.iterrows():
        dataset_name = str(ga_row.get("dataset", ""))
        ga_model_name = str(ga_row.get("model", "")).strip()
        ga_error_text = str(ga_row.get("error", "")).strip() if "error" in ga_row.index else ""
        ga_attempt_rows.append(
            {
                "run_name": run_name,
                "dataset": dataset_name,
                "ga_model": ga_model_name,
                "workflow": str(ga_row.get("workflow", "")),
                "error": ga_error_text,
                "has_error": bool(ga_error_text),
                "has_primary_metric": bool(pd.notna(ga_row.get("primary_metric_value", np.nan))),
            }
        )
        candidates = ga_baseline_candidates(ga_model_name)
        if not candidates:
            continue

        dataset_rows = metrics.loc[metrics["dataset"].astype(str) == dataset_name].copy()
        baseline_rows = dataset_rows.loc[dataset_rows["model"].astype(str).isin(candidates)].copy()
        baseline_rows = baseline_rows.loc[~baseline_rows["model"].astype(str).str.endswith(" GA")]
        if baseline_rows.empty:
            continue

        baseline = baseline_rows.iloc[0]

        ga_primary_name = str(ga_row.get("primary_metric", "")).strip().lower()
        baseline_primary_name = str(baseline.get("primary_metric", "")).strip().lower()
        use_primary = (
            "primary_metric_value" in ga_row.index
            and "primary_metric_value" in baseline.index
            and pd.notna(ga_row.get("primary_metric_value", np.nan))
            and pd.notna(baseline.get("primary_metric_value", np.nan))
            and ga_primary_name
            and baseline_primary_name
            and ga_primary_name == baseline_primary_name
        )

        if use_primary:
            metric_name = ga_primary_name
            ga_value = float(ga_row.get("primary_metric_value", np.nan))
            base_value = float(baseline.get("primary_metric_value", np.nan))
        elif rmse_col is not None and rmse_col in ga_row.index and rmse_col in baseline.index:
            metric_name = "rmse"
            ga_value = float(ga_row.get(rmse_col, np.nan))
            base_value = float(baseline.get(rmse_col, np.nan))
        else:
            continue

        if not np.isfinite(ga_value) or not np.isfinite(base_value):
            continue

        lower_better = metric_is_lower_better(metric_name)
        delta = (base_value - ga_value) if lower_better else (ga_value - base_value)
        denom = abs(base_value) if abs(base_value) > 1e-12 else np.nan
        rel_delta = (delta / denom) if np.isfinite(denom) else np.nan

        ga_rows.append(
            {
                "run_name": run_name,
                "dataset": dataset_name,
                "ga_model": ga_model_name,
                "baseline_model": str(baseline.get("model", "")),
                "metric": metric_name,
                "lower_is_better": bool(lower_better),
                "ga_value": float(ga_value),
                "baseline_value": float(base_value),
                "delta_vs_baseline": float(delta),
                "relative_delta": float(rel_delta) if pd.notna(rel_delta) else np.nan,
                "ga_better": bool(delta > 0.0),
            }
        )

GA_IMPACT = pd.DataFrame(ga_rows)
GA_ATTEMPTS = pd.DataFrame(ga_attempt_rows)

if GA_IMPACT.empty:
    if GA_ATTEMPTS.empty:
        display_note("No GA model rows were found in the scanned run directories.")
    else:
        display_note("No comparable GA-vs-baseline pairs with valid metrics were found. GA attempt diagnostics are shown below.")
        attempt_summary = (
            GA_ATTEMPTS.groupby("run_name", dropna=False)
            .agg(
                ga_rows=("ga_model", "size"),
                ga_rows_with_error=("has_error", "sum"),
                ga_rows_with_primary_metric=("has_primary_metric", "sum"),
            )
            .reset_index()
        )
        display(attempt_summary.sort_values("run_name").reset_index(drop=True))

        error_rows = GA_ATTEMPTS.loc[GA_ATTEMPTS["has_error"]].copy()
        if not error_rows.empty:
            error_summary = (
                error_rows.groupby(["run_name", "ga_model", "error"], dropna=False)
                .size()
                .reset_index(name="count")
                .sort_values("count", ascending=False)
            )
            display_note("Top GA error signatures")
            display(error_summary.head(20).reset_index(drop=True))
else:
    display_note("### GA uplift by dataset")
    display(
        GA_IMPACT.sort_values(["ga_better", "delta_vs_baseline"], ascending=[False, False])
        .reset_index(drop=True)
    )

    ga_summary = (
        GA_IMPACT.groupby(["run_name", "ga_model"], dropna=False)
        .agg(
            datasets_compared=("dataset", "nunique"),
            wins=("ga_better", "sum"),
            mean_delta=("delta_vs_baseline", "mean"),
            median_delta=("delta_vs_baseline", "median"),
            mean_relative_delta=("relative_delta", "mean"),
        )
        .reset_index()
    )
    ga_summary["win_rate"] = ga_summary["wins"] / ga_summary["datasets_compared"].replace(0, np.nan)

    display_note("### GA uplift summary by run and GA model")
    display(ga_summary.sort_values(["win_rate", "mean_delta"], ascending=[False, False]).reset_index(drop=True))

    any_wins = GA_IMPACT.loc[GA_IMPACT["ga_better"]].copy()
    if any_wins.empty:
        display_note("GA did not beat its non-GA counterpart on any comparable dataset in the scanned runs.")
    else:
        best_wins = any_wins.sort_values("delta_vs_baseline", ascending=False).head(25)
        display_note("### Datasets where GA improved performance")
        display(best_wins.reset_index(drop=True))

    fig = px.bar(
        ga_summary.sort_values("win_rate", ascending=False),
        x="ga_model",
        y="win_rate",
        color="run_name",
        barmode="group",
        hover_data=[c for c in ["datasets_compared", "wins", "mean_delta", "mean_relative_delta"] if c in ga_summary.columns],
        title="GA win rate vs non-GA baseline",
    )
    fig.update_layout(xaxis_title="GA model", yaxis_title="Win rate")
    fig.show()
        


No GA model rows were found in the scanned run directories.

In [58]:
# Architecture coverage and prediction-diversity diagnostics


def classify_architecture_family(workflow_name, model_name):
    workflow_text = str(workflow_name or "").strip().lower()
    model_text = str(model_name or "").strip().lower()

    if "ensemble" in workflow_text or model_text.startswith("ensemble ("):
        return "Ensemble meta-model"
    if "cfa" in workflow_text or model_text.startswith("cfa ("):
        return "CFA combinatorial fusion"
    if "tabpfn" in workflow_text or "tabpfn" in model_text:
        return "TabPFN foundation model"
    if "chemprop" in workflow_text or "chemprop" in model_text:
        return "Graph neural networks (Chemprop)"
    if "maplight + gnn" in workflow_text or "maplight + gnn" in model_text:
        return "Graph transfer + tabular head"
    if "uni-mol" in workflow_text or "uni-mol" in model_text or "unimol" in model_text:
        return "3D foundation model"
    if "chemml deep learning" in workflow_text or "chemml" in model_text:
        return "Deep tabular neural nets"
    if "tuned conventional" in workflow_text:
        return "Tuned conventional ML"
    if "conventional" in workflow_text:
        return "Conventional ML"
    return "Other / unknown"


def classify_architecture_subfamily(model_name):
    model_text = str(model_name or "").strip().lower()
    if "cmpnn" in model_text:
        return "CMPNN"
    if "attentivefp" in model_text:
        return "AttentiveFP"
    if "d-mpnn" in model_text or "dmpnn" in model_text:
        return "D-MPNN"
    if "maplight + gnn" in model_text:
        return "MapLight + GNN"
    if model_text.startswith("cfa (") or "combinatorial fusion" in model_text:
        return "CFA fusion"
    if "tabpfn" in model_text:
        return "TabPFN"
    if "chemml mlp" in model_text:
        return "ChemML MLP"
    if "catboost" in model_text:
        return "CatBoost-family"
    if "xgboost" in model_text:
        return "XGBoost-family"
    if "random forest" in model_text:
        return "RandomForest-family"
    if "svr" in model_text:
        return "SVR"
    if "elasticnet" in model_text:
        return "ElasticNet-family"
    if model_text.startswith("ensemble ("):
        return "Stacking / averaging"
    return str(model_name or "")


coverage_source = SUMMARY_ANALYSIS_METRICS.copy()
error_col = pick_column(coverage_source.columns, ["error", "Error"])
if error_col is not None:
    coverage_source = coverage_source.loc[
        coverage_source[error_col].isna()
        | (coverage_source[error_col].astype(str).str.strip() == "")
    ].copy()

coverage_source = add_analysis_metric_columns(coverage_source, rmse_col=RMSE_COL, classification_col=BALANCED_ACCURACY_COL)

if coverage_source.empty:
    display_note("No successful model rows were available for architecture-coverage diagnostics.")
else:
    coverage_source["architecture_family"] = coverage_source.apply(
        lambda row: classify_architecture_family(row.get("workflow", ""), row.get("model", "")),
        axis=1,
    )
    coverage_source["architecture_subfamily"] = coverage_source["model"].apply(classify_architecture_subfamily)

    family_summary = (
        coverage_source.groupby("architecture_family", dropna=False)
        .agg(
            datasets=("dataset", "nunique"),
            models=("model", "nunique"),
            rows=("model", "size"),
            mean_delta_from_dataset_best=("analysis_delta_from_best", "mean"),
            mean_rmse=(RMSE_COL, "mean") if RMSE_COL in coverage_source.columns else ("model", "size"),
            mean_balanced_accuracy=(BALANCED_ACCURACY_COL, "mean") if BALANCED_ACCURACY_COL in coverage_source.columns else ("model", "size"),
        )
        .reset_index()
        .sort_values(["datasets", "rows"], ascending=[False, False])
    )

    display_note("### Architecture Coverage")
    display(family_summary)

    best_architecture_rows = best_rows_by_dataset(coverage_source)
    if not best_architecture_rows.empty:
        best_architecture_rows["architecture_family"] = best_architecture_rows.apply(
            lambda row: classify_architecture_family(row.get("workflow", ""), row.get("model", "")),
            axis=1,
        )
        best_architecture_rows["architecture_subfamily"] = best_architecture_rows["model"].apply(classify_architecture_subfamily)

        best_model_counts = (
            best_architecture_rows.groupby(["model", "workflow", "task_kind"], dropna=False)
            .agg(best_dataset_count=("dataset", "nunique"))
            .reset_index()
            .sort_values(["best_dataset_count", "model"], ascending=[False, True])
        )
        display_note("### Overall best model counts")
        display(best_model_counts.reset_index(drop=True))

        best_family_counts = (
            best_architecture_rows.groupby(["architecture_family", "task_kind"], dropna=False)
            .agg(best_dataset_count=("dataset", "nunique"))
            .reset_index()
            .sort_values(["best_dataset_count", "architecture_family"], ascending=[False, True])
        )
        display_note("### Overall best architecture-family counts")
        display(best_family_counts.reset_index(drop=True))

        best_family_overall = (
            best_architecture_rows.groupby("architecture_family", dropna=False)
            .agg(
                winning_datasets=("dataset", "nunique"),
                regression_wins=("task_kind", lambda s: int((s.astype(str) == "regression").sum())),
                classification_wins=("task_kind", lambda s: int((s.astype(str) == "classification").sum())),
            )
            .reset_index()
            .sort_values(["winning_datasets", "architecture_family"], ascending=[False, True])
        )
        best_family_overall["win_fraction"] = best_family_overall["winning_datasets"] / max(float(best_family_overall["winning_datasets"].sum()), 1.0)
        display_note("### Publication Figure: Best-Model Win Counts by Family")
        display(best_family_overall.reset_index(drop=True))

        family_pie = px.pie(
            best_family_overall,
            names="architecture_family",
            values="winning_datasets",
            title="Best model family by dataset",
            hover_data=["regression_wins", "classification_wins", "win_fraction"],
        )
        family_pie.update_traces(textposition="inside", textinfo="percent+label")
        family_pie.show()

        family_rank_source = coverage_source.loc[coverage_source["analysis_delta_from_best"].notna()].copy()
        if not family_rank_source.empty:
            family_rank_source["family_rank_within_dataset"] = family_rank_source.groupby("dataset")["analysis_delta_from_best"].rank(method="min", ascending=True)
            family_rank_heatmap = (
                family_rank_source.groupby(["dataset", "architecture_family"], dropna=False)["family_rank_within_dataset"]
                .min()
                .unstack()
                .sort_index()
            )
            display_note("### Publication Figure: Best Rank Reached by Each Family and Dataset")
            display(family_rank_heatmap.reset_index())
            rank_fig = px.imshow(
                family_rank_heatmap,
                aspect="auto",
                color_continuous_scale="Viridis_r",
                labels={"x": "Architecture family", "y": "Dataset", "color": "Best rank"},
                title="Best rank reached by each model family within each dataset",
                text_auto=".0f",
            )
            rank_fig.update_layout(height=max(420, 45 * len(family_rank_heatmap.index)))
            rank_fig.show()

        winning_model_table = best_architecture_rows[[
            col for col in ["dataset", "task_kind", "architecture_family", "architecture_subfamily", "workflow", "model", "analysis_metric", "analysis_metric_value"]
            if col in best_architecture_rows.columns
        ]].sort_values(["task_kind", "dataset"]).reset_index(drop=True)
        display_note("### Publication Table: Winning Model by Dataset")
        display(winning_model_table)

    dataset_family_presence = (
        coverage_source.assign(present=1)
        .groupby(["dataset", "architecture_family"], dropna=False)["present"]
        .max()
        .unstack(fill_value=0)
        .sort_index()
    )

    coverage_fraction = dataset_family_presence.mean(axis=1).rename("coverage_fraction").reset_index()
    coverage_fraction = coverage_fraction.sort_values("coverage_fraction", ascending=False)
    display_note("### Per-dataset architecture-family coverage")
    display(coverage_fraction)

    coverage_fig = px.imshow(
        dataset_family_presence,
        aspect="auto",
        color_continuous_scale=[[0.0, "#f0f0f0"], [1.0, "#2ca25f"]],
        labels={"x": "Architecture family", "y": "Dataset", "color": "Present"},
        title="Architecture-family coverage matrix (dataset x family)",
        text_auto=True,
    )
    coverage_fig.update_layout(height=max(420, 45 * len(dataset_family_presence.index)))
    coverage_fig.show()

    if PREDICTIONS is None or PREDICTIONS.empty:
        display_note("No predictions table was found; skipping architecture-diversity diagnostics.")
    else:
        pred_df = PREDICTIONS.copy()
        dataset_col = pick_column(pred_df.columns, ["dataset", "dataset_id", "dataset_name"])
        model_col = pick_column(pred_df.columns, ["model", "Model"])
        predicted_col = pick_column(pred_df.columns, ["predicted", "prediction", "y_pred"])
        split_col = pick_column(pred_df.columns, ["split", "Split"])
        smiles_col = pick_column(pred_df.columns, ["smiles", "SMILES", "canonical_smiles"])
        row_col = pick_column(pred_df.columns, ["row_id", "Row", "index"])

        required = [dataset_col, model_col, predicted_col]
        if any(col is None for col in required):
            display_note("Predictions table is missing dataset/model/prediction columns; skipping diversity diagnostics.")
        else:
            pred_df = pred_df.rename(columns={dataset_col: "dataset", model_col: "model", predicted_col: "predicted"})
            if split_col is not None:
                pred_df = pred_df.rename(columns={split_col: "split"})
                pred_df = pred_df.loc[pred_df["split"].astype(str).str.lower() == "test"].copy()

            if smiles_col is not None:
                pred_df = pred_df.rename(columns={smiles_col: "smiles_key"})
            elif row_col is not None:
                pred_df = pred_df.rename(columns={row_col: "smiles_key"})
            else:
                pred_df["smiles_key"] = pred_df.groupby(["dataset", "model"]).cumcount().astype(str)

            pred_df["predicted"] = pd.to_numeric(pred_df["predicted"], errors="coerce")
            pred_df = pred_df.loc[pred_df["predicted"].notna()].copy()

            diversity_rows = []
            pair_rows = []
            for dataset_name, group in pred_df.groupby("dataset", sort=True):
                wide = group.pivot_table(index="smiles_key", columns="model", values="predicted", aggfunc="mean")
                if wide.shape[1] < 2:
                    continue

                corr = wide.corr(method="spearman")
                if corr.shape[0] < 2:
                    continue

                corr_values = corr.to_numpy(dtype=float)
                tri = np.triu_indices_from(corr_values, k=1)
                pair_corr = corr_values[tri]
                pair_corr = pair_corr[np.isfinite(pair_corr)]
                if pair_corr.size == 0:
                    continue

                abs_corr = np.abs(pair_corr)
                diversity_rows.append(
                    {
                        "dataset": dataset_name,
                        "models_in_pairwise_corr": int(corr.shape[0]),
                        "pair_count": int(pair_corr.size),
                        "mean_abs_spearman": float(np.mean(abs_corr)),
                        "median_abs_spearman": float(np.median(abs_corr)),
                        "max_abs_spearman": float(np.max(abs_corr)),
                        "diversity_index": float(1.0 - np.median(abs_corr)),
                    }
                )

                cols = list(corr.columns)
                for i in range(len(cols)):
                    for j in range(i + 1, len(cols)):
                        value = corr.iat[i, j]
                        if pd.isna(value):
                            continue
                        pair_rows.append(
                            {
                                "dataset": dataset_name,
                                "model_a": str(cols[i]),
                                "model_b": str(cols[j]),
                                "spearman": float(value),
                                "abs_spearman": float(abs(value)),
                            }
                        )

            if not diversity_rows:
                display_note("Prediction diversity diagnostics could not be computed (insufficient aligned test predictions).")
            else:
                diversity_df = pd.DataFrame(diversity_rows).sort_values("diversity_index", ascending=False).reset_index(drop=True)
                display_note("### Prediction Diversity Across Architectures")
                display(diversity_df)

                diversity_fig = px.bar(
                    diversity_df.sort_values("diversity_index", ascending=True),
                    x="dataset",
                    y="diversity_index",
                    hover_data=["models_in_pairwise_corr", "median_abs_spearman", "max_abs_spearman"],
                    title="Architecture diversity index by dataset (1 - median |Spearman|)",
                )
                diversity_fig.update_layout(yaxis_title="Diversity index (higher is better)", xaxis_title="Dataset")
                diversity_fig.show()

                if pair_rows:
                    pair_df = pd.DataFrame(pair_rows)
                    pair_df["pair"] = pair_df.apply(
                        lambda row: " / ".join(sorted([str(row["model_a"]), str(row["model_b"])])),
                        axis=1,
                    )
                    pair_summary = (
                        pair_df.groupby("pair", dropna=False)
                        .agg(
                            datasets=("dataset", "nunique"),
                            mean_abs_spearman=("abs_spearman", "mean"),
                            median_abs_spearman=("abs_spearman", "median"),
                            max_abs_spearman=("abs_spearman", "max"),
                        )
                        .reset_index()
                        .sort_values(["mean_abs_spearman", "datasets"], ascending=[False, False])
                    )
                    display_note("### Most redundant model pairs (cross-dataset)")
                    display(pair_summary.head(20))



### Architecture Coverage

,architecture_family,datasets,models,rows,mean_delta_from_dataset_best,mean_rmse,mean_balanced_accuracy
1,Conventional ML,44,15,484,0.692046,6.817332,0.817593
4,Graph neural networks (Chemprop),44,2,84,0.877385,7.047637,0.821860
3,Ensemble meta-model,44,2,66,0.338790,5.975035,0.844050
0,CFA combinatorial fusion,44,1,44,0.453360,6.464336,0.832514
2,Deep tabular neural nets,44,1,44,0.695313,6.896099,0.788712
5,Graph transfer + tabular head,44,1,44,0.204065,5.883022,0.756991
6,TabPFN foundation model,44,2,44,0.472461,10.425320,0.777952


### Overall best model counts

,model,workflow,task_kind,best_dataset_count
0,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,classification,7
1,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,regression,6
2,TabPFNRegressor,TabPFN deep learning,regression,5
3,CFA (Combinatorial Fusion),CFA fusion,regression,4
4,Ensemble (Weighted average (inverse train RMSE)),ensemble,classification,4
5,XGBoost,conventional,regression,4
6,CatBoost,conventional,classification,3
7,AdaBoost,conventional,classification,2
8,CFA (Combinatorial Fusion),CFA fusion,classification,1
9,CatBoost,conventional,regression,1


### Overall best architecture-family counts

...

### Publication Table: Winning Model by Dataset

,dataset,task_kind,architecture_family,architecture_subfamily,workflow,model,analysis_metric,analysis_metric_value
0,tdc_ames,classification,Ensemble meta-model,Stacking / averaging,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_roc_auc,0.876017
1,tdc_bbb_martins,classification,Conventional ML,RandomForest-family,conventional,Random forest,test_roc_auc,0.932419
2,tdc_bioavailability_ma,classification,Conventional ML,CatBoost-family,conventional,CatBoost,test_roc_auc,0.776688
3,tdc_carcinogens_lagunin,classification,Conventional ML,AdaBoost,conventional,AdaBoost,test_roc_auc,0.862626
4,tdc_clintox,classification,Conventional ML,CatBoost-family,conventional,CatBoost,test_roc_auc,0.948884
5,tdc_cyp1a2_veith,classification,Ensemble meta-model,Stacking / averaging,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_roc_auc,0.951752
6,tdc_cyp2c19_veith,classification,Ensemble meta-model,Stacking / averaging,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_roc_auc,0.904288
7,tdc_cyp2c9_substrate_carbonmangels,classification,Ensemble meta-model,Stacking / averaging,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_auprc,0.427480
8,tdc_cyp2c9_veith,classification,Ensemble meta-model,Stacking / averaging,ensemble,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",test_auprc,0.810239
9,tdc_cyp2d6_substrate_carbonmangels,classification,Graph neural networks (Chemprop),AttentiveFP,Chemprop v2,"Chemprop v2 (AttentiveFP, ensemble=1)",test_auprc,0.683938


### Per-dataset architecture-family coverage

,dataset,coverage_fraction
0,chemml_cep_homo,1.0
1,chemml_organic_density,1.0
2,esol_delaney,1.0
3,freesolv_sampl,1.0
4,lipophilicity,1.0
5,poduam_pod_nc_std,1.0
6,poduam_pod_rd_std,1.0
7,polaris_adme_fang_hppb_1,1.0
8,polaris_adme_fang_perm_1,1.0
9,polaris_adme_fang_rclint_1,1.0


No predictions table was found; skipping architecture-diversity diagnostics.

In [59]:
# Feature-family representation analysis from selector outputs

feature_family_drop_presence_threshold = 0.20
feature_family_drop_enrichment_threshold = 0.25
feature_family_drop_selected_total_threshold = 25
feature_family_watch_enrichment_threshold = 0.35


def feature_family_from_name(feature_name):
    name = str(feature_name or "")
    if name.startswith("morgan_bit_"):
        return "morgan"
    if name.startswith("ecfp6_bit_"):
        return "ecfp6"
    if name.startswith("fcfp6_bit_"):
        return "fcfp6"
    if name.startswith("layered_bit_"):
        return "layered"
    if name.startswith("atom_pair_bit_"):
        return "atom_pair"
    if name.startswith("topological_torsion_bit_"):
        return "topological_torsion"
    if name.startswith("rdk_path_bit_"):
        return "rdk_path"
    if name.startswith("maccs_bit_"):
        return "maccs"
    if name.startswith("rdkit_"):
        return "rdkit"
    if name.startswith("avalon_count_"):
        return "avalon"
    if name.startswith("erg_"):
        return "erg"
    if name.startswith("maplight_morgan_") or name.startswith("maplight_desc_"):
        return "maplight"
    if name.startswith("gin_emb_"):
        return "gin_embedding"
    return "other"


def infer_feature_col(frame):
    return pick_column(frame.columns, ["feature", "Feature", "name"])


def infer_magnitude_col(frame):
    if "abs_coefficient" in frame.columns:
        return "abs_coefficient"
    if "importance" in frame.columns:
        return "importance"
    if "coefficient" in frame.columns:
        return "coefficient"
    return None


selector_rows = []
missing_selector_artifacts = []
for dataset_dir in sorted([p for p in RUN_DIR.iterdir() if p.is_dir()]):
    dataset_name = dataset_dir.name
    selected_path = dataset_dir / "selected_features.csv"
    coeff_path = dataset_dir / "selector_coefficients.csv"
    if not selected_path.exists() or not coeff_path.exists():
        missing_selector_artifacts.append(dataset_name)
        continue

    try:
        selected_df = pd.read_csv(selected_path)
        coeff_df = pd.read_csv(coeff_path)
    except Exception as exc:
        missing_selector_artifacts.append(f"{dataset_name} ({type(exc).__name__})")
        continue

    feature_col = infer_feature_col(coeff_df)
    if feature_col is None:
        missing_selector_artifacts.append(f"{dataset_name} (no feature column)")
        continue

    selected_feature_col = infer_feature_col(selected_df)
    selected_set = set()
    if selected_feature_col is not None:
        selected_set = set(selected_df[selected_feature_col].astype(str).tolist())

    coeff_work = coeff_df.copy()
    coeff_work["feature"] = coeff_work[feature_col].astype(str)
    coeff_work["feature_family"] = coeff_work["feature"].map(feature_family_from_name)
    coeff_work["is_selected"] = coeff_work["feature"].isin(selected_set)

    magnitude_col = infer_magnitude_col(coeff_work)
    if magnitude_col is None:
        coeff_work["feature_magnitude"] = np.nan
    elif magnitude_col == "coefficient":
        coeff_work["feature_magnitude"] = pd.to_numeric(coeff_work[magnitude_col], errors="coerce").abs()
    else:
        coeff_work["feature_magnitude"] = pd.to_numeric(coeff_work[magnitude_col], errors="coerce")

    grouped = coeff_work.groupby("feature_family", dropna=False)
    for family_name, family_df in grouped:
        selected_mask = family_df["is_selected"].astype(bool)
        selector_rows.append(
            {
                "dataset": dataset_name,
                "feature_family": str(family_name),
                "available_features": int(len(family_df)),
                "selected_features": int(selected_mask.sum()),
                "selected_magnitude_sum": float(
                    family_df.loc[selected_mask, "feature_magnitude"].fillna(0.0).sum()
                ),
            }
        )

if not selector_rows:
    display_note("No selector feature artifacts were found in this run; skipping feature-family representation analysis.")
else:
    selector_family_df = pd.DataFrame(selector_rows)

    family_summary = (
        selector_family_df.groupby("feature_family", dropna=False)
        .agg(
            datasets_with_family=("dataset", "nunique"),
            datasets_selected=("selected_features", lambda s: int((pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum())),
            available_features_total=("available_features", "sum"),
            selected_features_total=("selected_features", "sum"),
            mean_selected_per_dataset=("selected_features", "mean"),
            median_selected_per_dataset=("selected_features", "median"),
            selected_magnitude_total=("selected_magnitude_sum", "sum"),
        )
        .reset_index()
    )

    family_summary["dataset_presence_rate"] = family_summary["datasets_selected"] / family_summary["datasets_with_family"].replace(0, np.nan)
    family_summary["selection_rate"] = family_summary["selected_features_total"] / family_summary["available_features_total"].replace(0, np.nan)

    total_available = float(family_summary["available_features_total"].sum())
    total_selected = float(family_summary["selected_features_total"].sum())
    total_magnitude = float(family_summary["selected_magnitude_total"].sum())

    if total_available > 0 and total_selected > 0:
        family_summary["expected_selected_if_uniform"] = family_summary["available_features_total"] / total_available * total_selected
        family_summary["enrichment_vs_uniform"] = family_summary["selected_features_total"] / family_summary["expected_selected_if_uniform"].replace(0, np.nan)
    else:
        family_summary["expected_selected_if_uniform"] = np.nan
        family_summary["enrichment_vs_uniform"] = np.nan

    family_summary["selected_share"] = family_summary["selected_features_total"] / max(total_selected, 1.0)
    family_summary["selected_magnitude_share"] = family_summary["selected_magnitude_total"] / max(total_magnitude, 1.0)

    family_summary = family_summary.sort_values(["selected_features_total", "selection_rate"], ascending=[False, False]).reset_index(drop=True)

    display_note("### Feature-Family Representation In Selected Features")
    display(family_summary)

    selected_heatmap = (
        selector_family_df.pivot_table(
            index="dataset",
            columns="feature_family",
            values="selected_features",
            aggfunc="sum",
            fill_value=0,
        )
        .sort_index()
    )

    fig = px.imshow(
        selected_heatmap,
        aspect="auto",
        color_continuous_scale="Viridis",
        labels={"x": "Feature family", "y": "Dataset", "color": "# selected"},
        title="Selected feature counts by dataset and family",
        text_auto=True,
    )
    fig.update_layout(height=max(420, 45 * len(selected_heatmap.index)))
    fig.show()

    enrich_plot_df = family_summary.sort_values("enrichment_vs_uniform", ascending=True)
    fig = px.bar(
        enrich_plot_df,
        x="feature_family",
        y="enrichment_vs_uniform",
        hover_data=["selected_features_total", "available_features_total", "dataset_presence_rate", "selection_rate"],
        title="Feature-family selection enrichment vs uniform-by-dimension baseline",
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="black")
    fig.update_layout(xaxis_title="Feature family", yaxis_title="Enrichment (>1 means over-represented)")
    fig.show()

    def is_tree_based_model_name(model_name):
        text = str(model_name or "").strip().lower()
        return any(key in text for key in ["catboost", "xgboost", "random forest", "extra trees", "histgradient", "adaboost", "lightgbm"])

    importance_rows = []
    tree_metric_source = SUMMARY_ANALYSIS_METRICS.copy()
    tree_error_col = pick_column(tree_metric_source.columns, ["error", "Error"])
    if tree_error_col is not None:
        tree_metric_source = tree_metric_source.loc[
            tree_metric_source[tree_error_col].isna()
            | (tree_metric_source[tree_error_col].astype(str).str.strip() == "")
        ].copy()
    tree_metric_source = add_analysis_metric_columns(tree_metric_source, rmse_col=RMSE_COL, classification_col=BALANCED_ACCURACY_COL)
    tree_metric_source = tree_metric_source.loc[tree_metric_source["model"].map(is_tree_based_model_name)].copy()
    if not tree_metric_source.empty:
        best_tree_models = (
            tree_metric_source.sort_values(["dataset", "analysis_delta_from_best", "model"], ascending=[True, True, True])
            .groupby("dataset", as_index=False)
            .first()[["dataset", "model", "workflow", "analysis_metric", "analysis_metric_value"]]
        )
    else:
        best_tree_models = pd.DataFrame(columns=["dataset", "model", "workflow", "analysis_metric", "analysis_metric_value"])

    for dataset_dir in sorted([p for p in RUN_DIR.iterdir() if p.is_dir()]):
        importance_path = dataset_dir / "model_feature_importances.csv"
        if not importance_path.exists():
            continue
        try:
            imp_df = pd.read_csv(importance_path)
        except Exception:
            continue
        feature_col = infer_feature_col(imp_df)
        importance_col = infer_magnitude_col(imp_df)
        model_col = pick_column(imp_df.columns, ["model", "Model"])
        workflow_col = pick_column(imp_df.columns, ["workflow", "Workflow"])
        if feature_col is None or importance_col is None or model_col is None:
            continue
        work = imp_df.copy()
        work["dataset"] = dataset_dir.name
        work["model"] = work[model_col].astype(str)
        work["workflow"] = work[workflow_col].astype(str) if workflow_col is not None else ""
        work["feature"] = work[feature_col].astype(str)
        work["feature_family"] = work["feature"].map(feature_family_from_name)
        work["importance_weight"] = pd.to_numeric(work[importance_col], errors="coerce").abs()
        work = work.loc[work["importance_weight"].notna()].copy()
        if work.empty:
            continue
        local_best = best_tree_models.loc[best_tree_models["dataset"].astype(str).eq(dataset_dir.name)]
        if not local_best.empty:
            best_model_name = str(local_best.iloc[0]["model"])
            work = work.loc[work["model"].astype(str).eq(best_model_name)].copy()
        if work.empty:
            continue
        grouped_importance = (
            work.groupby(["dataset", "model", "workflow", "feature_family"], dropna=False)
            .agg(importance_weight=("importance_weight", "sum"), features=("feature", "nunique"))
            .reset_index()
        )
        grouped_importance["importance_source"] = "model_feature_importances.csv"
        importance_rows.extend(grouped_importance.to_dict(orient="records"))

    if importance_rows:
        feature_importance_family_df = pd.DataFrame(importance_rows)
    else:
        feature_importance_family_df = selector_family_df.copy()
        feature_importance_family_df["model"] = "selector"
        feature_importance_family_df["workflow"] = "feature selection"
        feature_importance_family_df["importance_weight"] = pd.to_numeric(feature_importance_family_df["selected_magnitude_sum"], errors="coerce")
        missing_weight = ~feature_importance_family_df["importance_weight"].gt(0)
        feature_importance_family_df.loc[missing_weight, "importance_weight"] = pd.to_numeric(feature_importance_family_df.loc[missing_weight, "selected_features"], errors="coerce")
        feature_importance_family_df["features"] = feature_importance_family_df["selected_features"]
        feature_importance_family_df["importance_source"] = "selector_coefficients_fallback"

    if not feature_importance_family_df.empty:
        feature_importance_family_df["importance_weight"] = pd.to_numeric(feature_importance_family_df["importance_weight"], errors="coerce").fillna(0.0)
        dataset_totals = feature_importance_family_df.groupby("dataset")["importance_weight"].transform("sum").replace(0, np.nan)
        feature_importance_family_df["importance_share_within_dataset"] = feature_importance_family_df["importance_weight"] / dataset_totals
        feature_importance_family_df["family_rank_within_dataset"] = feature_importance_family_df.groupby("dataset")["importance_share_within_dataset"].rank(method="min", ascending=False)

        importance_summary = (
            feature_importance_family_df.groupby(["feature_family", "importance_source"], dropna=False)
            .agg(
                datasets=("dataset", "nunique"),
                total_importance_weight=("importance_weight", "sum"),
                mean_importance_share=("importance_share_within_dataset", "mean"),
                median_importance_share=("importance_share_within_dataset", "median"),
                mean_family_rank=("family_rank_within_dataset", "mean"),
                median_family_rank=("family_rank_within_dataset", "median"),
            )
            .reset_index()
            .sort_values(["mean_importance_share", "datasets"], ascending=[False, False])
        )
        display_note("### Feature-Family Importance Summary")
        display(importance_summary.reset_index(drop=True))

        importance_heatmap = (
            feature_importance_family_df.pivot_table(
                index="dataset",
                columns="feature_family",
                values="importance_share_within_dataset",
                aggfunc="sum",
                fill_value=0.0,
            )
            .sort_index()
        )
        display_note("### Publication Figure: Feature-Family Importance by Dataset")
        display(importance_heatmap.reset_index())
        fig = px.imshow(
            importance_heatmap,
            aspect="auto",
            color_continuous_scale="Viridis",
            labels={"x": "Feature family", "y": "Dataset", "color": "Importance share"},
            title="Feature-family importance share by dataset",
        )
        fig.update_layout(height=max(420, 45 * len(importance_heatmap.index)))
        fig.show()
        if not importance_rows:
            display_note(
                "No per-model `model_feature_importances.csv` artifacts were found, so this section uses selector coefficient/importances as a fallback. "
                "Future benchmark runs can replace this with best-tree-model importances automatically."
            )

    drop_candidates = family_summary.loc[
        (family_summary["dataset_presence_rate"].fillna(0.0) <= float(feature_family_drop_presence_threshold))
        & (family_summary["enrichment_vs_uniform"].fillna(np.inf) <= float(feature_family_drop_enrichment_threshold))
        & (family_summary["selected_features_total"].fillna(0.0) <= float(feature_family_drop_selected_total_threshold))
    ].copy()

    watchlist = family_summary.loc[
        (family_summary["enrichment_vs_uniform"].fillna(np.inf) <= float(feature_family_watch_enrichment_threshold))
        & (family_summary["dataset_presence_rate"].fillna(0.0) >= 0.5)
    ].copy()

    if drop_candidates.empty:
        display_note(
            "No feature family met the strict drop-candidate rule in this run. "
            "That means no family looked both rare across datasets and consistently unselected."
        )
    else:
        display_note("### Suggested drop candidates (strict rule)")
        display(
            drop_candidates[[
                "feature_family",
                "datasets_selected",
                "datasets_with_family",
                "dataset_presence_rate",
                "selected_features_total",
                "available_features_total",
                "selection_rate",
                "enrichment_vs_uniform",
            ]].sort_values(["dataset_presence_rate", "selected_features_total"], ascending=[True, True]).reset_index(drop=True)
        )

    if not watchlist.empty:
        display_note("### Watchlist: under-enriched but still used across many datasets")
        display(
            watchlist[[
                "feature_family",
                "datasets_selected",
                "datasets_with_family",
                "dataset_presence_rate",
                "selected_features_total",
                "available_features_total",
                "selection_rate",
                "enrichment_vs_uniform",
            ]].sort_values("enrichment_vs_uniform", ascending=True).reset_index(drop=True)
        )

    if missing_selector_artifacts:
        display_note(
            "Selector artifacts were missing for some datasets and were excluded from this section: "
            + ", ".join(sorted(set(missing_selector_artifacts)))
        )




### Feature-Family Representation In Selected Features

,feature_family,datasets_with_family,datasets_selected,available_features_total,selected_features_total,mean_selected_per_dataset,median_selected_per_dataset,selected_magnitude_total,dataset_presence_rate,selection_rate,expected_selected_if_uniform,enrichment_vs_uniform,selected_share,selected_magnitude_share
0,avalon,44,43,44376,2731,62.068182,35.0,116.191365,0.977273,0.061542,1842.305717,1.482382,0.173210,0.068858
1,fcfp6,44,43,42540,1908,43.363636,38.0,305.189177,0.977273,0.044852,1766.082684,1.080357,0.121012,0.180863
2,ecfp6,44,42,43435,1562,35.500000,30.0,259.898758,0.954545,0.035962,1803.239337,0.866219,0.099068,0.154022
3,atom_pair,44,43,39608,1448,32.909091,23.0,270.904451,0.977273,0.036558,1644.358320,0.880587,0.091837,0.160545
4,rdkit,44,41,8260,1380,31.363636,10.5,41.782560,0.931818,0.167070,342.920615,4.024255,0.087525,0.024761
5,rdk_path,44,42,45036,1305,29.659091,24.5,163.960632,0.954545,0.028977,1869.706153,0.697971,0.082768,0.097167
6,morgan,44,41,40782,1140,25.909091,22.0,183.549403,0.931818,0.027954,1693.097885,0.673322,0.072303,0.108776
7,layered,44,43,43621,1109,25.204545,25.0,113.374569,0.977273,0.025424,1810.961278,0.612382,0.070337,0.067189
8,erg,44,41,12549,1067,24.250000,13.0,47.635277,0.931818,0.085027,520.981937,2.048056,0.067673,0.028230
9,maplight,44,43,30024,1051,23.886364,20.5,81.050956,0.977273,0.035005,1246.470769,0.843181,0.066658,0.048033


### Feature-Family Importance Summary

,feature_family,importance_source,datasets,total_importance_weight,mean_importance_share,median_importance_share,mean_family_rank,median_family_rank
0,rdkit,selector_coefficients_fallback,44,41.782560,0.199632,0.103999,5.568182,4.5
1,avalon,selector_coefficients_fallback,44,116.191365,0.112295,0.099739,4.545455,4.0
2,fcfp6,selector_coefficients_fallback,44,305.189177,0.105473,0.114968,3.909091,3.0
3,atom_pair,selector_coefficients_fallback,44,270.904451,0.092129,0.088023,5.113636,5.0
4,layered,selector_coefficients_fallback,44,113.374569,0.080713,0.068845,6.568182,6.0
5,ecfp6,selector_coefficients_fallback,44,259.898758,0.079155,0.083541,5.750000,5.0
6,rdk_path,selector_coefficients_fallback,44,163.960632,0.075345,0.071922,6.113636,6.0
7,morgan,selector_coefficients_fallback,44,183.549403,0.059583,0.060809,7.295455,7.0
8,erg,selector_coefficients_fallback,44,47.635277,0.054741,0.037760,7.568182,8.0
9,topological_torsion,selector_coefficients_fallback,44,83.294958,0.048537,0.052360,8.409091,8.5


### Publication Figure: Feature-Family Importance by Dataset

feature_family,dataset,atom_pair,avalon,ecfp6,erg,fcfp6,layered,maccs,maplight,morgan,rdk_path,rdkit,topological_torsion
0,chemml_cep_homo,0.075964,0.234145,0.061797,0.063200,0.123114,0.063654,0.010448,0.073347,0.030002,0.145832,0.114578,0.003919
1,chemml_organic_density,0.040995,0.175720,0.031285,0.030337,0.057066,0.033025,0.074716,0.075767,0.021841,0.061901,0.361714,0.035633
2,esol_delaney,0.150231,0.099138,0.106172,0.004144,0.113353,0.076198,0.012964,0.070472,0.052080,0.070978,0.194203,0.050066
3,freesolv_sampl,0.059041,0.108149,0.083729,0.019706,0.080103,0.083223,0.057709,0.045718,0.063190,0.092647,0.288203,0.018581
4,lipophilicity,0.069215,0.109937,0.086424,0.110834,0.135726,0.075856,0.026796,0.041607,0.045751,0.101935,0.131627,0.064292
5,poduam_pod_nc_std,0.019318,0.180410,0.109894,0.006125,0.131402,0.109267,0.022074,0.127080,0.100169,0.060578,0.069871,0.063811
6,poduam_pod_rd_std,0.080297,0.090827,0.122269,0.003918,0.162883,0.086438,0.031609,0.075252,0.119044,0.103265,0.038996,0.085202
7,polaris_adme_fang_hppb_1,0.102193,0.092261,0.129590,0.035858,0.119935,0.058321,0.036820,0.044289,0.077870,0.118106,0.125445,0.059311
8,polaris_adme_fang_perm_1,0.112157,0.119621,0.095562,0.059706,0.131828,0.073770,0.051967,0.043778,0.086173,0.084336,0.078738,0.062363
9,polaris_adme_fang_rclint_1,0.127290,0.087899,0.079359,0.050722,0.112472,0.067064,0.022592,0.071055,0.094996,0.096852,0.112754,0.076946


No per-model `model_feature_importances.csv` artifacts were found, so this section uses selector coefficient/importances as a fallback. Future benchmark runs can replace this with best-tree-model importances automatically.

No feature family met the strict drop-candidate rule in this run. That means no family looked both rare across datasets and consistently unselected.

Selector artifacts were missing for some datasets and were excluded from this section: tdc_herg_central

In [60]:
# Leaderboard-relative performance summary (using run artifacts + manual reference table)

def leaderboard_metric_kind(text):
    t = str(text or "").strip().lower()
    t = t.replace("-", "_").replace(" ", "_")
    if not t:
        return None
    if "mean_squared_error" in t or t == "mse" or t.endswith("_mse"):
        return "mse"
    if "rmse" in t:
        return "rmse"
    if "mae" in t:
        return "mae"
    if "auroc" in t or "roc_auc" in t or "auc_roc" in t or t == "auc" or ("roc" in t and "auc" in t):
        return "roc_auc"
    if "auprc" in t or "pr_auc" in t or "auc_pr" in t or "average_precision" in t:
        return "auprc"
    if "balanced_accuracy" in t or t == "ba":
        return "balanced_accuracy"
    if t in {"accuracy", "acc"} or t.endswith("_accuracy"):
        return "accuracy"
    if t == "mcc" or "matthews" in t:
        return "mcc"
    if "spearman" in t:
        return "spearman"
    if "pearson" in t:
        return "pearson"
    if t in {"r2", "r^2", "r_squared"} or "r2" in t or "r^2" in t:
        return "r2"
    return t


def leaderboard_direction(metric_kind, explicit_direction=None):
    explicit = str(explicit_direction or "").strip().lower()
    if explicit.startswith("lower"):
        return "lower"
    if explicit.startswith("higher"):
        return "higher"
    if metric_kind in {"mse", "rmse", "mae"}:
        return "lower"
    if metric_kind in {"r2", "spearman", "pearson", "roc_auc", "auprc", "balanced_accuracy", "accuracy", "mcc"}:
        return "higher"
    return "lower"


def model_metric_column(frame, metric_kind):
    if metric_kind in {"mse", "rmse"}:
        return metric_column(frame, "AUTO", metric_hint="rmse")
    if metric_kind == "r2":
        return metric_column(frame, "AUTO", metric_hint="r2")
    if metric_kind == "mae":
        return pick_column(frame.columns, ["test_mae", "Test MAE", "cv_mae", "CV MAE", "mae", "MAE"])
    if metric_kind == "spearman":
        return pick_column(frame.columns, ["test_spearman", "Test Spearman", "cv_spearman", "CV Spearman", "spearman", "Spearman"])
    if metric_kind == "pearson":
        return pick_column(frame.columns, ["test_pearson", "Test Pearson", "cv_pearson", "CV Pearson", "pearson", "Pearson"])
    if metric_kind == "roc_auc":
        return pick_column(frame.columns, ["test_roc_auc", "Test ROC AUC", "roc_auc", "ROC AUC", "AUROC", "test_auroc", "cv_roc_auc", "CV ROC AUC"])
    if metric_kind == "auprc":
        return pick_column(frame.columns, ["test_auprc", "AUPRC", "auprc", "cv_auprc", "CV AUPRC", "average_precision"])
    if metric_kind == "balanced_accuracy":
        return pick_column(frame.columns, ["test_balanced_accuracy", "Test Balanced Accuracy", "balanced_accuracy", "Balanced Accuracy", "cv_balanced_accuracy"])
    if metric_kind == "accuracy":
        return pick_column(frame.columns, ["test_accuracy", "accuracy", "Accuracy", "cv_accuracy", "test_balanced_accuracy", "balanced_accuracy"])
    if metric_kind == "mcc":
        return pick_column(frame.columns, ["test_mcc", "MCC", "mcc", "cv_mcc"])
    return None


def parse_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def to_leaderboard_scale(value, metric_kind):
    value = float(value)
    if metric_kind == "mse":
        return value ** 2
    return value


def bool_like(value):
    return str(value).strip().lower() in {"true", "1", "yes", "y", "t"}


def choose_reference_subset(ref, dataset_name, benchmark_id=""):
    if ref is None or ref.empty:
        return pd.DataFrame()
    ref_subset = pd.DataFrame()
    if benchmark_id and "benchmark_id" in ref.columns:
        ref_subset = ref.loc[ref["benchmark_id"].astype(str).str.strip().str.lower() == str(benchmark_id).strip().lower()].copy()
    if ref_subset.empty and "dataset" in ref.columns:
        ref_subset = ref.loc[ref["dataset"].astype(str).str.strip().str.lower() == str(dataset_name).strip().lower()].copy()
    return ref_subset


def best_model_for_reference_metric(dataset_frame, metric_kind):
    metric_col = model_metric_column(dataset_frame, metric_kind)
    candidate = dataset_frame.copy()
    if "error" in candidate.columns:
        candidate = candidate.loc[candidate["error"].isna() | (candidate["error"].astype(str).str.strip() == "")].copy()
    if metric_col is not None and metric_col in candidate.columns:
        candidate["_comparison_value_raw"] = parse_numeric(candidate[metric_col])
        candidate["_comparison_metric_column"] = metric_col
    elif {"primary_metric", "primary_metric_value"}.issubset(candidate.columns):
        primary_kind = candidate["primary_metric"].astype(str).apply(leaderboard_metric_kind)
        candidate = candidate.loc[primary_kind == metric_kind].copy()
        candidate["_comparison_value_raw"] = parse_numeric(candidate["primary_metric_value"])
        candidate["_comparison_metric_column"] = "primary_metric_value"
    else:
        return None, pd.DataFrame()

    candidate = candidate.loc[candidate["_comparison_value_raw"].notna()].copy()
    if candidate.empty:
        return None, candidate
    candidate["_leaderboard_scale_value"] = candidate["_comparison_value_raw"].apply(lambda v: to_leaderboard_scale(v, metric_kind))
    direction = leaderboard_direction(metric_kind)
    best_idx = candidate["_leaderboard_scale_value"].idxmax() if direction == "higher" else candidate["_leaderboard_scale_value"].idxmin()
    return candidate.loc[best_idx], candidate


def reference_values_for_metric(ref_subset, metric_kind):
    work = ref_subset.copy()
    if "leaderboard_metric_name" in work.columns:
        work["_metric_kind"] = work["leaderboard_metric_name"].astype(str).apply(leaderboard_metric_kind)
        work = work.loc[work["_metric_kind"] == metric_kind].copy()
    if work.empty or "metric_value_numeric" not in work.columns:
        return work, np.array([], dtype=float)
    work["metric_value_numeric"] = pd.to_numeric(work["metric_value_numeric"], errors="coerce")
    work = work.loc[work["metric_value_numeric"].notna()].copy()
    return work, work["metric_value_numeric"].to_numpy(dtype=float)


def comparison_rows_from_reference(ref):
    rows = []
    if ref is None or ref.empty:
        return pd.DataFrame()

    work_ref = ref.copy()
    if "is_top10_entry" in work_ref.columns:
        include_mask = work_ref["is_top10_entry"].isna() | work_ref["is_top10_entry"].apply(bool_like)
        work_ref = work_ref.loc[include_mask].copy()

    for dataset_name, dataset_frame in SUMMARY_METRICS.groupby("dataset", sort=True):
        dataset_frame = dataset_frame.copy()
        benchmark_id = ""
        if "benchmark_id" in dataset_frame.columns:
            non_empty_ids = dataset_frame["benchmark_id"].dropna().astype(str)
            non_empty_ids = non_empty_ids[non_empty_ids.str.strip() != ""]
            if not non_empty_ids.empty:
                benchmark_id = non_empty_ids.iloc[0]

        ref_subset = choose_reference_subset(work_ref, dataset_name, benchmark_id)
        if ref_subset.empty:
            continue

        metric_names = ref_subset.get("leaderboard_metric_name", pd.Series(dtype=object)).dropna().astype(str)
        metric_kinds = []
        for metric_name in metric_names:
            kind = leaderboard_metric_kind(metric_name)
            if kind and kind not in metric_kinds:
                metric_kinds.append(kind)

        for metric_kind in metric_kinds:
            metric_ref, metric_values = reference_values_for_metric(ref_subset, metric_kind)
            if len(metric_values) == 0:
                continue

            best_row, candidate = best_model_for_reference_metric(dataset_frame, metric_kind)
            if best_row is None:
                continue

            explicit_direction = ""
            if "leaderboard_metric_direction" in metric_ref.columns:
                directions = metric_ref["leaderboard_metric_direction"].dropna().astype(str)
                directions = directions[directions.str.strip() != ""]
                if not directions.empty:
                    explicit_direction = directions.iloc[0]
            direction = leaderboard_direction(metric_kind, explicit_direction)
            sorted_values = np.sort(metric_values)
            if direction == "higher":
                sorted_values = sorted_values[::-1]

            top1 = float(sorted_values[0])
            cutoff_index = min(9, len(sorted_values) - 1)
            cutoff = float(sorted_values[cutoff_index])
            best_value = float(best_row["_leaderboard_scale_value"])
            best_value_raw = float(best_row["_comparison_value_raw"])

            if direction == "lower":
                estimated_rank = 1 + int(np.sum(metric_values < best_value))
                gap_top1 = best_value - top1
                gap_top10 = best_value - cutoff
            else:
                estimated_rank = 1 + int(np.sum(metric_values > best_value))
                gap_top1 = top1 - best_value
                gap_top10 = cutoff - best_value

            ref_metric_name = str(metric_ref["leaderboard_metric_name"].dropna().astype(str).iloc[0]) if "leaderboard_metric_name" in metric_ref.columns and not metric_ref["leaderboard_metric_name"].dropna().empty else metric_kind
            sources = sorted(metric_ref.get("reference_source", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()) if "reference_source" in metric_ref.columns else []
            models = metric_ref.get("model", pd.Series(dtype=object)).dropna().astype(str).tolist() if "model" in metric_ref.columns else []

            rows.append(
                {
                    "dataset": dataset_name,
                    "benchmark_id": benchmark_id,
                    "leaderboard_metric_name": ref_metric_name,
                    "metric_kind": metric_kind,
                    "direction": direction,
                    "best_model": str(best_row.get("model", "")),
                    "best_workflow": str(best_row.get("workflow", "")),
                    "best_value": best_value,
                    "best_value_raw_model_metric": best_value_raw,
                    "best_value_metric_column": str(best_row.get("_comparison_metric_column", "")),
                    "leaderboard_top1": top1,
                    "leaderboard_top10_cutoff": cutoff,
                    "known_reference_count": int(len(metric_values)),
                    "top10_reference_count": int(min(10, len(metric_values))),
                    "reference_sources": "; ".join(sources),
                    "reference_models": "; ".join(models[:20]),
                    "estimated_rank_vs_top10": int(estimated_rank) if estimated_rank <= 10 else 11,
                    "in_top10_estimate": bool(estimated_rank <= 10),
                    "gap_to_top1": float(gap_top1),
                    "gap_to_top10_cutoff": float(gap_top10),
                }
            )
    return pd.DataFrame(rows)


reference_for_comparison = REFERENCE_LEADERBOARD_ROWS.copy() if "REFERENCE_LEADERBOARD_ROWS" in globals() and REFERENCE_LEADERBOARD_ROWS is not None else pd.DataFrame()
leaderboard_eval = comparison_rows_from_reference(reference_for_comparison)

if (leaderboard_eval is None or leaderboard_eval.empty) and LEADERBOARD_COMPARISON is not None and not LEADERBOARD_COMPARISON.empty:
    leaderboard_eval = LEADERBOARD_COMPARISON.copy()

    rename_map = {
        "leaderboard_rank": "estimated_rank_vs_top10",
        "leaderboard_estimated_rank_vs_top10": "estimated_rank_vs_top10",
        "leaderboard_estimated_in_top10": "in_top10_estimate",
        "leaderboard_gap_to_top10_cutoff": "gap_to_top10_cutoff",
        "leaderboard_delta_primary": "gap_to_top1",
        "primary_metric_value": "best_value_raw_model_metric",
        "primary_metric": "best_value_metric_column",
    }
    applicable_renames = {k: v for k, v in rename_map.items() if k in leaderboard_eval.columns and v not in leaderboard_eval.columns}
    if applicable_renames:
        leaderboard_eval = leaderboard_eval.rename(columns=applicable_renames)

if leaderboard_eval is None or leaderboard_eval.empty:
    display_note("No leaderboard comparison was available for this run.")
else:
    leaderboard_eval = leaderboard_eval.copy()
    if "estimated_rank_vs_top10" in leaderboard_eval.columns:
        leaderboard_eval["estimated_rank_vs_top10"] = pd.to_numeric(leaderboard_eval["estimated_rank_vs_top10"], errors="coerce")
        leaderboard_eval["estimated_rank_label"] = leaderboard_eval["estimated_rank_vs_top10"].apply(lambda x: ">10" if pd.notna(x) and int(x) > 10 else (str(int(x)) if pd.notna(x) else "NA"))
        if "in_top10_estimate" not in leaderboard_eval.columns:
            leaderboard_eval["in_top10_estimate"] = leaderboard_eval["estimated_rank_vs_top10"].fillna(11).astype(float) <= 10

    total = int(len(leaderboard_eval))
    dataset_total = int(leaderboard_eval["dataset"].nunique()) if "dataset" in leaderboard_eval.columns else total
    rank_series = pd.to_numeric(leaderboard_eval.get("estimated_rank_vs_top10", pd.Series(dtype=float)), errors="coerce")
    comparable_mask = rank_series.notna()
    if not comparable_mask.any() and "gap_to_top10_cutoff" in leaderboard_eval.columns:
        comparable_mask = pd.to_numeric(leaderboard_eval["gap_to_top10_cutoff"], errors="coerce").notna()
    comparable = int(comparable_mask.sum())

    in_top10_series = pd.to_numeric(leaderboard_eval.get("in_top10_estimate", pd.Series(dtype=float)), errors="coerce")
    in_top10 = int(in_top10_series[comparable_mask].fillna(0).astype(int).sum()) if comparable > 0 else 0
    top1_count = int((rank_series[comparable_mask] == 1).sum()) if comparable > 0 else 0

    display_note(
        f"### Leaderboard Performance Overview\n"
        f"Comparison rows: **{total}** across **{dataset_total}** datasets  \n"
        f"Rows with leaderboard-comparable metrics: **{comparable}**  \n"
        f"Estimated in top-10: **{in_top10}/{comparable if comparable > 0 else 1}**  \n"
        f"Estimated #1: **{top1_count}**"
    )

    display_cols = [
        "dataset",
        "leaderboard_metric_name",
        "metric_kind",
        "direction",
        "best_model",
        "best_workflow",
        "best_value",
        "best_value_raw_model_metric",
        "best_value_metric_column",
        "leaderboard_top1",
        "leaderboard_top10_cutoff",
        "known_reference_count",
        "reference_sources",
        "estimated_rank_label",
        "in_top10_estimate",
        "gap_to_top1",
        "gap_to_top10_cutoff",
    ]
    display_cols = [c for c in display_cols if c in leaderboard_eval.columns]
    sort_cols = []
    sort_ascending = []
    if "in_top10_estimate" in leaderboard_eval.columns:
        sort_cols.append("in_top10_estimate")
        sort_ascending.append(False)
    if "gap_to_top10_cutoff" in leaderboard_eval.columns:
        sort_cols.append("gap_to_top10_cutoff")
        sort_ascending.append(True)

    display_df = leaderboard_eval[display_cols].copy()
    if sort_cols:
        display_df = display_df.sort_values(by=sort_cols, ascending=sort_ascending)
    display(display_df.reset_index(drop=True))

    if "gap_to_top10_cutoff" in leaderboard_eval.columns:
        plot_df = leaderboard_eval.sort_values("gap_to_top10_cutoff", ascending=True).copy()
        fig = px.bar(
            plot_df,
            x="dataset",
            y="gap_to_top10_cutoff",
            color="metric_kind" if "metric_kind" in plot_df.columns else None,
            hover_data=[col for col in ["best_model", "best_value", "leaderboard_top10_cutoff", "estimated_rank_label", "known_reference_count", "reference_sources"] if col in plot_df.columns],
            title="Gap to leaderboard/reference top-10 cutoff (negative = better than cutoff)",
        )
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(yaxis_title="Gap to top-10 cutoff", xaxis_title="Dataset")
        fig.show()

    if "estimated_rank_label" in leaderboard_eval.columns:
        rank_order = [str(i) for i in range(1, 11)] + [">10"]
        rank_counts = (
            leaderboard_eval.groupby("estimated_rank_label", dropna=False)
            .size()
            .reset_index(name="comparison_rows")
        )
        rank_counts["estimated_rank_label"] = pd.Categorical(rank_counts["estimated_rank_label"], categories=rank_order, ordered=True)
        rank_counts = rank_counts.sort_values("estimated_rank_label")
        fig = px.bar(rank_counts, x="estimated_rank_label", y="comparison_rows", title="Estimated leaderboard rank distribution")
        fig.update_layout(xaxis_title="Estimated rank vs top-10", yaxis_title="Comparison rows")
        fig.show()


### Leaderboard Performance Overview
Comparison rows: **37** across **37** datasets  
Rows with leaderboard-comparable metrics: **37**  
Estimated in top-10: **35/37**  
Estimated #1: **5**

,dataset,leaderboard_metric_name,metric_kind,direction,best_model,best_workflow,best_value,best_value_raw_model_metric,best_value_metric_column,leaderboard_top1,leaderboard_top10_cutoff,known_reference_count,reference_sources,estimated_rank_label,in_top10_estimate,gap_to_top1,gap_to_top10_cutoff
0,tdc_hydrationfreeenergy_freesolv,RMSE,rmse,lower,CFA (Combinatorial Fusion),CFA fusion,0.644869,0.644869,test_rmse,0.654,1.261,6,manual_tdc_admet_reference,1,True,-0.009131,-0.616131
1,tdc_ppbr_az,MAE,mae,lower,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,7.329016,7.329016,test_mae,0.679,7.914,16,manual_tdc_admet_reference; tdc,3,True,6.650016,-0.584984
2,polaris_adme_fang_rppb_1,mean_squared_error,mse,lower,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,0.243599,0.493558,test_rmse,0.230,0.634,10,polaris,3,True,0.013599,-0.390401
3,tdc_skin_reaction,AUROC,roc_auc,higher,Ensemble (Weighted average (inverse train RMSE)),ensemble,0.774839,0.774839,test_roc_auc,0.741,0.499,11,manual_tdc_admet_reference,1,True,-0.033839,-0.275839
4,tdc_cyp2d6_veith,AUPRC,auprc,higher,Ensemble (Weighted average (inverse train RMSE)),ensemble,0.727009,0.727009,test_auprc,0.811,0.464,6,manual_tdc_admet_reference,2,True,0.083991,-0.263009
5,tdc_clintox,AUROC,roc_auc,higher,CatBoost,conventional,0.948884,0.948884,test_roc_auc,0.996,0.732,6,manual_tdc_admet_reference,2,True,0.047116,-0.216884
6,polaris_adme_fang_hppb_1,mean_squared_error,mse,lower,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,0.199498,0.446652,test_rmse,0.143,0.383,10,polaris,3,True,0.056498,-0.183502
7,tdc_cyp3a4_veith,AUPRC,auprc,higher,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,0.889038,0.889038,test_auprc,0.923,0.750,6,manual_tdc_admet_reference,2,True,0.033962,-0.139038
8,tdc_bioavailability_ma,AUROC,roc_auc,higher,CatBoost,conventional,0.776688,0.776688,test_roc_auc,0.748,0.640,8,manual_tdc_admet_reference,1,True,-0.028688,-0.136688
9,tdc_carcinogens_lagunin,AUROC,roc_auc,higher,AdaBoost,conventional,0.862626,0.862626,test_roc_auc,0.848,0.730,10,manual_tdc_admet_reference,1,True,-0.014626,-0.132626


In [61]:
# Run completeness and leaderboard-reference context

if RUN_STATUS is None or RUN_STATUS.empty:
    display_note("No per-dataset `run_status.json` files were found in this run directory.")
else:
    status_df = RUN_STATUS.copy()
    status_df["status"] = status_df["status"].astype(str)
    display_note("### Run completeness by dataset")
    display(status_df.sort_values(["status", "dataset"], ascending=[True, True]).reset_index(drop=True))

    status_counts = status_df.groupby("status", dropna=False).size().reset_index(name="datasets")
    fig = px.bar(status_counts, x="status", y="datasets", title="Run status counts")
    fig.update_layout(xaxis_title="Status", yaxis_title="Dataset count")
    fig.show()

ref = REFERENCE_LEADERBOARD_ROWS.copy() if "REFERENCE_LEADERBOARD_ROWS" in globals() and REFERENCE_LEADERBOARD_ROWS is not None else pd.DataFrame()
if ref.empty:
    display_note("No combined leaderboard/reference rows were available.")
else:
    if "is_top10_entry" in ref.columns:
        include_mask = ref["is_top10_entry"].isna() | ref["is_top10_entry"].apply(bool_like)
        ref = ref.loc[include_mask].copy()

    display_note("### Leaderboard/reference coverage")
    coverage_cols = [
        "dataset",
        "benchmark_id",
        "benchmark_suite",
        "leaderboard_metric_name",
        "leaderboard_metric_direction",
        "leaderboard_dataset_split",
        "rank",
        "model",
        "metric_value",
        "reference_source",
        "paper_short",
        "year",
    ]
    coverage_cols = [c for c in coverage_cols if c in ref.columns]
    display(ref[coverage_cols].head(50))

    ref["metric_value_numeric"] = pd.to_numeric(ref.get("metric_value_numeric", pd.Series(dtype=float)), errors="coerce")
    if "leaderboard_metric_name" in ref.columns:
        ref["metric_kind"] = ref["leaderboard_metric_name"].astype(str).apply(leaderboard_metric_kind)
    else:
        ref["metric_kind"] = ""
    if "leaderboard_metric_direction" in ref.columns:
        ref["direction"] = [leaderboard_direction(kind, explicit) for kind, explicit in zip(ref["metric_kind"], ref["leaderboard_metric_direction"])]
    else:
        ref["direction"] = ref["metric_kind"].apply(leaderboard_direction)

    coverage_rows = []
    group_cols = [c for c in ["dataset", "leaderboard_metric_name", "metric_kind", "direction"] if c in ref.columns]
    for keys, group in ref.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        payload = dict(zip(group_cols, keys))
        values = group["metric_value_numeric"].dropna().to_numpy(dtype=float)
        direction = str(payload.get("direction", "lower"))
        if len(values):
            sorted_values = np.sort(values)
            if direction == "higher":
                sorted_values = sorted_values[::-1]
            payload["reference_rows"] = int(len(group))
            payload["best_reference"] = float(sorted_values[0])
            payload["top10_or_worst_reference"] = float(sorted_values[min(9, len(sorted_values) - 1)])
        else:
            payload["reference_rows"] = int(len(group))
            payload["best_reference"] = np.nan
            payload["top10_or_worst_reference"] = np.nan
        if "reference_source" in group.columns:
            payload["reference_sources"] = "; ".join(sorted(group["reference_source"].dropna().astype(str).unique()))
        coverage_rows.append(payload)

    coverage_summary = pd.DataFrame(coverage_rows)
    if not coverage_summary.empty:
        display(coverage_summary.sort_values([c for c in ["dataset", "leaderboard_metric_name"] if c in coverage_summary.columns]).reset_index(drop=True))


### Run completeness by dataset

,dataset,status,checkpoint_stage,n_metrics_rows,started_at,completed_at,elapsed_seconds
0,chemml_cep_homo,completed,,20.0,,2026-04-24 16:33:56,942.579
1,chemml_organic_density,completed,,20.0,,2026-04-24 16:55:56,1320.199
2,esol_delaney,completed,,22.0,,2026-04-24 20:22:10,1063.510
3,freesolv_sampl,completed,,20.0,,2026-04-24 17:49:23,1019.320
4,lipophilicity,completed,,20.0,,2026-04-25 07:37:07,7962.171
5,poduam_pod_nc_std,completed,,22.0,,2026-04-24 23:55:13,3787.873
6,poduam_pod_rd_std,completed,,22.0,,2026-04-25 02:27:39,3667.893
7,polaris_adme_fang_hppb_1,completed,,22.0,,2026-04-24 22:52:05,2714.944
8,polaris_adme_fang_perm_1,completed,,20.0,,2026-04-25 03:03:08,2128.857
9,polaris_adme_fang_rclint_1,completed,,22.0,,2026-04-25 05:24:25,2809.578


### Leaderboard/reference coverage

,dataset,benchmark_id,benchmark_suite,leaderboard_metric_name,leaderboard_metric_direction,leaderboard_dataset_split,rank,model,metric_value,reference_source,paper_short,year
0,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,1,PrismNet,0.558 +/- 0.027,literature_static,NaN,NaN
1,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,2,HiGNN,0.570 +/- 0.061,literature_static,NaN,NaN
2,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,3,DMPNN,0.575 +/- 0.073,literature_static,NaN,NaN
3,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,4,MGPhyche,0.608 +/- 0.013,literature_static,NaN,NaN
4,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,5,DeeperGCN,0.615 +/- 0.044,literature_static,NaN,NaN
5,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,6,AutoQSAR (TabPFN),0.621,literature_static,NaN,NaN
6,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,7,FP-GNN,0.658 +/- 0.006,literature_static,NaN,NaN
7,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,8,MV-Mol,0.670 +/- 0.019,literature_static,NaN,NaN
8,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,9,KCL,0.732,literature_static,NaN,NaN
9,esol_delaney,esol_delaney,moleculenet,RMSE,NaN,scaffold,10,BiLSTM,0.743 +/- 0.038,literature_static,NaN,NaN


,dataset,leaderboard_metric_name,metric_kind,direction,reference_rows,best_reference,top10_or_worst_reference,reference_sources
0,esol_delaney,RMSE,rmse,lower,10,0.558,0.743,literature_static
1,lipophilicity,RMSE,rmse,lower,10,0.549,0.632,literature_static
2,poduam_pod_nc_std,RMSE,rmse,lower,2,0.550,0.730,literature_static
3,poduam_pod_rd_std,RMSE,rmse,lower,2,0.410,0.630,literature_static
4,polaris_adme_fang_hppb_1,mean_squared_error,mse,lower,10,0.143,0.383,polaris
5,polaris_adme_fang_perm_1,mean_squared_error,mse,lower,10,0.113,0.257,polaris
6,polaris_adme_fang_rclint_1,mean_squared_error,mse,lower,10,0.216,0.403,polaris
7,polaris_adme_fang_rppb_1,mean_squared_error,mse,lower,10,0.230,0.634,polaris
8,polaris_adme_fang_solu_1,mean_squared_error,mse,lower,10,0.222,0.329,polaris
9,tdc_ames,AUROC,roc_auc,higher,7,0.912,0.834,manual_tdc_admet_reference


### Cost vs value diagnostics across runs

This section compares all run folders under `benchmark_results/` and summarizes runtime cost vs performance value.

- Run-level value prefers leaderboard-relative metrics when available (`leaderboard_delta_top10_best`, then `leaderboard_delta_primary`) and falls back to held-out RMSE.
- Feature-selection scaling uses explicit selector-stage timings when available and otherwise uses a clearly labeled proxy (`selector timeout cap` or `time-to-first-model`).
        


In [62]:
# COST_VS_VALUE_DIAGNOSTICS_MARKER
# Cross-run cost/value diagnostics + feature-selection scaling diagnostics.

benchmark_root = RUN_DIR.parent if RUN_DIR.parent.name == "benchmark_results" else Path("./benchmark_results").resolve()
run_dirs = sorted([p for p in benchmark_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)


def load_run_metrics(run_dir):
    summary = standardize_metric_frame(read_optional_csv(run_dir / "summary_metrics.csv"))
    if summary is not None and not summary.empty:
        return summary

    parts = []
    for metrics_path in sorted(run_dir.glob("*/metrics.csv")):
        dataset_name = metrics_path.parent.name
        frame = standardize_metric_frame(pd.read_csv(metrics_path))
        if frame is None or frame.empty:
            continue
        if "dataset" not in frame.columns:
            frame["dataset"] = dataset_name
        parts.append(frame)

    if not parts:
        return None
    return pd.concat(parts, ignore_index=True)


def load_run_status(run_dir):
    rows = []
    for status_path in sorted(run_dir.glob("*/run_status.json")):
        payload = read_optional_json(status_path)
        if not isinstance(payload, dict):
            continue
        rows.append(
            {
                "dataset": status_path.parent.name,
                "status": payload.get("status", "unknown"),
                "checkpoint_stage": payload.get("checkpoint_stage", ""),
                "started_at": payload.get("started_at", ""),
                "completed_at": payload.get("completed_at", ""),
                "elapsed_seconds": payload.get("elapsed_seconds", np.nan),
            }
        )
    status_df = pd.DataFrame(rows)
    if not status_df.empty and "elapsed_seconds" in status_df.columns:
        status_df["elapsed_seconds"] = pd.to_numeric(status_df["elapsed_seconds"], errors="coerce")
    return status_df


def bool_like_series(series):
    if series is None:
        return pd.Series(dtype=bool)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y", "t"})


def parse_stage_runtime_seconds(stage_frame):
    if stage_frame is None or stage_frame.empty:
        return np.nan
    label_col = pick_column(stage_frame.columns, ["stage_label", "stage", "step", "name"])
    seconds_col = pick_column(stage_frame.columns, ["elapsed_seconds", "duration_seconds", "seconds", "elapsed"])
    if label_col is None or seconds_col is None:
        return np.nan

    labels = stage_frame[label_col].astype(str).str.lower()
    mask = labels.str.contains("feature") | labels.str.contains("select")
    selector_rows = stage_frame.loc[mask].copy()
    if selector_rows.empty:
        return np.nan

    selector_rows[seconds_col] = pd.to_numeric(selector_rows[seconds_col], errors="coerce")
    return float(selector_rows[seconds_col].sum()) if selector_rows[seconds_col].notna().any() else np.nan


run_rows = []
dataset_cost_value_rows = []
selector_rows = []

for run_dir in run_dirs:
    run_name = run_dir.name
    run_metrics = load_run_metrics(run_dir)
    if run_metrics is None or run_metrics.empty:
        continue

    run_metrics = run_metrics.copy()
    rmse_col_run = metric_column(run_metrics, "AUTO", metric_hint="rmse")
    if rmse_col_run is None:
        continue

    run_metrics[rmse_col_run] = pd.to_numeric(run_metrics[rmse_col_run], errors="coerce")

    numeric_candidates = [
        "elapsed_seconds",
        "n_molecules",
        "n_train",
        "n_test",
        "selected_feature_count",
        "original_feature_count",
        "leaderboard_delta_top10_best",
        "leaderboard_delta_primary",
        "leaderboard_rank",
        "leaderboard_estimated_rank_vs_top10",
        "leaderboard_gap_to_top10_cutoff",
        "selector_elasticnet_timeout_seconds",
    ]
    for col in numeric_candidates:
        if col in run_metrics.columns:
            run_metrics[col] = pd.to_numeric(run_metrics[col], errors="coerce")

    run_status = load_run_status(run_dir)

    best_by_dataset = (
        run_metrics.sort_values(rmse_col_run, ascending=True, na_position="last")
        .groupby("dataset", as_index=False)
        .first()
    )

    dataset_elapsed_map = {}
    if not run_status.empty and "dataset" in run_status.columns and "elapsed_seconds" in run_status.columns:
        dataset_elapsed_map = run_status.set_index("dataset")["elapsed_seconds"].to_dict()

    status_completed_count = np.nan
    if not run_status.empty and "status" in run_status.columns:
        status_completed_count = int((run_status["status"].astype(str).str.lower() == "completed").sum())

    total_elapsed_seconds = np.nan
    if not run_status.empty and "elapsed_seconds" in run_status.columns and run_status["elapsed_seconds"].notna().any():
        total_elapsed_seconds = float(run_status["elapsed_seconds"].sum())

    if not np.isfinite(total_elapsed_seconds):
        if "dataset" in run_metrics.columns and "elapsed_seconds" in run_metrics.columns:
            elapsed_proxy = run_metrics.groupby("dataset", dropna=False)["elapsed_seconds"].max()
            if elapsed_proxy.notna().any():
                total_elapsed_seconds = float(elapsed_proxy.sum())

    delta_col = pick_column(best_by_dataset.columns, ["leaderboard_delta_top10_best", "leaderboard_delta_primary"])
    rank_col = pick_column(best_by_dataset.columns, ["leaderboard_estimated_rank_vs_top10", "leaderboard_rank"])
    top10_col = pick_column(best_by_dataset.columns, ["leaderboard_estimated_in_top10", "leaderboard_in_top10"])

    delta_values = pd.to_numeric(best_by_dataset[delta_col], errors="coerce") if delta_col is not None else pd.Series(dtype=float)
    rank_values = pd.to_numeric(best_by_dataset[rank_col], errors="coerce") if rank_col is not None else pd.Series(dtype=float)

    top10_rate = np.nan
    if top10_col is not None:
        bool_series = best_by_dataset[top10_col].astype(str).str.lower().map({"true": True, "false": False})
        if bool_series.notna().any():
            top10_rate = float(bool_series.mean())

    run_rows.append(
        {
            "run_name": run_name,
            "run_mtime": pd.Timestamp(run_dir.stat().st_mtime, unit="s"),
            "datasets": int(best_by_dataset["dataset"].nunique()),
            "completed_datasets": status_completed_count,
            "models": int(run_metrics["model"].nunique()) if "model" in run_metrics.columns else np.nan,
            "total_elapsed_seconds": total_elapsed_seconds,
            "total_elapsed_hours": total_elapsed_seconds / 3600.0 if np.isfinite(total_elapsed_seconds) else np.nan,
            "mean_best_rmse": float(best_by_dataset[rmse_col_run].mean(skipna=True)),
            "median_best_rmse": float(best_by_dataset[rmse_col_run].median(skipna=True)),
            "mean_leaderboard_delta": float(delta_values.mean(skipna=True)) if len(delta_values) and delta_values.notna().any() else np.nan,
            "median_leaderboard_delta": float(delta_values.median(skipna=True)) if len(delta_values) and delta_values.notna().any() else np.nan,
            "mean_estimated_rank": float(rank_values.mean(skipna=True)) if len(rank_values) and rank_values.notna().any() else np.nan,
            "top10_hit_rate": top10_rate,
            "value_metric_used": delta_col if delta_col is not None else rmse_col_run,
        }
    )

    # Explicit selector stage timings (if emitted by future benchmark runs)
    explicit_selector_seconds_by_dataset = {}
    step_runtime_summary = read_optional_csv(run_dir / "step_runtime_summary.csv")
    if step_runtime_summary is not None and not step_runtime_summary.empty:
        ds_col = pick_column(step_runtime_summary.columns, ["dataset", "dataset_name"])
        label_col = pick_column(step_runtime_summary.columns, ["stage_label", "stage", "step", "name"])
        sec_col = pick_column(step_runtime_summary.columns, ["elapsed_seconds", "duration_seconds", "seconds", "elapsed"])
        if ds_col is not None and label_col is not None and sec_col is not None:
            tmp = step_runtime_summary.copy()
            tmp[sec_col] = pd.to_numeric(tmp[sec_col], errors="coerce")
            tmp = tmp.loc[tmp[label_col].astype(str).str.lower().str.contains("feature|select", regex=True)].copy()
            if not tmp.empty:
                grouped = tmp.groupby(ds_col, dropna=False)[sec_col].sum(min_count=1)
                explicit_selector_seconds_by_dataset.update(grouped.to_dict())

    for step_path in run_dir.glob("*/step_runtime.csv"):
        dataset_name = step_path.parent.name
        if dataset_name in explicit_selector_seconds_by_dataset:
            continue
        stage_frame = read_optional_csv(step_path)
        seconds = parse_stage_runtime_seconds(stage_frame)
        if np.isfinite(seconds):
            explicit_selector_seconds_by_dataset[dataset_name] = float(seconds)

    for dataset_name, dataset_frame in run_metrics.groupby("dataset", dropna=False):
        dataset_frame = dataset_frame.copy()
        dataset_frame = dataset_frame.sort_values(rmse_col_run, ascending=True, na_position="last")
        best_row = dataset_frame.iloc[0]

        dataset_elapsed = dataset_elapsed_map.get(dataset_name, np.nan)
        if not np.isfinite(dataset_elapsed) and "elapsed_seconds" in dataset_frame.columns:
            dataset_elapsed = float(dataset_frame["elapsed_seconds"].max(skipna=True))

        selector_seconds = np.nan
        selector_seconds_source = "unavailable"

        if dataset_name in explicit_selector_seconds_by_dataset:
            selector_seconds = float(explicit_selector_seconds_by_dataset[dataset_name])
            selector_seconds_source = "stage_runtime"
        else:
            timed_out = False
            if "selector_timed_out" in dataset_frame.columns:
                timed_out = bool(bool_like_series(dataset_frame["selector_timed_out"]).any())

            timeout_cap = np.nan
            if "selector_elasticnet_timeout_seconds" in dataset_frame.columns:
                timeout_candidates = pd.to_numeric(dataset_frame["selector_elasticnet_timeout_seconds"], errors="coerce")
                if timeout_candidates.notna().any():
                    timeout_cap = float(timeout_candidates.dropna().iloc[0])

            if timed_out and np.isfinite(timeout_cap):
                selector_seconds = timeout_cap
                selector_seconds_source = "selector_timeout_cap"
            elif "elapsed_seconds" in dataset_frame.columns:
                first_elapsed = float(pd.to_numeric(dataset_frame["elapsed_seconds"], errors="coerce").min(skipna=True))
                if np.isfinite(first_elapsed):
                    selector_seconds = first_elapsed
                    selector_seconds_source = "first_model_elapsed_proxy"

        n_molecules_value = best_row.get("n_molecules", np.nan)
        n_train_value = best_row.get("n_train", np.nan)
        n_test_value = best_row.get("n_test", np.nan)

        selector_rows.append(
            {
                "run_name": run_name,
                "dataset": dataset_name,
                "n_molecules": pd.to_numeric(n_molecules_value, errors="coerce"),
                "n_train": pd.to_numeric(n_train_value, errors="coerce"),
                "n_test": pd.to_numeric(n_test_value, errors="coerce"),
                "selector_seconds": pd.to_numeric(selector_seconds, errors="coerce"),
                "selector_seconds_source": selector_seconds_source,
                "selector_timed_out": bool(bool_like_series(dataset_frame.get("selector_timed_out", pd.Series([False]))).any()),
                "selector_method": best_row.get("selector_method", ""),
                "selected_feature_count": pd.to_numeric(best_row.get("selected_feature_count", np.nan), errors="coerce"),
                "original_feature_count": pd.to_numeric(best_row.get("original_feature_count", np.nan), errors="coerce"),
            }
        )

        dataset_cost_value_rows.append(
            {
                "run_name": run_name,
                "dataset": dataset_name,
                "dataset_elapsed_seconds": pd.to_numeric(dataset_elapsed, errors="coerce"),
                "best_rmse": pd.to_numeric(best_row.get(rmse_col_run, np.nan), errors="coerce"),
                "leaderboard_delta_top10_best": pd.to_numeric(best_row.get("leaderboard_delta_top10_best", np.nan), errors="coerce"),
                "leaderboard_delta_primary": pd.to_numeric(best_row.get("leaderboard_delta_primary", np.nan), errors="coerce"),
                "leaderboard_estimated_rank_vs_top10": pd.to_numeric(best_row.get("leaderboard_estimated_rank_vs_top10", np.nan), errors="coerce"),
                "n_molecules": pd.to_numeric(best_row.get("n_molecules", np.nan), errors="coerce"),
            }
        )

RUN_COST_VALUE = pd.DataFrame(run_rows)
DATASET_COST_VALUE = pd.DataFrame(dataset_cost_value_rows)
SELECTOR_SCALING = pd.DataFrame(selector_rows)

if RUN_COST_VALUE.empty:
    display_note("No benchmark runs were detected under `benchmark_results/` with usable metrics.")
else:
    run_table = RUN_COST_VALUE.copy().sort_values("run_mtime", ascending=False)

    if not SELECTOR_SCALING.empty:
        selector_agg = (
            SELECTOR_SCALING.groupby("run_name", dropna=False)
            .agg(
                selector_timeout_rate=("selector_timed_out", "mean"),
                mean_selector_seconds=("selector_seconds", "mean"),
                median_selector_seconds=("selector_seconds", "median"),
                selector_rows=("dataset", "size"),
            )
            .reset_index()
        )
        run_table = run_table.merge(selector_agg, on="run_name", how="left")

    display_note("### Run-level cost vs value")
    display(
        run_table[
            [
                col
                for col in [
                    "run_name",
                    "run_mtime",
                    "datasets",
                    "completed_datasets",
                    "models",
                    "total_elapsed_hours",
                    "mean_best_rmse",
                    "mean_leaderboard_delta",
                    "mean_estimated_rank",
                    "top10_hit_rate",
                    "selector_timeout_rate",
                    "mean_selector_seconds",
                    "value_metric_used",
                ]
                if col in run_table.columns
            ]
        ].reset_index(drop=True)
    )

    run_plot_df = run_table.copy()
    if run_plot_df["total_elapsed_hours"].notna().sum() >= 2:
        value_col = "mean_leaderboard_delta" if run_plot_df["mean_leaderboard_delta"].notna().sum() >= 2 else "mean_best_rmse"
        fig = px.scatter(
            run_plot_df,
            x="total_elapsed_hours",
            y=value_col,
            color="run_name",
            hover_data=[c for c in ["datasets", "completed_datasets", "models", "mean_selector_seconds", "selector_timeout_rate"] if c in run_plot_df.columns],
            title=f"Run-level cost vs value ({value_col})",
        )
        fig.update_layout(xaxis_title="Total runtime across datasets (hours)", yaxis_title=value_col)
        fig.show()

    if not DATASET_COST_VALUE.empty:
        dataset_plot = DATASET_COST_VALUE.copy()
        dataset_plot["dataset_elapsed_hours"] = dataset_plot["dataset_elapsed_seconds"] / 3600.0
        value_col_dataset = (
            "leaderboard_delta_top10_best"
            if dataset_plot["leaderboard_delta_top10_best"].notna().sum() >= 3
            else "best_rmse"
        )
        fig = px.scatter(
            dataset_plot,
            x="dataset_elapsed_hours",
            y=value_col_dataset,
            color="run_name",
            hover_data=[c for c in ["dataset", "n_molecules", "leaderboard_estimated_rank_vs_top10"] if c in dataset_plot.columns],
            title=f"Dataset-level runtime vs value ({value_col_dataset})",
        )
        fig.update_layout(xaxis_title="Dataset runtime (hours)", yaxis_title=value_col_dataset)
        fig.show()


# Current-run model-level cost vs relative performance.
MODEL_COST_VALUE = pd.DataFrame()
model_cost_source = SUMMARY_ANALYSIS_METRICS.copy()
model_cost_error_col = pick_column(model_cost_source.columns, ["error", "Error"])
if model_cost_error_col is not None:
    model_cost_source = model_cost_source.loc[
        model_cost_source[model_cost_error_col].isna()
        | (model_cost_source[model_cost_error_col].astype(str).str.strip() == "")
    ].copy()

model_cost_source = add_analysis_metric_columns(model_cost_source, rmse_col=RMSE_COL, classification_col=BALANCED_ACCURACY_COL)

if "elapsed_seconds" in model_cost_source.columns and not model_cost_source.empty:
    model_cost_source["elapsed_seconds"] = pd.to_numeric(model_cost_source["elapsed_seconds"], errors="coerce")
    model_cost_source["analysis_metric_value"] = pd.to_numeric(model_cost_source["analysis_metric_value"], errors="coerce")
    model_cost_source["analysis_delta_from_best"] = pd.to_numeric(model_cost_source["analysis_delta_from_best"], errors="coerce")
    MODEL_COST_VALUE = model_cost_source.loc[
        model_cost_source["elapsed_seconds"].notna()
        & model_cost_source["analysis_metric_value"].notna()
        & model_cost_source["analysis_delta_from_best"].notna()
    ].copy()

if MODEL_COST_VALUE.empty:
    display_note("No per-model runtime rows with usable relative performance metrics were available.")
else:
    display_note("### Model cost vs relative performance")
    display(
        MODEL_COST_VALUE.sort_values(["task_kind", "analysis_delta_from_best", "elapsed_seconds"])
        [[
            col
            for col in [
                "dataset",
                "task_kind",
                "model",
                "workflow",
                "elapsed_seconds",
                "analysis_metric",
                "analysis_metric_value",
                "analysis_delta_from_best",
            ]
            if col in MODEL_COST_VALUE.columns
        ]]
        .reset_index(drop=True)
    )

    cost_fig = px.scatter(
        MODEL_COST_VALUE,
        x="elapsed_seconds",
        y="analysis_delta_from_best",
        color="model",
        symbol="task_kind",
        facet_col="task_kind",
        facet_col_wrap=2,
        hover_data=[c for c in ["dataset", "workflow", "analysis_metric", "analysis_metric_value"] if c in MODEL_COST_VALUE.columns],
        title="Model runtime vs delta from dataset-best performance",
    )
    cost_fig.update_xaxes(type="log", title="Model runtime (seconds, log scale)")
    cost_fig.update_yaxes(title="Delta from dataset-best metric (lower is better)")
    cost_fig.show()

    model_cost_summary = (
        MODEL_COST_VALUE.groupby(["model", "workflow", "task_kind"], dropna=False)
        .agg(
            datasets=("dataset", "nunique"),
            median_elapsed_seconds=("elapsed_seconds", "median"),
            mean_elapsed_seconds=("elapsed_seconds", "mean"),
            median_delta_from_best=("analysis_delta_from_best", "median"),
            mean_delta_from_best=("analysis_delta_from_best", "mean"),
        )
        .reset_index()
        .sort_values(["median_delta_from_best", "median_elapsed_seconds"], ascending=[True, True])
    )
    display_note("### Model cost/value summary")
    display(model_cost_summary.reset_index(drop=True))

    def publication_model_family(model_name, workflow_name=""):
        model_text = str(model_name or "").strip().lower()
        workflow_text = str(workflow_name or "").strip().lower()
        if model_text.startswith("cfa") or "cfa" in workflow_text:
            return "CFA"
        if model_text.startswith("ensemble") or "ensemble" in workflow_text:
            return "Ensemble"
        if "maplight + gnn" in workflow_text or "maplight + gnn" in model_text:
            return "MapLight+GNN"
        if "chemprop" in workflow_text or "chemprop" in model_text:
            return "Chemprop"
        if "tabpfn" in workflow_text or "tabpfn" in model_text:
            return "TabPFN"
        if "chemml" in workflow_text or "mlp" in model_text or "cnn" in model_text:
            return "Other neural baselines"
        return "Conventional ML"

    def current_run_hardware_note(model_family):
        gpu_available = bool(RUN_CONFIG.get("gpu_available", False)) if isinstance(RUN_CONFIG, dict) else False
        if gpu_available:
            return "GPU available in current run; CPU/GPU use depends on backend"
        if model_family in {"Chemprop", "TabPFN", "MapLight+GNN", "Other neural baselines"}:
            return "CPU in current run; backend may support GPU"
        return "CPU in current run"

    def trainable_parameter_note(model_family, exact_parameter_value=np.nan):
        if pd.notna(exact_parameter_value):
            return f"median recorded trainable parameters: {int(round(float(exact_parameter_value))):,}"
        return {
            "Conventional ML": "Not applicable (non-neural estimators)",
            "CFA": "Not applicable (fusion over fitted model outputs)",
            "Ensemble": "Not applicable for averaging/linear meta-model; base model parameters counted separately",
            "MapLight+GNN": "Not recorded in benchmark artifacts; CatBoost head plus transferred graph features",
            "Chemprop": "Model-dependent; not recorded in benchmark artifacts",
            "TabPFN": "Pretrained model; no task-specific fine-tuning parameters recorded",
            "Other neural baselines": "Model-dependent; not recorded in benchmark artifacts",
        }.get(model_family, "Not recorded")

    inference_col = pick_column(
        MODEL_COST_VALUE.columns,
        [
            "test_prediction_seconds",
            "test_inference_seconds",
            "prediction_seconds",
            "predict_seconds",
            "inference_seconds",
            "inference_elapsed_seconds",
            "prediction_elapsed_seconds",
        ],
    )
    inference_count_col = pick_column(MODEL_COST_VALUE.columns, ["test_inference_molecules", "n_test", "test_rows", "inference_molecules", "n_molecules"])
    parameter_count_col = pick_column(MODEL_COST_VALUE.columns, ["model_trainable_parameters", "trainable_parameters", "parameter_count", "n_parameters"])

    publication_cost_source = MODEL_COST_VALUE.copy()
    workflow_values = publication_cost_source["workflow"] if "workflow" in publication_cost_source.columns else pd.Series("", index=publication_cost_source.index)
    publication_cost_source["publication_model_family"] = [
        publication_model_family(model_name, workflow_name)
        for model_name, workflow_name in zip(publication_cost_source["model"], workflow_values)
    ]
    if inference_col is not None and inference_count_col is not None:
        publication_cost_source[inference_col] = pd.to_numeric(publication_cost_source[inference_col], errors="coerce")
        publication_cost_source[inference_count_col] = pd.to_numeric(publication_cost_source[inference_count_col], errors="coerce")
        publication_cost_source["inference_seconds_per_1000_molecules"] = (
            publication_cost_source[inference_col] / publication_cost_source[inference_count_col].replace(0, np.nan) * 1000.0
        )
    else:
        publication_cost_source["inference_seconds_per_1000_molecules"] = np.nan
    if parameter_count_col is not None:
        publication_cost_source[parameter_count_col] = pd.to_numeric(publication_cost_source[parameter_count_col], errors="coerce")
        publication_cost_source["recorded_trainable_parameters"] = publication_cost_source[parameter_count_col]
    else:
        publication_cost_source["recorded_trainable_parameters"] = np.nan

    autoqsar_publication_cost = (
        publication_cost_source.groupby("publication_model_family", dropna=False)
        .agg(
            source=("publication_model_family", lambda _: "AutoQSAR current benchmark run"),
            datasets=("dataset", "nunique"),
            model_rows=("model", "size"),
            median_wall_clock_seconds=("elapsed_seconds", "median"),
            mean_wall_clock_seconds=("elapsed_seconds", "mean"),
            median_inference_seconds_per_1000_molecules=("inference_seconds_per_1000_molecules", "median"),
            median_recorded_trainable_parameters=("recorded_trainable_parameters", "median"),
        )
        .reset_index()
        .rename(columns={"publication_model_family": "model_family"})
    )
    autoqsar_publication_cost["hardware"] = autoqsar_publication_cost["model_family"].map(current_run_hardware_note)
    autoqsar_publication_cost["trainable_parameters"] = autoqsar_publication_cost.apply(
        lambda row: trainable_parameter_note(row["model_family"], row.get("median_recorded_trainable_parameters", np.nan)),
        axis=1,
    )
    autoqsar_publication_cost["published_cost_comparison"] = "Measured in this benchmark run; inference-only timing/parameter fields are populated when emitted by benchmark artifacts"

    external_cost_rows = pd.DataFrame(
        [
            {
                "model_family": "MolGPS",
                "source": "Published comparator",
                "datasets": np.nan,
                "model_rows": np.nan,
                "median_wall_clock_seconds": np.nan,
                "mean_wall_clock_seconds": np.nan,
                "median_inference_seconds_per_1000_molecules": np.nan,
                "hardware": "GPU pretraining/inference reported in literature",
                "trainable_parameters": "~3B parameters",
                "published_cost_comparison": "Foundation-scale graph model; include exact runtime from cited paper if used in manuscript table",
            },
            {
                "model_family": "MolE",
                "source": "Published comparator",
                "datasets": np.nan,
                "model_rows": np.nan,
                "median_wall_clock_seconds": np.nan,
                "mean_wall_clock_seconds": np.nan,
                "median_inference_seconds_per_1000_molecules": np.nan,
                "hardware": "GPU pretraining reported in literature",
                "trainable_parameters": "~100M parameters",
                "published_cost_comparison": "Pretrained on ~842M molecules; include exact runtime from cited paper if used in manuscript table",
            },
            {
                "model_family": "ADMET-AI",
                "source": "Published comparator",
                "datasets": np.nan,
                "model_rows": np.nan,
                "median_wall_clock_seconds": np.nan,
                "mean_wall_clock_seconds": np.nan,
                "median_inference_seconds_per_1000_molecules": np.nan,
                "hardware": "GPU-capable Chemprop-RDKit deployment",
                "trainable_parameters": "Chemprop-RDKit; exact parameter count not recorded here",
                "published_cost_comparison": "Published throughput reference: ~3.1 hours for 1M molecules",
            },
        ]
    )

    PUBLICATION_COST_TABLE = pd.concat([autoqsar_publication_cost, external_cost_rows], ignore_index=True, sort=False)
    display_note("### Publication-grade computational cost comparison")
    display(
        PUBLICATION_COST_TABLE[
            [
                "model_family",
                "source",
                "datasets",
                "model_rows",
                "median_wall_clock_seconds",
                "mean_wall_clock_seconds",
                "hardware",
                "trainable_parameters",
                "median_recorded_trainable_parameters",
                "median_inference_seconds_per_1000_molecules",
                "published_cost_comparison",
            ]
        ].sort_values(["source", "model_family"]).reset_index(drop=True)
    )
    if inference_col is None:
        display_note(
            "Inference-only timing columns were not found in the current benchmark artifacts. "
            "The table keeps the inference-time field explicit so future runs can populate it without changing the manuscript table schema."
        )

    # Paired statistical comparison of cost-performance effectiveness.
    effectiveness_df = MODEL_COST_VALUE.copy()
    effectiveness_df["elapsed_seconds"] = pd.to_numeric(effectiveness_df["elapsed_seconds"], errors="coerce")
    effectiveness_df["analysis_delta_from_best"] = pd.to_numeric(effectiveness_df["analysis_delta_from_best"], errors="coerce")
    effectiveness_df = effectiveness_df.loc[
        effectiveness_df["elapsed_seconds"].gt(0)
        & effectiveness_df["analysis_delta_from_best"].notna()
    ].copy()

    if effectiveness_df.empty:
        display_note("No rows were available for cost-performance statistical testing.")
    else:
        effectiveness_df["log_elapsed_seconds"] = np.log10(effectiveness_df["elapsed_seconds"].clip(lower=1e-9))
        effectiveness_df["performance_percentile"] = effectiveness_df.groupby("dataset")["analysis_delta_from_best"].rank(method="average", pct=True, ascending=True)
        effectiveness_df["runtime_percentile"] = effectiveness_df.groupby("dataset")["log_elapsed_seconds"].rank(method="average", pct=True, ascending=True)
        effectiveness_df["cost_performance_score"] = 0.75 * effectiveness_df["performance_percentile"] + 0.25 * effectiveness_df["runtime_percentile"]
        effectiveness_df["cost_performance_ratio"] = effectiveness_df["analysis_delta_from_best"] / effectiveness_df["elapsed_seconds"].clip(lower=1e-9)

        effectiveness_summary = (
            effectiveness_df.groupby(["model", "workflow"], dropna=False)
            .agg(
                datasets=("dataset", "nunique"),
                median_cost_performance_score=("cost_performance_score", "median"),
                mean_cost_performance_score=("cost_performance_score", "mean"),
                median_performance_percentile=("performance_percentile", "median"),
                median_runtime_percentile=("runtime_percentile", "median"),
                median_delta_from_best=("analysis_delta_from_best", "median"),
                median_elapsed_seconds=("elapsed_seconds", "median"),
            )
            .reset_index()
            .sort_values(["median_cost_performance_score", "median_delta_from_best", "median_elapsed_seconds"], ascending=[True, True, True])
        )
        display_note("### Cost-performance effectiveness summary")
        display(effectiveness_summary.reset_index(drop=True))

        min_datasets_for_test = max(5, int(math.ceil(0.50 * effectiveness_df["dataset"].nunique())))
        eligible_models = (
            effectiveness_df.groupby("model")["dataset"]
            .nunique()
            .loc[lambda s: s >= min_datasets_for_test]
            .index
            .tolist()
        )

        score_matrix = effectiveness_df.loc[effectiveness_df["model"].isin(eligible_models)].pivot_table(
            index="dataset",
            columns="model",
            values="cost_performance_score",
            aggfunc="mean",
        )
        complete_matrix = score_matrix.dropna(axis=0, how="any")

        stat_notes = []
        test_result_rows = []
        pairwise_rows = []
        best_effective_model = None

        try:
            from scipy import stats
        except Exception as exc:
            stats = None
            stat_notes.append(f"SciPy was not available, so formal Friedman/Wilcoxon tests were skipped: {exc}")

        if stats is None:
            display_note("### Cost-performance statistical test")
            display_note(" ".join(stat_notes))
        elif complete_matrix.shape[0] < 5 or complete_matrix.shape[1] < 3:
            display_note("### Cost-performance statistical test")
            display_note(
                "Not enough paired complete model-by-dataset scores for a Friedman test "
                f"(datasets={complete_matrix.shape[0]}, models={complete_matrix.shape[1]})."
            )
        else:
            friedman_stat, friedman_p = stats.friedmanchisquare(*[complete_matrix[col].to_numpy(dtype=float) for col in complete_matrix.columns])
            test_result_rows.append(
                {
                    "test": "Friedman repeated-measures test",
                    "datasets": int(complete_matrix.shape[0]),
                    "models": int(complete_matrix.shape[1]),
                    "statistic": float(friedman_stat),
                    "p_value": float(friedman_p),
                    "score": "0.75*within-dataset performance percentile + 0.25*within-dataset runtime percentile; lower is better",
                }
            )

            model_medians = complete_matrix.median(axis=0).sort_values(ascending=True)
            best_effective_model = str(model_medians.index[0])
            best_scores = complete_matrix[best_effective_model]

            raw_pairwise = []
            for model_name in model_medians.index[1:]:
                candidate_scores = complete_matrix[model_name]
                diff = candidate_scores - best_scores
                if np.allclose(diff.to_numpy(dtype=float), 0.0, equal_nan=False):
                    stat_value, p_value = 0.0, 1.0
                else:
                    stat_value, p_value = stats.wilcoxon(candidate_scores, best_scores, alternative="greater", zero_method="wilcox")
                raw_pairwise.append(
                    {
                        "model": str(model_name),
                        "comparison": f"{model_name} worse than {best_effective_model}",
                        "datasets": int(len(diff)),
                        "median_score": float(candidate_scores.median()),
                        "median_score_delta_vs_best": float(diff.median()),
                        "wilcoxon_statistic": float(stat_value),
                        "raw_p_value": float(p_value),
                    }
                )

            if raw_pairwise:
                order = sorted(range(len(raw_pairwise)), key=lambda i: raw_pairwise[i]["raw_p_value"])
                adjusted = [np.nan] * len(raw_pairwise)
                running_max = 0.0
                m = len(raw_pairwise)
                for rank, row_idx in enumerate(order, start=1):
                    holm_value = min(1.0, raw_pairwise[row_idx]["raw_p_value"] * (m - rank + 1))
                    running_max = max(running_max, holm_value)
                    adjusted[row_idx] = running_max
                for row, adj in zip(raw_pairwise, adjusted):
                    row["holm_adjusted_p_value"] = float(adj)
                    row["significantly_worse_than_best_alpha_0_05"] = bool(adj < 0.05)
                pairwise_rows = raw_pairwise

            display_note("### Cost-performance statistical test")
            display(pd.DataFrame(test_result_rows))
            if pairwise_rows:
                pairwise_df = pd.DataFrame(pairwise_rows).sort_values(["holm_adjusted_p_value", "median_score_delta_vs_best"], ascending=[True, False])
                display_note("Pairwise Wilcoxon tests vs the best median cost-performance model")
                display(pairwise_df.reset_index(drop=True))

            recommendation_source = effectiveness_summary.copy()
            recommendation_source = recommendation_source.loc[recommendation_source["model"].isin(complete_matrix.columns)].copy()
            if best_effective_model is not None and not recommendation_source.empty:
                best_row = recommendation_source.loc[recommendation_source["model"].eq(best_effective_model)].head(1)
                if not best_row.empty:
                    row = best_row.iloc[0]
                    display_note(
                        "Best single-model choice by paired cost-performance score: "
                        f"**{best_effective_model}** "
                        f"(median score={row['median_cost_performance_score']:.3f}, "
                        f"median delta={row['median_delta_from_best']:.3g}, "
                        f"median runtime={row['median_elapsed_seconds']:.1f}s). "
                        "Lower score balances better relative performance and faster runtime within each dataset."
                    )

if SELECTOR_SCALING.empty:
    display_note("No selector diagnostics could be inferred for these runs.")
else:
    selector_df = SELECTOR_SCALING.copy()
    selector_df["n_molecules"] = pd.to_numeric(selector_df["n_molecules"], errors="coerce")
    selector_df["selector_seconds"] = pd.to_numeric(selector_df["selector_seconds"], errors="coerce")
    selector_df["selector_hours"] = selector_df["selector_seconds"] / 3600.0

    display_note("### Feature-selection time vs dataset size")
    display(
        selector_df.sort_values("selector_seconds", ascending=False)
        [[
            col
            for col in [
                "run_name",
                "dataset",
                "n_molecules",
                "selector_seconds",
                "selector_hours",
                "selector_seconds_source",
                "selector_timed_out",
                "selector_method",
                "selected_feature_count",
                "original_feature_count",
            ]
            if col in selector_df.columns
        ]]
        .reset_index(drop=True)
    )

    fit_df = selector_df.loc[
        selector_df["n_molecules"].gt(0)
        & selector_df["selector_seconds"].gt(0)
    ].copy()

    if len(fit_df) >= 3:
        log_x = np.log10(fit_df["n_molecules"].to_numpy(dtype=float))
        log_y = np.log10(fit_df["selector_seconds"].to_numpy(dtype=float))
        corr = float(np.corrcoef(log_x, log_y)[0, 1])
        slope, intercept = np.polyfit(log_x, log_y, 1)
        print(
            "Selector scaling fit (log10 seconds vs log10 n_molecules): "
            f"slope={slope:.3f}, intercept={intercept:.3f}, corr={corr:.3f}"
        )

        fig = px.scatter(
            fit_df,
            x="n_molecules",
            y="selector_seconds",
            color="run_name",
            symbol="selector_seconds_source",
            hover_data=[c for c in ["dataset", "selected_feature_count", "selector_timed_out"] if c in fit_df.columns],
            title="Feature-selection cost scaling across runs",
        )
        fig.update_xaxes(type="log", title="Dataset size (n_molecules, log scale)")
        fig.update_yaxes(type="log", title="Selector runtime (seconds, log scale)")
        fig.show()
    else:
        display_note("Insufficient positive selector-time samples to fit a size-scaling trend.")

    source_counts = (
        selector_df.groupby("selector_seconds_source", dropna=False)
        .size()
        .reset_index(name="datasets")
        .sort_values("datasets", ascending=False)
    )
    display_note("Selector-time source diagnostics")
    display(source_counts)

    if "first_model_elapsed_proxy" in source_counts["selector_seconds_source"].astype(str).tolist():
        display_note(
            "`first_model_elapsed_proxy` indicates an estimated upper bound for selector time when explicit stage timings were not saved "
            "for that run."
        )
        


### Run-level cost vs value

,run_name,run_mtime,datasets,completed_datasets,models,total_elapsed_hours,mean_best_rmse,mean_leaderboard_delta,mean_estimated_rank,top10_hit_rate,selector_timeout_rate,mean_selector_seconds,value_metric_used
0,all_benchmarks_no_unimol_20260425_223505,2026-04-30 17:23:57.256783484,44,44,25,37.312244,5.573154,-0.050699,4.388889,0.833333,0.0,1480.793386,leaderboard_delta_top10_best


### Model cost vs relative performance

,dataset,task_kind,model,workflow,elapsed_seconds,analysis_metric,analysis_metric_value,analysis_delta_from_best
0,tdc_cyp3a4_substrate_carbonmangels,classification,LogisticRegression,conventional,35.506,test_auprc,0.716619,0.000000
1,tdc_carcinogens_lagunin,classification,AdaBoost,conventional,66.059,test_roc_auc,0.862626,0.000000
2,tdc_bioavailability_ma,classification,CatBoost,conventional,155.509,test_roc_auc,0.776688,0.000000
3,tdc_pgp_broccatelli,classification,HistGradientBoosting,conventional,206.544,test_roc_auc,0.924020,0.000000
4,tdc_herg,classification,AdaBoost,conventional,255.366,test_roc_auc,0.860825,0.000000
...,...,...,...,...,...,...,...,...
782,tdc_half_life_obach,regression,Chemprop v2 (AttentiveFP + Selected descriptor...,Chemprop v2,982.339,test_rmse,34.500207,17.289241
783,tdc_half_life_obach,regression,ElasticNetCV,conventional,210.167,test_rmse,36.479861,19.268894
784,poduam_pod_nc_std,regression,"Chemprop v2 (AttentiveFP, ensemble=1)",Chemprop v2,3045.986,test_rmse,20.394337,19.694487
785,tdc_half_life_obach,regression,Tabular CNN,conventional,528.881,test_rmse,40.262106,23.051140


### Model cost/value summary

...

### Feature-selection time vs dataset size

,run_name,dataset,n_molecules,selector_seconds,selector_hours,selector_seconds_source,selector_timed_out,selector_method,selected_feature_count,original_feature_count
0,all_benchmarks_no_unimol_20260425_223505,tdc_cyp2d6_veith,13130,5396.254,1.498959,stage_runtime,False,random_forest_importance_fallback,512,10115
1,all_benchmarks_no_unimol_20260425_223505,lipophilicity,4200,5135.859,1.426627,stage_runtime,False,elasticnet_cv,512,10115
2,all_benchmarks_no_unimol_20260425_223505,tdc_cyp2c19_veith,12665,5050.329,1.402869,stage_runtime,False,random_forest_importance_fallback,512,10115
3,all_benchmarks_no_unimol_20260425_223505,tdc_herg_karim,13445,4953.110,1.375864,stage_runtime,False,random_forest_importance_fallback,512,10115
4,all_benchmarks_no_unimol_20260425_223505,tdc_cyp2c9_veith,12092,4552.423,1.264562,stage_runtime,False,random_forest_importance_fallback,512,10115
5,all_benchmarks_no_unimol_20260425_223505,tdc_cyp1a2_veith,12579,4361.116,1.211421,stage_runtime,False,random_forest_importance_fallback,512,10115
6,all_benchmarks_no_unimol_20260425_223505,tdc_tox21,7258,4114.586,1.142941,stage_runtime,False,random_forest_importance_fallback,512,10115
7,all_benchmarks_no_unimol_20260425_223505,tdc_cyp3a4_veith,12328,3975.231,1.104231,stage_runtime,False,random_forest_importance_fallback,512,10115
8,all_benchmarks_no_unimol_20260425_223505,tdc_lipophilicity_astrazeneca,4200,3389.606,0.941557,stage_runtime,False,elasticnet_cv,512,10115
9,all_benchmarks_no_unimol_20260425_223505,tdc_ppbr_az,2790,3255.492,0.904303,stage_runtime,False,elasticnet_cv,512,10115


Selector scaling fit (log10 seconds vs log10 n_molecules): slope=1.091, intercept=-0.751, corr=0.918


Selector-time source diagnostics

,selector_seconds_source,datasets
0,stage_runtime,44


In [63]:
metric_plot_source = SUMMARY_ANALYSIS_METRICS.copy()
error_col = pick_column(metric_plot_source.columns, ["error", "Error"])
if error_col is not None:
    metric_plot_source = metric_plot_source.loc[
        metric_plot_source[error_col].isna() | (metric_plot_source[error_col].astype(str).str.strip() == "")
    ].copy()
metric_plot_source = add_analysis_metric_columns(metric_plot_source, rmse_col=RMSE_COL, classification_col=BALANCED_ACCURACY_COL)
metric_plot_source["analysis_metric_value"] = pd.to_numeric(metric_plot_source["analysis_metric_value"], errors="coerce")
metric_plot_source["analysis_delta_from_best"] = pd.to_numeric(metric_plot_source["analysis_delta_from_best"], errors="coerce")
metric_plot_source = metric_plot_source.loc[metric_plot_source["analysis_metric_value"].notna()].copy()

if metric_plot_source.empty:
    display_note("No successful regression/classification metric rows were available for model performance plots.")
else:
    for task_kind, task_df in metric_plot_source.groupby("task_kind", sort=True):
        metric_names = sorted(task_df["analysis_metric"].dropna().astype(str).unique().tolist())
        metric_label = metric_names[0] if len(metric_names) == 1 else "primary metric"
        heatmap_df = task_df.pivot_table(index="dataset", columns="model", values="analysis_metric_value", aggfunc="mean")
        color_scale = "Viridis_r" if task_df["analysis_metric_direction"].astype(str).eq("lower").all() else "Viridis"
        fig = px.imshow(
            heatmap_df,
            aspect="auto",
            color_continuous_scale=color_scale,
            labels={"x": "Model", "y": "Dataset", "color": metric_label},
            text_auto='.3f',
            title=f"Held-out {metric_label} by dataset and model ({task_kind})",
        )
        fig.update_layout(height=max(450, 60 + 40 * len(heatmap_df.index)))
        fig.show()

        fig = px.box(
            task_df,
            x="model",
            y="analysis_metric_value",
            color="workflow" if "workflow" in task_df.columns else None,
            points="all",
            title=f"{metric_label} distribution across {task_kind} datasets",
        )
        fig.update_layout(xaxis_title="Model", yaxis_title=metric_label, xaxis_tickangle=45)
        fig.show()

    delta_df = metric_plot_source.loc[metric_plot_source["analysis_delta_from_best"].notna()].copy()

    if delta_df.empty:
        display_note("No valid rows available for delta-from-best model boxplot.")
    else:
        model_order = (
            delta_df.groupby("model", as_index=False)["analysis_delta_from_best"]
            .median()
            .sort_values("analysis_delta_from_best", ascending=True)["model"]
            .tolist()
        )

        positive_deltas = delta_df.loc[delta_df["analysis_delta_from_best"] > 0, "analysis_delta_from_best"]
        delta_floor = float(positive_deltas.min() * 0.5) if not positive_deltas.empty else 1e-12
        delta_floor = max(delta_floor, 1e-12)
        delta_df["delta_from_best_log10"] = np.log10(delta_df["analysis_delta_from_best"] + delta_floor)

        fig = px.box(
            delta_df,
            x="model",
            y="delta_from_best_log10",
            color="task_kind",
            points="all",
            category_orders={"model": model_order},
            title="Model delta from dataset-best model (RMSE or balanced accuracy, log10 scale)",
            hover_data=["dataset", "analysis_metric", "analysis_metric_value", "analysis_delta_from_best"],
        )
        fig.update_layout(
            xaxis_title="Model",
            yaxis_title=f"log10(delta from best + {delta_floor:.2e}); lower is better",
            xaxis_tickangle=45,
        )
        fig.show()


In [64]:
if PREDICTIONS is not None and not PREDICTIONS.empty:
    pred_df = PREDICTIONS.copy()
    dataset_col = pick_column(pred_df.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(pred_df.columns, ["model", "Model"])
    observed_col = pick_column(pred_df.columns, ["observed", "y_true", "actual"])
    predicted_col = pick_column(pred_df.columns, ["predicted", "y_pred", "prediction"])
    split_col = pick_column(pred_df.columns, ["split", "Split"])

    if all(col is not None for col in [dataset_col, model_col, observed_col, predicted_col]):
        pred_df = pred_df.rename(columns={
            dataset_col: "dataset",
            model_col: "model",
            observed_col: "observed",
            predicted_col: "predicted",
        })
        if split_col is not None:
            pred_df = pred_df.rename(columns={split_col: "split"})
            pred_df = pred_df.loc[pred_df["split"].astype(str).str.lower() == "test"].copy()

        pred_df["observed"] = pd.to_numeric(pred_df["observed"], errors="coerce")
        pred_df["predicted"] = pd.to_numeric(pred_df["predicted"], errors="coerce")
        pred_df = pred_df.loc[pred_df["observed"].notna() & pred_df["predicted"].notna()].copy()

        metrics_best = SUMMARY_METRICS.copy()
        metrics_error_col = pick_column(metrics_best.columns, ["error", "Error"])
        if metrics_error_col is not None:
            metrics_best = metrics_best.loc[
                metrics_best[metrics_error_col].isna() | (metrics_best[metrics_error_col].astype(str).str.strip() == "")
            ].copy()
        metrics_best[RMSE_COL] = pd.to_numeric(metrics_best[RMSE_COL], errors="coerce")
        metrics_best = metrics_best.loc[metrics_best[RMSE_COL].notna()].copy()

        if metrics_best.empty:
            display_note("No successful metric rows available to resolve best model per dataset for held-out scatter.")
        else:
            best_model_by_dataset = (
                metrics_best.sort_values(RMSE_COL, ascending=True)
                .groupby("dataset", as_index=False)
                .first()[["dataset", "model"]]
                .rename(columns={"model": "best_model"})
            )

            pred_df = pred_df.merge(best_model_by_dataset, on="dataset", how="inner")
            pred_df = pred_df.loc[pred_df["model"].astype(str) == pred_df["best_model"].astype(str)].copy()

            if pred_df.empty:
                display_note("No held-out prediction rows matched each dataset's best model.")
            else:
                if len(pred_df) > int(prediction_sample_cap):
                    dataset_count = max(1, pred_df["dataset"].nunique())
                    per_dataset_cap = max(1, int(prediction_sample_cap // dataset_count))
                    pred_df = (
                        pred_df.groupby("dataset", group_keys=False)
                        .apply(lambda frame: frame.sample(n=min(len(frame), per_dataset_cap), random_state=42))
                        .reset_index(drop=True)
                    )

                fig = px.scatter(
                    pred_df,
                    x="observed",
                    y="predicted",
                    color="best_model",
                    facet_col="dataset",
                    facet_col_wrap=3,
                    opacity=0.70,
                    title="Observed vs predicted (held-out rows) for each dataset's best model",
                    hover_data=["dataset", "best_model"],
                )
                fig.update_traces(marker=dict(size=5))
                fig.update_layout(legend_title_text="Best model")
                fig.show()
    else:
        display_note("Predictions table exists, but the expected observed/predicted columns were not found.")
else:
    display_note("No aggregate predictions table was found for this run.")

if GA_HISTORY is not None and not GA_HISTORY.empty:
    ga_df = GA_HISTORY.copy()
    dataset_col = pick_column(ga_df.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(ga_df.columns, ["model", "Model"])
    generation_col = pick_column(ga_df.columns, ["generation", "Generation"])
    best_col = pick_column(ga_df.columns, ["best_fitness", "best_rmse", "best_score", "Best fitness"])
    if all(col is not None for col in [dataset_col, model_col, generation_col, best_col]):
        ga_df = ga_df.rename(columns={dataset_col: "dataset", model_col: "model", generation_col: "generation", best_col: "best_value"})
        fig = px.line(ga_df, x="generation", y="best_value", color="model", facet_col="dataset", facet_col_wrap=3, title="GA convergence history")
        fig.update_layout(yaxis_title=best_col)
        fig.show()
    else:
        display_note("GA history exists, but the expected columns were not found for plotting.")


No aggregate predictions table was found for this run.

In [65]:
# Ensemble diagnostics: method-aware value-add vs best base model

ENSEMBLE_EPS = 1e-12
METHOD_CFA = "CFA (Combinatorial Fusion)"
METHOD_OOF = "OOF stacking (RidgeCV)"
METHOD_INV_RMSE = "Model average (inverse train RMSE)"
EXPECTED_ENSEMBLE_METHODS = [METHOD_CFA, METHOD_OOF, METHOD_INV_RMSE]
BASELINE_LABEL = "Best base model"


def classify_ensemble_method(model_name, workflow_name=""):
    model_text = str(model_name or "").strip().lower()
    workflow_text = str(workflow_name or "").strip().lower()

    if model_text.startswith("cfa (") or "cfa" in workflow_text:
        return METHOD_CFA

    if model_text.startswith("ensemble (") or "ensemble" in workflow_text:
        if "oof stacking" in model_text:
            return METHOD_OOF
        if "weighted average" in model_text and "inverse" in model_text and "rmse" in model_text:
            return METHOD_INV_RMSE
    return None


perf_df = SUMMARY_METRICS.copy()
error_col = pick_column(perf_df.columns, ["error", "Error"])
if error_col is not None:
    perf_df = perf_df.loc[perf_df[error_col].isna() | (perf_df[error_col].astype(str).str.strip() == "")].copy()

perf_df[RMSE_COL] = pd.to_numeric(perf_df[RMSE_COL], errors="coerce")
perf_df = perf_df.loc[perf_df[RMSE_COL].notna()].copy()

if "workflow" not in perf_df.columns:
    perf_df["workflow"] = ""

perf_df["ensemble_method"] = [
    classify_ensemble_method(model_name, workflow_name)
    for model_name, workflow_name in zip(perf_df["model"], perf_df["workflow"])
]

ensemble_df = perf_df.loc[perf_df["ensemble_method"].notna()].copy()
base_df = perf_df.loc[perf_df["ensemble_method"].isna()].copy()

if perf_df.empty:
    display_note("No valid RMSE rows available for ensemble-vs-base diagnostics.")
elif ensemble_df.empty or base_df.empty:
    display_note("Could not compare ensemble methods vs best base model because one side is missing.")
else:
    method_availability = (
        ensemble_df.groupby("ensemble_method", dropna=False)
        .agg(rows=("dataset", "size"), datasets=("dataset", "nunique"))
        .reset_index()
    )
    method_availability = (
        method_availability.set_index("ensemble_method")
        .reindex(EXPECTED_ENSEMBLE_METHODS)
        .reset_index()
        .rename(columns={"index": "ensemble_method"})
    )
    method_availability[["rows", "datasets"]] = method_availability[["rows", "datasets"]].fillna(0).astype(int)

    display_note("### Ensemble Method Coverage")
    display(method_availability)

    # Optional weight diagnostics when ensemble weight artifacts are present.
    ensemble_best_method_by_dataset = (
        ensemble_df.sort_values(RMSE_COL, ascending=True)
        .groupby("dataset", as_index=False)
        .first()[["dataset", "ensemble_method", "model"]]
        .rename(columns={"ensemble_method": "selected_ensemble_method", "model": "selected_ensemble_model"})
    )
    selected_method_lookup = dict(zip(ensemble_best_method_by_dataset["dataset"], ensemble_best_method_by_dataset["selected_ensemble_method"]))

    ensemble_weight_rows = []
    for weights_path in sorted(RUN_DIR.glob("*/ensemble_weights.csv")):
        dataset_name = weights_path.parent.name
        try:
            wdf = pd.read_csv(weights_path)
        except Exception:
            continue
        if wdf.empty:
            continue
        model_col = pick_column(wdf.columns, ["model", "Model"])
        weight_col = pick_column(wdf.columns, ["weight", "Weight"])
        abs_col = pick_column(wdf.columns, ["abs normalized contribution", "Abs normalized contribution", "abs_normalized_contribution"])
        workflow_col = pick_column(wdf.columns, ["workflow", "Workflow"])
        if model_col is None or weight_col is None:
            continue

        local = pd.DataFrame(
            {
                "dataset": dataset_name,
                "model": wdf[model_col].astype(str),
                "weight": pd.to_numeric(wdf[weight_col], errors="coerce"),
                "abs_normalized_contribution": pd.to_numeric(wdf[abs_col], errors="coerce") if abs_col is not None else np.nan,
                "workflow": wdf[workflow_col].astype(str) if workflow_col is not None else "",
                "selected_ensemble_method": selected_method_lookup.get(dataset_name, np.nan),
            }
        )
        local = local.loc[local["model"].str.strip().ne("") & local["weight"].notna()].copy()
        if not local.empty:
            ensemble_weight_rows.append(local)

    if ensemble_weight_rows:
        ensemble_weights_df = pd.concat(ensemble_weight_rows, ignore_index=True)
        ensemble_weights_df["abs_weight"] = ensemble_weights_df["weight"].abs()

        usage_summary = (
            ensemble_weights_df.groupby(["selected_ensemble_method", "model"], dropna=False)
            .agg(
                datasets=("dataset", "nunique"),
                total_rows=("dataset", "size"),
                mean_weight=("weight", "mean"),
                median_weight=("weight", "median"),
                mean_abs_weight=("abs_weight", "mean"),
                median_abs_weight=("abs_weight", "median"),
                max_abs_weight=("abs_weight", "max"),
                mean_abs_contribution=("abs_normalized_contribution", "mean"),
                median_abs_contribution=("abs_normalized_contribution", "median"),
            )
            .reset_index()
            .sort_values(["selected_ensemble_method", "mean_abs_contribution", "mean_abs_weight", "datasets"], ascending=[True, False, False, False])
        )

        display_note("### Ensemble Weight Usage Statistics (when `ensemble_weights.csv` is present)")
        display(usage_summary)

        weight_plot_df = ensemble_weights_df.copy()
        weight_plot_df["selected_ensemble_method"] = weight_plot_df["selected_ensemble_method"].fillna("Unknown")
        fig = px.box(
            weight_plot_df,
            x="model",
            y="weight",
            color="selected_ensemble_method",
            points="all",
            title="Distribution of ensemble weights by selected ensemble method",
        )
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(xaxis_title="Model", yaxis_title="Weight", xaxis_tickangle=45)
        fig.show()
    else:
        display_note("No `ensemble_weights.csv` files were found for this run; skipping weight diagnostics.")

    method_best = (
        ensemble_df.groupby(["dataset", "ensemble_method"], as_index=False)[RMSE_COL]
        .min()
        .rename(columns={RMSE_COL: "ensemble_rmse"})
    )
    base_best = (
        base_df.groupby("dataset", as_index=False)[RMSE_COL]
        .min()
        .rename(columns={RMSE_COL: "best_base_rmse"})
    )

    comparison = method_best.merge(base_best, on="dataset", how="inner")
    comparison["delta_rmse_ensemble_minus_best_base"] = comparison["ensemble_rmse"] - comparison["best_base_rmse"]
    comparison["relative_delta_vs_best_base"] = np.where(
        comparison["best_base_rmse"].abs() > ENSEMBLE_EPS,
        comparison["delta_rmse_ensemble_minus_best_base"] / comparison["best_base_rmse"],
        np.nan,
    )
    comparison["ensemble_beats_best_base"] = comparison["delta_rmse_ensemble_minus_best_base"] < -ENSEMBLE_EPS
    comparison["ensemble_ties_best_base"] = comparison["delta_rmse_ensemble_minus_best_base"].abs() <= ENSEMBLE_EPS

    if comparison.empty:
        display_note("No overlapping datasets with both ensemble-method and base-model RMSE values were found.")
    else:
        summary_rows = []
        for method_name in EXPECTED_ENSEMBLE_METHODS:
            local = comparison.loc[comparison["ensemble_method"] == method_name].copy()
            if local.empty:
                summary_rows.append(
                    {
                        "ensemble_method": method_name,
                        "datasets_compared": 0,
                        "ensemble_beats_base_count": 0,
                        "ensemble_ties_base_count": 0,
                        "ensemble_loses_to_base_count": 0,
                        "ensemble_beats_base_fraction": np.nan,
                        "mean_delta_rmse": np.nan,
                        "median_delta_rmse": np.nan,
                        "std_delta_rmse": np.nan,
                        "min_delta_rmse": np.nan,
                        "max_delta_rmse": np.nan,
                        "mean_relative_delta": np.nan,
                        "median_relative_delta": np.nan,
                    }
                )
                continue

            beats = int((local["delta_rmse_ensemble_minus_best_base"] < -ENSEMBLE_EPS).sum())
            ties = int((local["delta_rmse_ensemble_minus_best_base"].abs() <= ENSEMBLE_EPS).sum())
            losses = int((local["delta_rmse_ensemble_minus_best_base"] > ENSEMBLE_EPS).sum())
            total = int(len(local))

            summary_rows.append(
                {
                    "ensemble_method": method_name,
                    "datasets_compared": total,
                    "ensemble_beats_base_count": beats,
                    "ensemble_ties_base_count": ties,
                    "ensemble_loses_to_base_count": losses,
                    "ensemble_beats_base_fraction": beats / total if total else np.nan,
                    "mean_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].mean()),
                    "median_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].median()),
                    "std_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].std(ddof=1)) if total >= 2 else np.nan,
                    "min_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].min()),
                    "max_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].max()),
                    "mean_relative_delta": float(local["relative_delta_vs_best_base"].mean(skipna=True)),
                    "median_relative_delta": float(local["relative_delta_vs_best_base"].median(skipna=True)),
                }
            )

        method_vs_base_summary = pd.DataFrame(summary_rows).sort_values(
            ["ensemble_beats_base_fraction", "mean_delta_rmse"],
            ascending=[False, True],
        ).reset_index(drop=True)

        display_note("### Ensemble Method Delta vs Best Base Model (across datasets)")
        display(method_vs_base_summary)
        display(comparison.sort_values(["ensemble_method", "delta_rmse_ensemble_minus_best_base"], ascending=[True, True]).reset_index(drop=True))

        fig = px.box(
            comparison,
            x="ensemble_method",
            y="delta_rmse_ensemble_minus_best_base",
            color="ensemble_method",
            points="all",
            title="Ensemble RMSE delta vs best base model across datasets (by method)",
        )
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(
            xaxis_title="Ensemble method",
            yaxis_title="ensemble RMSE - best base RMSE (negative is better)",
            showlegend=False,
        )
        fig.show()

        # Pairwise superiority between ensemble methods on overlapping datasets.
        method_matrix = method_best.pivot(index="dataset", columns="ensemble_method", values="ensemble_rmse")
        pairwise_rows = []
        for i, method_a in enumerate(EXPECTED_ENSEMBLE_METHODS):
            if method_a not in method_matrix.columns:
                continue
            for method_b in EXPECTED_ENSEMBLE_METHODS[i + 1 :]:
                if method_b not in method_matrix.columns:
                    continue
                pair_df = method_matrix[[method_a, method_b]].dropna().copy()
                if pair_df.empty:
                    continue
                delta = pair_df[method_a] - pair_df[method_b]
                wins_a = int((delta < -ENSEMBLE_EPS).sum())
                wins_b = int((delta > ENSEMBLE_EPS).sum())
                ties = int((delta.abs() <= ENSEMBLE_EPS).sum())
                compared = int(len(pair_df))
                pairwise_rows.append(
                    {
                        "method_a": method_a,
                        "method_b": method_b,
                        "datasets_compared": compared,
                        "method_a_wins": wins_a,
                        "method_b_wins": wins_b,
                        "ties": ties,
                        "method_a_win_fraction": wins_a / compared if compared else np.nan,
                        "mean_delta_rmse_a_minus_b": float(delta.mean()),
                        "median_delta_rmse_a_minus_b": float(delta.median()),
                    }
                )

        if pairwise_rows:
            pairwise_df = pd.DataFrame(pairwise_rows).sort_values(
                ["datasets_compared", "method_a_win_fraction", "mean_delta_rmse_a_minus_b"],
                ascending=[False, False, True],
            ).reset_index(drop=True)

            superiority_score = {m: {"wins": 0, "losses": 0, "ties": 0, "comparisons": 0} for m in EXPECTED_ENSEMBLE_METHODS}
            for _, row in pairwise_df.iterrows():
                a = row["method_a"]
                b = row["method_b"]
                aw = int(row["method_a_wins"])
                bw = int(row["method_b_wins"])
                tw = int(row["ties"])
                superiority_score[a]["wins"] += aw
                superiority_score[a]["losses"] += bw
                superiority_score[a]["ties"] += tw
                superiority_score[a]["comparisons"] += int(row["datasets_compared"])
                superiority_score[b]["wins"] += bw
                superiority_score[b]["losses"] += aw
                superiority_score[b]["ties"] += tw
                superiority_score[b]["comparisons"] += int(row["datasets_compared"])

            superiority_table = pd.DataFrame(
                [
                    {
                        "ensemble_method": method_name,
                        "pairwise_wins": payload["wins"],
                        "pairwise_losses": payload["losses"],
                        "pairwise_ties": payload["ties"],
                        "pairwise_compared": payload["comparisons"],
                        "pairwise_win_fraction": payload["wins"] / payload["comparisons"] if payload["comparisons"] else np.nan,
                    }
                    for method_name, payload in superiority_score.items()
                ]
            ).sort_values(["pairwise_win_fraction", "pairwise_wins"], ascending=[False, False]).reset_index(drop=True)

            display_note("### Ensemble Method Pairwise Superiority (head-to-head)")
            display(pairwise_df)
            display(superiority_table)
        else:
            display_note("No overlapping datasets had RMSE values for pairwise method-vs-method ensemble comparisons.")

        # Dataset-level winner summary including the best base model.
        best_base_series = base_best.set_index("dataset")["best_base_rmse"]
        winner_matrix = method_matrix.copy()
        winner_matrix[BASELINE_LABEL] = best_base_series
        winner_matrix = winner_matrix.dropna(how="all")

        if winner_matrix.empty:
            display_note("Could not compute cross-dataset winner summary (no overlap between ensemble methods and base model).")
        else:
            winner_weights = {name: 0.0 for name in [BASELINE_LABEL] + EXPECTED_ENSEMBLE_METHODS}
            considered_datasets = 0
            for _, row in winner_matrix.iterrows():
                row_valid = row.dropna()
                if row_valid.empty:
                    continue
                considered_datasets += 1
                best_val = float(row_valid.min())
                winners = [name for name, value in row_valid.items() if abs(float(value) - best_val) <= ENSEMBLE_EPS]
                if not winners:
                    continue
                share = 1.0 / len(winners)
                for winner in winners:
                    winner_weights[winner] = winner_weights.get(winner, 0.0) + share

            winner_summary = pd.DataFrame(
                [
                    {
                        "method_or_baseline": name,
                        "dataset_best_weighted_wins": float(winner_weights.get(name, 0.0)),
                        "dataset_best_weighted_win_fraction": (
                            float(winner_weights.get(name, 0.0)) / considered_datasets if considered_datasets else np.nan
                        ),
                        "datasets_considered": considered_datasets,
                    }
                    for name in [BASELINE_LABEL] + EXPECTED_ENSEMBLE_METHODS
                ]
            ).sort_values(["dataset_best_weighted_win_fraction", "dataset_best_weighted_wins"], ascending=[False, False]).reset_index(drop=True)

            display_note("### Cross-Dataset Winner Summary (best RMSE among base + ensemble methods)")
            display(winner_summary)


### Ensemble Method Coverage

,ensemble_method,rows,datasets
0,CFA (Combinatorial Fusion),22,22
1,OOF stacking (RidgeCV),22,22
2,Model average (inverse train RMSE),9,9


No `ensemble_weights.csv` files were found for this run; skipping weight diagnostics.

### Ensemble Method Delta vs Best Base Model (across datasets)

,ensemble_method,datasets_compared,ensemble_beats_base_count,ensemble_ties_base_count,ensemble_loses_to_base_count,ensemble_beats_base_fraction,mean_delta_rmse,median_delta_rmse,std_delta_rmse,min_delta_rmse,max_delta_rmse,mean_relative_delta,median_relative_delta
0,Model average (inverse train RMSE),9,3,0,6,0.333333,0.081111,0.000514,0.177705,-0.014651,0.520467,0.055602,0.014325
1,CFA (Combinatorial Fusion),22,4,0,18,0.181818,0.873428,0.032066,1.739276,-0.084269,5.429306,0.153507,0.081835
2,OOF stacking (RidgeCV),22,2,0,20,0.090909,0.939874,0.029225,2.278927,-0.076333,8.350251,0.061666,0.042270


...

### Ensemble Method Pairwise Superiority (head-to-head)

,method_a,method_b,datasets_compared,method_a_wins,method_b_wins,ties,method_a_win_fraction,mean_delta_rmse_a_minus_b,median_delta_rmse_a_minus_b
0,CFA (Combinatorial Fusion),OOF stacking (RidgeCV),22,9,13,0,0.409091,-0.066446,0.003451
1,OOF stacking (RidgeCV),Model average (inverse train RMSE),9,4,5,0,0.444444,0.624955,0.001369
2,CFA (Combinatorial Fusion),Model average (inverse train RMSE),9,3,6,0,0.333333,0.345498,0.016114


,ensemble_method,pairwise_wins,pairwise_losses,pairwise_ties,pairwise_compared,pairwise_win_fraction
0,Model average (inverse train RMSE),11,7,0,18,0.611111
1,OOF stacking (RidgeCV),17,14,0,31,0.548387
2,CFA (Combinatorial Fusion),12,19,0,31,0.387097


### Cross-Dataset Winner Summary (best RMSE among base + ensemble methods)

,method_or_baseline,dataset_best_weighted_wins,dataset_best_weighted_win_fraction,datasets_considered
0,Best base model,17.0,0.772727,22
1,CFA (Combinatorial Fusion),4.0,0.181818,22
2,Model average (inverse train RMSE),1.0,0.045455,22
3,OOF stacking (RidgeCV),0.0,0.000000,22


In [66]:
# NO_RERUN_PUBLICATION_HELPERS
# Shared helpers for no-rerun publication analyses.

import hashlib
import subprocess

NO_RERUN_EPS = 1e-12


def successful_primary_metric_rows(frame=None):
    work = (SUMMARY_ANALYSIS_METRICS if frame is None else frame).copy()
    if work is None or work.empty:
        return pd.DataFrame()
    error_col = pick_column(work.columns, ["error", "Error"])
    if error_col is not None:
        work = work.loc[work[error_col].isna() | (work[error_col].astype(str).str.strip() == "")].copy()
    if "analysis_metric_value" not in work.columns:
        work = add_analysis_metric_columns(work, rmse_col=RMSE_COL, classification_col=CLASSIFICATION_FALLBACK_COL)
    work["analysis_metric_value"] = pd.to_numeric(work["analysis_metric_value"], errors="coerce")
    work = work.loc[work["analysis_metric_value"].notna()].copy()
    if work.empty:
        return work
    work["analysis_metric_direction"] = work["analysis_metric_direction"].fillna("lower").astype(str)
    work["comparison_score"] = np.where(
        work["analysis_metric_direction"].str.lower().eq("higher"),
        -work["analysis_metric_value"],
        work["analysis_metric_value"],
    )
    work["rank_within_dataset"] = work.groupby("dataset")["comparison_score"].rank(method="min", ascending=True)
    if "architecture_family" not in work.columns:
        if "classify_architecture_family" in globals():
            work["architecture_family"] = work.apply(
                lambda row: classify_architecture_family(row.get("workflow", ""), row.get("model", "")),
                axis=1,
            )
        else:
            work["architecture_family"] = work.get("workflow", pd.Series("", index=work.index)).astype(str)
    return work


def classify_publication_ensemble_method(model_name, workflow_name=""):
    model_text = str(model_name or "").strip().lower()
    workflow_text = str(workflow_name or "").strip().lower()
    if model_text.startswith("cfa (") or "cfa" in workflow_text:
        return "CFA fusion"
    if model_text.startswith("ensemble (") or "ensemble" in workflow_text:
        if "oof stacking" in model_text:
            return "OOF stacking"
        if "weighted average" in model_text and "inverse" in model_text and "rmse" in model_text:
            return "Inverse-RMSE weighted average"
        return "Other ensemble"
    return None


def metric_relative_improvement(base_value, challenger_value, direction):
    if pd.isna(base_value) or pd.isna(challenger_value):
        return np.nan
    denom = max(abs(float(base_value)), NO_RERUN_EPS)
    if str(direction).lower() == "higher":
        return (float(challenger_value) - float(base_value)) / denom
    return (float(base_value) - float(challenger_value)) / denom


def write_publication_csv(frame, filename):
    path = RUN_DIR / filename
    if frame is None:
        frame = pd.DataFrame()
    frame.to_csv(path, index=False)
    return path


def sha256_file(path, block_size=1024 * 1024):
    path = Path(path)
    if not path.exists() or not path.is_file():
        return ""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(block_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def git_value(args):
    try:
        return subprocess.check_output(["git", *args], cwd=Path(".").resolve(), text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return ""


def load_publication_predictions():
    if PREDICTIONS is not None and not PREDICTIONS.empty:
        return PREDICTIONS.copy()
    prediction_tables = []
    for predictions_path in sorted(RUN_DIR.glob("*/predictions.csv")):
        try:
            frame = pd.read_csv(predictions_path)
        except Exception:
            continue
        if frame.empty:
            continue
        if "dataset" not in frame.columns:
            frame["dataset"] = predictions_path.parent.name
        prediction_tables.append(frame)
    if not prediction_tables:
        return pd.DataFrame()
    return pd.concat(prediction_tables, ignore_index=True, sort=False)


In [67]:
# NO_RERUN_PUBLICATION_ENSEMBLE_VALUE
# Publication analysis 1: primary-metric-aware ensemble/CFA value-add.

ensemble_source = successful_primary_metric_rows()

if ensemble_source.empty:
    display_note("No successful metric rows were available for ensemble value-add analysis.")
else:
    ensemble_source["publication_ensemble_method"] = [
        classify_publication_ensemble_method(model_name, workflow_name)
        for model_name, workflow_name in zip(ensemble_source["model"], ensemble_source.get("workflow", pd.Series("", index=ensemble_source.index)))
    ]
    ensemble_rows = ensemble_source.loc[ensemble_source["publication_ensemble_method"].notna()].copy()
    base_rows = ensemble_source.loc[ensemble_source["publication_ensemble_method"].isna()].copy()

    if ensemble_rows.empty or base_rows.empty:
        display_note("Could not compute ensemble value-add because either ensemble/CFA rows or base-model rows are missing.")
    else:
        base_best = (
            base_rows.sort_values(["dataset", "comparison_score", "model"], ascending=[True, True, True])
            .groupby("dataset", as_index=False, dropna=False)
            .first()[[
                "dataset", "model", "workflow", "architecture_family", "analysis_metric", "analysis_metric_kind",
                "analysis_metric_direction", "analysis_metric_value", "comparison_score"
            ]]
            .rename(columns={
                "model": "best_base_model",
                "workflow": "best_base_workflow",
                "architecture_family": "best_base_family",
                "analysis_metric": "base_metric_column",
                "analysis_metric_kind": "base_metric_kind",
                "analysis_metric_direction": "metric_direction",
                "analysis_metric_value": "best_base_metric_value",
                "comparison_score": "best_base_comparison_score",
            })
        )

        competitive_rows = base_rows.merge(
            base_best[["dataset", "best_base_comparison_score"]],
            on="dataset",
            how="left",
        )
        competitive_rows["base_delta_from_best_base"] = competitive_rows["comparison_score"] - competitive_rows["best_base_comparison_score"]
        competitive_rows["competitive_threshold"] = np.maximum(0.05 * competitive_rows["best_base_comparison_score"].abs(), NO_RERUN_EPS)
        competitive_base_counts = (
            competitive_rows.loc[competitive_rows["base_delta_from_best_base"] <= competitive_rows["competitive_threshold"]]
            .groupby("dataset", as_index=False)["model"]
            .nunique()
            .rename(columns={"model": "competitive_base_model_count_5pct"})
        )

        ensemble_best = (
            ensemble_rows.sort_values(["dataset", "publication_ensemble_method", "comparison_score", "model"], ascending=[True, True, True, True])
            .groupby(["dataset", "publication_ensemble_method"], as_index=False, dropna=False)
            .first()[[
                "dataset", "publication_ensemble_method", "model", "workflow", "architecture_family", "task_kind",
                "analysis_metric", "analysis_metric_kind", "analysis_metric_direction", "analysis_metric_value",
                "comparison_score", "rank_within_dataset"
            ]]
            .rename(columns={
                "model": "ensemble_model",
                "workflow": "ensemble_workflow",
                "architecture_family": "ensemble_family",
                "analysis_metric": "ensemble_metric_column",
                "analysis_metric_kind": "ensemble_metric_kind",
                "analysis_metric_direction": "ensemble_metric_direction",
                "analysis_metric_value": "ensemble_metric_value",
                "comparison_score": "ensemble_comparison_score",
                "rank_within_dataset": "ensemble_rank_within_dataset",
            })
        )

        ensemble_value_add = ensemble_best.merge(base_best, on="dataset", how="inner")
        ensemble_value_add = ensemble_value_add.merge(competitive_base_counts, on="dataset", how="left")
        ensemble_value_add["competitive_base_model_count_5pct"] = ensemble_value_add["competitive_base_model_count_5pct"].fillna(1).astype(int)
        ensemble_value_add["score_improvement_vs_best_base"] = ensemble_value_add["best_base_comparison_score"] - ensemble_value_add["ensemble_comparison_score"]
        ensemble_value_add["relative_metric_improvement_vs_best_base"] = [
            metric_relative_improvement(base_value, ensemble_value, direction)
            for base_value, ensemble_value, direction in zip(
                ensemble_value_add["best_base_metric_value"],
                ensemble_value_add["ensemble_metric_value"],
                ensemble_value_add["metric_direction"],
            )
        ]
        ensemble_value_add["ensemble_beats_best_base"] = ensemble_value_add["score_improvement_vs_best_base"] > NO_RERUN_EPS
        ensemble_value_add["ensemble_ties_best_base"] = ensemble_value_add["score_improvement_vs_best_base"].abs() <= NO_RERUN_EPS
        ensemble_value_add["ensemble_loses_to_best_base"] = ensemble_value_add["score_improvement_vs_best_base"] < -NO_RERUN_EPS
        ensemble_value_add["ensemble_wins_dataset"] = pd.to_numeric(ensemble_value_add["ensemble_rank_within_dataset"], errors="coerce") <= 1.0
        ensemble_value_add["ensemble_top3_dataset"] = pd.to_numeric(ensemble_value_add["ensemble_rank_within_dataset"], errors="coerce") <= 3.0

        ensemble_value_summary = (
            ensemble_value_add.groupby(["publication_ensemble_method", "task_kind"], dropna=False)
            .agg(
                datasets=("dataset", "nunique"),
                wins_dataset=("ensemble_wins_dataset", "sum"),
                top3_datasets=("ensemble_top3_dataset", "sum"),
                beats_best_base=("ensemble_beats_best_base", "sum"),
                ties_best_base=("ensemble_ties_best_base", "sum"),
                loses_to_best_base=("ensemble_loses_to_best_base", "sum"),
                median_rank=("ensemble_rank_within_dataset", "median"),
                mean_rank=("ensemble_rank_within_dataset", "mean"),
                median_score_improvement=("score_improvement_vs_best_base", "median"),
                mean_score_improvement=("score_improvement_vs_best_base", "mean"),
                median_relative_metric_improvement=("relative_metric_improvement_vs_best_base", "median"),
                median_competitive_base_models=("competitive_base_model_count_5pct", "median"),
            )
            .reset_index()
            .sort_values(["task_kind", "top3_datasets", "beats_best_base", "median_score_improvement"], ascending=[True, False, False, False])
        )
        ensemble_value_summary["top3_fraction"] = ensemble_value_summary["top3_datasets"] / ensemble_value_summary["datasets"].replace(0, np.nan)
        ensemble_value_summary["beats_best_base_fraction"] = ensemble_value_summary["beats_best_base"] / ensemble_value_summary["datasets"].replace(0, np.nan)

        display_note("### Publication Analysis: Ensemble/CFA Value-Add")
        display(ensemble_value_summary.reset_index(drop=True))
        display(
            ensemble_value_add.sort_values(["publication_ensemble_method", "task_kind", "score_improvement_vs_best_base"], ascending=[True, True, False])
            .reset_index(drop=True)
        )

        write_publication_csv(ensemble_value_summary, "publication_ensemble_value_add_summary.csv")
        write_publication_csv(ensemble_value_add, "publication_ensemble_value_add_by_dataset.csv")

        fig = px.box(
            ensemble_value_add,
            x="publication_ensemble_method",
            y="score_improvement_vs_best_base",
            color="task_kind",
            points="all",
            hover_data=["dataset", "ensemble_model", "best_base_model", "ensemble_rank_within_dataset", "relative_metric_improvement_vs_best_base"],
            title="Ensemble/CFA value-add vs best base model (primary metric; positive is better)",
        )
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(xaxis_title="Fusion method", yaxis_title="Primary-metric improvement vs best base")
        fig.show()


### Publication Analysis: Ensemble/CFA Value-Add

,publication_ensemble_method,task_kind,datasets,wins_dataset,top3_datasets,beats_best_base,ties_best_base,loses_to_best_base,median_rank,mean_rank,median_score_improvement,mean_score_improvement,median_relative_metric_improvement,median_competitive_base_models,top3_fraction,beats_best_base_fraction
0,OOF stacking,classification,22,7,12,9,0,13,3.0,4.772727,-0.003356,-0.021946,-0.003474,8.5,0.545455,0.409091
1,Inverse-RMSE weighted average,classification,13,4,7,6,0,7,3.0,4.000000,-0.002174,-0.006150,-0.002397,10.0,0.538462,0.461538
2,CFA fusion,classification,22,1,6,5,0,17,6.0,6.545455,-0.015364,-0.023982,-0.019396,8.5,0.272727,0.227273
3,OOF stacking,regression,22,0,8,2,0,20,5.5,6.000000,-0.029225,-0.939874,-0.042270,3.0,0.363636,0.090909
4,Inverse-RMSE weighted average,regression,9,1,7,3,0,6,3.0,2.666667,-0.000514,-0.081111,-0.014325,3.0,0.777778,0.333333
5,CFA fusion,regression,22,4,6,4,0,18,7.0,6.590909,-0.032066,-0.873428,-0.081835,3.0,0.272727,0.181818


,dataset,publication_ensemble_method,ensemble_model,ensemble_workflow,ensemble_family,task_kind,ensemble_metric_column,ensemble_metric_kind,ensemble_metric_direction,ensemble_metric_value,ensemble_comparison_score,ensemble_rank_within_dataset,best_base_model,best_base_workflow,best_base_family,base_metric_column,base_metric_kind,metric_direction,best_base_metric_value,best_base_comparison_score,competitive_base_model_count_5pct,score_improvement_vs_best_base,relative_metric_improvement_vs_best_base,ensemble_beats_best_base,ensemble_ties_best_base,ensemble_loses_to_best_base,ensemble_wins_dataset,ensemble_top3_dataset
0,tdc_skin_reaction,CFA fusion,CFA (Combinatorial Fusion),CFA fusion,CFA combinatorial fusion,classification,test_roc_auc,roc_auc,higher,0.761935,-0.761935,2.0,TabPFNClassifier,TabPFN deep learning,TabPFN foundation model,test_roc_auc,roc_auc,higher,0.746774,-0.746774,9,0.015161,0.020302,True,False,False,False,True
1,tdc_cyp2c9_veith,CFA fusion,CFA (Combinatorial Fusion),CFA fusion,CFA combinatorial fusion,classification,test_auprc,auprc,higher,0.796950,-0.796950,3.0,HistGradientBoosting,conventional,Conventional ML,test_auprc,auprc,higher,0.794078,-0.794078,9,0.002872,0.003616,True,False,False,False,True
2,tdc_dili,CFA fusion,CFA (Combinatorial Fusion),CFA fusion,CFA combinatorial fusion,classification,test_roc_auc,roc_auc,higher,0.908913,-0.908913,2.0,Random forest,conventional,Conventional ML,test_roc_auc,roc_auc,higher,0.906957,-0.906957,12,0.001957,0.002157,True,False,False,False,True
3,tdc_herg_karim,CFA fusion,CFA (Combinatorial Fusion),CFA fusion,CFA combinatorial fusion,classification,test_roc_auc,roc_auc,higher,0.943185,-0.943185,2.0,Random forest,conventional,Conventional ML,test_roc_auc,roc_auc,higher,0.942149,-0.942149,10,0.001036,0.001099,True,False,False,False,True
4,tdc_hia_hou,CFA fusion,CFA (Combinatorial Fusion),CFA fusion,CFA combinatorial fusion,classification,test_roc_auc,roc_auc,higher,0.989918,-0.989918,1.0,CatBoost,conventional,Conventional ML,test_roc_auc,roc_auc,higher,0.988889,-0.988889,14,0.001029,0.001040,True,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,tdc_ppbr_az,OOF stacking,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,Ensemble meta-model,regression,test_rmse,rmse,lower,11.986943,11.986943,5.0,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,Graph transfer + tabular head,test_rmse,rmse,lower,11.378376,11.378376,4,-0.608567,-0.053485,False,False,True,False,False
106,tdc_vdss_lombardo,OOF stacking,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,Ensemble meta-model,regression,test_rmse,rmse,lower,5.541208,5.541208,7.0,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,Graph transfer + tabular head,test_rmse,rmse,lower,4.684131,4.684131,3,-0.857077,-0.182975,False,False,True,False,False
107,tdc_half_life_obach,OOF stacking,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,Ensemble meta-model,regression,test_rmse,rmse,lower,21.502196,21.502196,10.0,TabPFNRegressor,TabPFN deep learning,TabPFN foundation model,test_rmse,rmse,lower,17.210966,17.210966,1,-4.291229,-0.249331,False,False,True,False,False
108,tdc_clearance_microsome_az,OOF stacking,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,Ensemble meta-model,regression,test_rmse,rmse,lower,42.594895,42.594895,18.0,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,Graph transfer + tabular head,test_rmse,rmse,lower,36.331993,36.331993,5,-6.262902,-0.172380,False,False,True,False,False


In [68]:
# NO_RERUN_PUBLICATION_COMPONENT_ABLATION
# Publication analysis 2: staged component ablation derived from existing metrics.

ablation_source = successful_primary_metric_rows()


def atomic_component_family(row):
    model_text = str(row.get("model", "")).strip().lower()
    workflow_text = str(row.get("workflow", "")).strip().lower()
    ensemble_method = classify_publication_ensemble_method(row.get("model", ""), row.get("workflow", ""))
    if ensemble_method == "CFA fusion":
        return "cfa"
    if ensemble_method is not None:
        return "ensemble"
    if "maplight + gnn" in workflow_text or "maplight + gnn" in model_text:
        return "deep_backend"
    if any(token in workflow_text or token in model_text for token in ["chemprop", "tabpfn", "chemml", "uni-mol", "unimol"]):
        return "deep_backend"
    if "maplight" in model_text or "maplight" in workflow_text:
        return "maplight_classic"
    return "conventional_ml"


if ablation_source.empty:
    display_note("No successful metric rows were available for component-ablation analysis.")
else:
    ablation_source["atomic_component_family"] = ablation_source.apply(atomic_component_family, axis=1)
    stage_definitions = [
        (1, "Conventional ML only", {"conventional_ml"}),
        (2, "+ MapLight classic features", {"conventional_ml", "maplight_classic"}),
        (3, "+ neural/deep backends", {"conventional_ml", "maplight_classic", "deep_backend"}),
        (4, "+ CFA fusion", {"conventional_ml", "maplight_classic", "deep_backend", "cfa"}),
        (5, "Full pipeline incl. ensembles", {"conventional_ml", "maplight_classic", "deep_backend", "cfa", "ensemble"}),
    ]

    ablation_rows = []
    for stage_order, stage_name, allowed_components in stage_definitions:
        local = ablation_source.loc[ablation_source["atomic_component_family"].isin(allowed_components)].copy()
        if local.empty:
            continue
        best_local = (
            local.sort_values(["dataset", "comparison_score", "model"], ascending=[True, True, True])
            .groupby("dataset", as_index=False, dropna=False)
            .first()
        )
        for _, row in best_local.iterrows():
            ablation_rows.append(
                {
                    "dataset": row.get("dataset", ""),
                    "task_kind": row.get("task_kind", ""),
                    "stage_order": stage_order,
                    "stage": stage_name,
                    "best_model_at_stage": row.get("model", ""),
                    "best_workflow_at_stage": row.get("workflow", ""),
                    "best_component_family_at_stage": row.get("atomic_component_family", ""),
                    "analysis_metric": row.get("analysis_metric", ""),
                    "analysis_metric_kind": row.get("analysis_metric_kind", ""),
                    "analysis_metric_direction": row.get("analysis_metric_direction", ""),
                    "analysis_metric_value": row.get("analysis_metric_value", np.nan),
                    "comparison_score": row.get("comparison_score", np.nan),
                }
            )

    component_ablation_by_dataset = pd.DataFrame(ablation_rows)
    if component_ablation_by_dataset.empty:
        display_note("No staged ablation rows could be computed from the existing metrics.")
    else:
        component_ablation_by_dataset = component_ablation_by_dataset.sort_values(["dataset", "stage_order"]).reset_index(drop=True)
        component_ablation_by_dataset["previous_stage_score"] = component_ablation_by_dataset.groupby("dataset")["comparison_score"].shift(1)
        component_ablation_by_dataset["previous_stage_metric_value"] = component_ablation_by_dataset.groupby("dataset")["analysis_metric_value"].shift(1)
        component_ablation_by_dataset["marginal_score_improvement"] = component_ablation_by_dataset["previous_stage_score"] - component_ablation_by_dataset["comparison_score"]
        component_ablation_by_dataset.loc[component_ablation_by_dataset["previous_stage_score"].isna(), "marginal_score_improvement"] = np.nan
        component_ablation_by_dataset["marginal_relative_metric_improvement"] = [
            metric_relative_improvement(prev_value, current_value, direction)
            for prev_value, current_value, direction in zip(
                component_ablation_by_dataset["previous_stage_metric_value"],
                component_ablation_by_dataset["analysis_metric_value"],
                component_ablation_by_dataset["analysis_metric_direction"],
            )
        ]
        component_ablation_by_dataset["improved_vs_previous_stage"] = component_ablation_by_dataset["marginal_score_improvement"] > NO_RERUN_EPS
        component_ablation_by_dataset["stage_changed_winning_model"] = (
            component_ablation_by_dataset["best_model_at_stage"]
            != component_ablation_by_dataset.groupby("dataset")["best_model_at_stage"].shift(1)
        )
        component_ablation_by_dataset.loc[component_ablation_by_dataset["previous_stage_score"].isna(), "stage_changed_winning_model"] = False

        component_ablation_summary = (
            component_ablation_by_dataset.groupby(["stage_order", "stage"], dropna=False)
            .agg(
                datasets_evaluated=("dataset", "nunique"),
                datasets_improved_vs_previous=("improved_vs_previous_stage", "sum"),
                datasets_changed_winner=("stage_changed_winning_model", "sum"),
                median_marginal_score_improvement=("marginal_score_improvement", "median"),
                mean_marginal_score_improvement=("marginal_score_improvement", "mean"),
                median_relative_metric_improvement=("marginal_relative_metric_improvement", "median"),
            )
            .reset_index()
            .sort_values("stage_order")
        )
        component_ablation_summary["improvement_fraction"] = component_ablation_summary["datasets_improved_vs_previous"] / component_ablation_summary["datasets_evaluated"].replace(0, np.nan)

        display_note("### Publication Analysis: Component Ablation from Existing Metrics")
        display(component_ablation_summary.reset_index(drop=True))
        display(component_ablation_by_dataset.reset_index(drop=True))

        write_publication_csv(component_ablation_summary, "publication_component_ablation_summary.csv")
        write_publication_csv(component_ablation_by_dataset, "publication_component_ablation_by_dataset.csv")

        fig = px.bar(
            component_ablation_summary,
            x="stage",
            y="datasets_improved_vs_previous",
            hover_data=["datasets_evaluated", "improvement_fraction", "median_marginal_score_improvement"],
            title="Datasets improved by each added pipeline stage",
        )
        fig.update_layout(xaxis_title="Cumulative stage", yaxis_title="Datasets improved vs previous stage")
        fig.show()


### Publication Analysis: Component Ablation from Existing Metrics

,stage_order,stage,datasets_evaluated,datasets_improved_vs_previous,datasets_changed_winner,median_marginal_score_improvement,mean_marginal_score_improvement,median_relative_metric_improvement,improvement_fraction
0,1,Conventional ML only,44,0,0,NaN,NaN,NaN,0.000000
1,2,+ MapLight classic features,44,7,7,0.0,0.071573,0.0,0.159091
2,3,+ neural/deep backends,44,14,14,0.0,0.079094,0.0,0.318182
3,4,+ CFA fusion,44,9,9,0.0,0.002889,0.0,0.204545
4,5,Full pipeline incl. ensembles,44,12,12,0.0,0.001767,0.0,0.272727


,dataset,task_kind,stage_order,stage,best_model_at_stage,best_workflow_at_stage,best_component_family_at_stage,analysis_metric,analysis_metric_kind,analysis_metric_direction,analysis_metric_value,comparison_score,previous_stage_score,previous_stage_metric_value,marginal_score_improvement,marginal_relative_metric_improvement,improved_vs_previous_stage,stage_changed_winning_model
0,chemml_cep_homo,regression,1,Conventional ML only,HistGradientBoosting,conventional,conventional_ml,test_rmse,rmse,lower,0.103895,0.103895,NaN,NaN,NaN,NaN,False,False
1,chemml_cep_homo,regression,2,+ MapLight classic features,HistGradientBoosting,conventional,conventional_ml,test_rmse,rmse,lower,0.103895,0.103895,0.103895,0.103895,0.000000,0.000000,False,False
2,chemml_cep_homo,regression,3,+ neural/deep backends,TabPFNRegressor,TabPFN deep learning,deep_backend,test_rmse,rmse,lower,0.079691,0.079691,0.103895,0.103895,0.024204,0.232963,True,True
3,chemml_cep_homo,regression,4,+ CFA fusion,TabPFNRegressor,TabPFN deep learning,deep_backend,test_rmse,rmse,lower,0.079691,0.079691,0.079691,0.079691,0.000000,0.000000,False,False
4,chemml_cep_homo,regression,5,Full pipeline incl. ensembles,TabPFNRegressor,TabPFN deep learning,deep_backend,test_rmse,rmse,lower,0.079691,0.079691,0.079691,0.079691,0.000000,0.000000,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,tdc_vdss_lombardo,regression,1,Conventional ML only,SVR,conventional,conventional_ml,test_rmse,rmse,lower,4.897795,4.897795,NaN,NaN,NaN,NaN,False,False
216,tdc_vdss_lombardo,regression,2,+ MapLight classic features,MapLight CatBoost (Strict Parity),conventional,maplight_classic,test_rmse,rmse,lower,4.760954,4.760954,4.897795,4.897795,0.136841,0.027939,True,True
217,tdc_vdss_lombardo,regression,3,+ neural/deep backends,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,deep_backend,test_rmse,rmse,lower,4.684131,4.684131,4.760954,4.760954,0.076822,0.016136,True,True
218,tdc_vdss_lombardo,regression,4,+ CFA fusion,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,deep_backend,test_rmse,rmse,lower,4.684131,4.684131,4.684131,4.684131,0.000000,0.000000,False,False


In [69]:
# NO_RERUN_PUBLICATION_DATASET_DIFFICULTY
# Publication analysis 3: dataset difficulty stratification from existing artifacts.

difficulty_source = successful_primary_metric_rows()

if difficulty_source.empty:
    display_note("No successful metric rows were available for dataset difficulty stratification.")
else:
    best_difficulty_rows = (
        difficulty_source.sort_values(["dataset", "comparison_score", "model"], ascending=[True, True, True])
        .groupby("dataset", as_index=False, dropna=False)
        .first()
    )

    descriptor_cols = [
        col for col in [
            "dataset", "benchmark_suite", "benchmark_id", "n_molecules", "n_train", "n_test", "split_strategy",
            "target_transform", "selected_feature_count", "original_feature_count", "post_dedup_feature_count",
            "leaderboard_estimated_rank_vs_top10", "leaderboard_gap_to_top10_cutoff", "leaderboard_delta_top10_best",
            "leaderboard_delta_primary", "leaderboard_metric_normalized",
        ]
        if col in difficulty_source.columns
    ]
    dataset_descriptors = (
        difficulty_source.sort_values(["dataset", "comparison_score"])
        .groupby("dataset", as_index=False, dropna=False)
        .first()[descriptor_cols]
    )

    winner_cols = [
        "dataset", "task_kind", "model", "workflow", "architecture_family", "analysis_metric", "analysis_metric_kind",
        "analysis_metric_direction", "analysis_metric_value", "comparison_score"
    ]
    winner_cols = [col for col in winner_cols if col in best_difficulty_rows.columns]
    dataset_difficulty = dataset_descriptors.merge(
        best_difficulty_rows[winner_cols].rename(columns={
            "model": "winning_model",
            "workflow": "winning_workflow",
            "architecture_family": "winning_family",
            "analysis_metric_value": "best_primary_metric_value",
            "comparison_score": "best_comparison_score",
        }),
        on="dataset",
        how="left",
    )

    if "leaderboard_eval" in globals() and leaderboard_eval is not None and not leaderboard_eval.empty:
        rank_cols = [col for col in ["dataset", "estimated_rank_vs_top10", "in_top10_estimate", "gap_to_top1", "gap_to_top10_cutoff"] if col in leaderboard_eval.columns]
        if rank_cols:
            rank_best = leaderboard_eval[rank_cols].copy()
            for col in [c for c in rank_cols if c not in {"dataset", "in_top10_estimate"}]:
                rank_best[col] = pd.to_numeric(rank_best[col], errors="coerce")
            sort_col = "estimated_rank_vs_top10" if "estimated_rank_vs_top10" in rank_best.columns else rank_cols[-1]
            rank_best = rank_best.sort_values(["dataset", sort_col]).groupby("dataset", as_index=False).first()
            dataset_difficulty = dataset_difficulty.merge(rank_best, on="dataset", how="left", suffixes=("", "_reference"))

    prediction_descriptor_rows = []
    scaffold_note = "RDKit unavailable; scaffold diversity skipped."
    try:
        from rdkit import Chem
        from rdkit import RDLogger
        from rdkit.Chem.Scaffolds import MurckoScaffold
        RDLogger.DisableLog("rdApp.warning")
        rdkit_available = True
        scaffold_note = "Bemis-Murcko scaffold diversity computed from de-duplicated prediction SMILES."
    except Exception as exc:
        Chem = None
        MurckoScaffold = None
        rdkit_available = False
        scaffold_note = f"RDKit unavailable; scaffold diversity skipped ({exc})."

    publication_predictions = load_publication_predictions()
    if publication_predictions is not None and not publication_predictions.empty:
        pred = publication_predictions.copy()
        dataset_col = pick_column(pred.columns, ["dataset", "dataset_id", "dataset_name"])
        observed_col = pick_column(pred.columns, ["observed", "y_true", "actual"])
        split_col = pick_column(pred.columns, ["split", "Split"])
        smiles_col = pick_column(pred.columns, ["smiles", "SMILES", "canonical_smiles"])
        model_col = pick_column(pred.columns, ["model", "Model"])
        if dataset_col is not None and observed_col is not None:
            pred = pred.rename(columns={dataset_col: "dataset", observed_col: "observed"})
            pred["split"] = pred[split_col] if split_col is not None else "all"
            pred["smiles"] = pred[smiles_col] if smiles_col is not None else ""
            pred["model"] = pred[model_col] if model_col is not None else ""
            pred["observed"] = pd.to_numeric(pred["observed"], errors="coerce")
            dedupe_cols = ["dataset", "split", "smiles", "observed"] if smiles_col is not None else ["dataset", "split", "observed"]
            pred_unique = pred.loc[pred["observed"].notna()].drop_duplicates(dedupe_cols).copy()
            for dataset_name, group in pred_unique.groupby("dataset", sort=True):
                values = group["observed"].dropna()
                unique_values = sorted(values.unique().tolist())
                is_binary = len(unique_values) <= 2 and set(unique_values).issubset({0, 1, 0.0, 1.0})
                payload = {
                    "dataset": dataset_name,
                    "prediction_unique_rows": int(len(group)),
                    "observed_unique_values": int(len(unique_values)),
                    "class_positive_fraction": float(values.mean()) if is_binary and len(values) else np.nan,
                    "class_minority_fraction": float(min(values.mean(), 1.0 - values.mean())) if is_binary and len(values) else np.nan,
                }
                if rdkit_available and smiles_col is not None:
                    smiles_values = group["smiles"].dropna().astype(str).str.strip()
                    smiles_values = smiles_values[smiles_values.ne("")].drop_duplicates()
                    scaffolds = []
                    for smiles in smiles_values:
                        mol = Chem.MolFromSmiles(smiles)
                        if mol is None:
                            continue
                        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
                        scaffolds.append(scaffold or smiles)
                    payload["scaffold_count"] = int(len(set(scaffolds))) if scaffolds else np.nan
                    payload["scaffold_diversity"] = float(len(set(scaffolds)) / len(smiles_values)) if len(smiles_values) else np.nan
                prediction_descriptor_rows.append(payload)

    if prediction_descriptor_rows:
        dataset_difficulty = dataset_difficulty.merge(pd.DataFrame(prediction_descriptor_rows), on="dataset", how="left")

    for col in ["n_molecules", "n_train", "n_test", "selected_feature_count", "original_feature_count", "post_dedup_feature_count"]:
        if col in dataset_difficulty.columns:
            dataset_difficulty[col] = pd.to_numeric(dataset_difficulty[col], errors="coerce")
    if "n_molecules" not in dataset_difficulty.columns or dataset_difficulty["n_molecules"].isna().all():
        if {"n_train", "n_test"}.issubset(dataset_difficulty.columns):
            dataset_difficulty["n_molecules"] = pd.to_numeric(dataset_difficulty["n_train"], errors="coerce") + pd.to_numeric(dataset_difficulty["n_test"], errors="coerce")
    if "n_molecules" in dataset_difficulty.columns:
        try:
            dataset_difficulty["dataset_size_quartile"] = pd.qcut(dataset_difficulty["n_molecules"], q=4, duplicates="drop")
        except Exception:
            dataset_difficulty["dataset_size_quartile"] = pd.NA

    numeric_candidates = [
        "n_molecules", "n_train", "n_test", "selected_feature_count", "original_feature_count", "post_dedup_feature_count",
        "class_positive_fraction", "class_minority_fraction", "scaffold_count", "scaffold_diversity",
    ]
    outcome_candidates = [
        "best_primary_metric_value", "estimated_rank_vs_top10", "gap_to_top10_cutoff", "leaderboard_gap_to_top10_cutoff",
        "leaderboard_delta_primary", "leaderboard_delta_top10_best",
    ]
    corr_rows = []
    for x_col in [col for col in numeric_candidates if col in dataset_difficulty.columns]:
        for y_col in [col for col in outcome_candidates if col in dataset_difficulty.columns]:
            local = dataset_difficulty[[x_col, y_col]].apply(pd.to_numeric, errors="coerce").dropna()
            if len(local) >= 4 and local[x_col].nunique() > 1 and local[y_col].nunique() > 1:
                corr_rows.append({
                    "descriptor": x_col,
                    "outcome": y_col,
                    "datasets": int(len(local)),
                    "spearman_correlation": float(local[x_col].corr(local[y_col], method="spearman")),
                    "pearson_correlation": float(local[x_col].corr(local[y_col], method="pearson")),
                })
    dataset_difficulty_correlations = pd.DataFrame(corr_rows).sort_values(["outcome", "spearman_correlation"], ascending=[True, False]) if corr_rows else pd.DataFrame()

    family_profile_cols = [col for col in ["winning_family", "task_kind", "dataset_size_quartile"] if col in dataset_difficulty.columns]
    if family_profile_cols:
        family_difficulty_profile = (
            dataset_difficulty.groupby(family_profile_cols, dropna=False, observed=False)
            .agg(
                datasets=("dataset", "nunique"),
                median_n_molecules=("n_molecules", "median") if "n_molecules" in dataset_difficulty.columns else ("dataset", "size"),
                median_best_metric=("best_primary_metric_value", "median"),
                median_estimated_rank=("estimated_rank_vs_top10", "median") if "estimated_rank_vs_top10" in dataset_difficulty.columns else ("dataset", "size"),
            )
            .reset_index()
            .sort_values(["task_kind", "dataset_size_quartile", "datasets"], ascending=[True, True, False])
        )
    else:
        family_difficulty_profile = pd.DataFrame()

    display_note("### Publication Analysis: Dataset Difficulty Stratification")
    display(dataset_difficulty.sort_values(["task_kind", "n_molecules", "dataset"], ascending=[True, True, True]).reset_index(drop=True))
    if not dataset_difficulty_correlations.empty:
        display_note("Descriptor/outcome correlations")
        display(dataset_difficulty_correlations.reset_index(drop=True))
    if not family_difficulty_profile.empty:
        display_note("Winning-family profile by dataset size and task type")
        display(family_difficulty_profile.reset_index(drop=True))
    display_note(scaffold_note)

    write_publication_csv(dataset_difficulty, "publication_dataset_difficulty_stratification.csv")
    write_publication_csv(dataset_difficulty_correlations, "publication_dataset_difficulty_correlations.csv")
    write_publication_csv(family_difficulty_profile, "publication_winning_family_difficulty_profile.csv")

    if "n_molecules" in dataset_difficulty.columns:
        y_col = "estimated_rank_vs_top10" if "estimated_rank_vs_top10" in dataset_difficulty.columns and dataset_difficulty["estimated_rank_vs_top10"].notna().any() else "best_primary_metric_value"
        fig = px.scatter(
            dataset_difficulty,
            x="n_molecules",
            y=y_col,
            color="winning_family" if "winning_family" in dataset_difficulty.columns else "task_kind",
            symbol="task_kind" if "task_kind" in dataset_difficulty.columns else None,
            hover_data=[col for col in ["dataset", "winning_model", "analysis_metric_kind", "class_minority_fraction", "scaffold_diversity"] if col in dataset_difficulty.columns],
            title=f"Dataset size vs {y_col}, colored by winning family",
        )
        fig.update_xaxes(type="log", title="Dataset size (n molecules, log scale)")
        fig.show()


### Publication Analysis: Dataset Difficulty Stratification

,dataset,benchmark_suite,benchmark_id,n_molecules,n_train,n_test,split_strategy,target_transform,selected_feature_count,original_feature_count,post_dedup_feature_count,leaderboard_estimated_rank_vs_top10,leaderboard_gap_to_top10_cutoff,leaderboard_delta_top10_best,leaderboard_delta_primary,leaderboard_metric_normalized,task_kind,winning_model,winning_workflow,winning_family,analysis_metric,analysis_metric_kind,analysis_metric_direction,best_primary_metric_value,best_comparison_score,estimated_rank_vs_top10,in_top10_estimate,gap_to_top1,gap_to_top10_cutoff,prediction_unique_rows,observed_unique_values,class_positive_fraction,class_minority_fraction,scaffold_count,scaffold_diversity,dataset_size_quartile
0,tdc_carcinogens_lagunin,tdc,carcinogens_lagunin,280,224,56,random,raw,83,10115,8428,NaN,NaN,NaN,NaN,None,classification,AdaBoost,conventional,Conventional ML,test_roc_auc,roc_auc,higher,0.862626,-0.862626,1.0,True,-0.014626,-0.132626,280,2,0.214286,0.214286,202,0.726619,"(279.999, 667.0]"
1,tdc_skin_reaction,tdc,skin_reaction,404,323,81,random,raw,22,10115,7657,NaN,NaN,NaN,NaN,None,classification,Ensemble (Weighted average (inverse train RMSE)),ensemble,Ensemble meta-model,test_roc_auc,roc_auc,higher,0.774839,-0.774839,1.0,True,-0.033839,-0.275839,404,2,0.678218,0.321782,248,0.613861,"(279.999, 667.0]"
2,tdc_dili,tdc,dili,475,379,96,predefined,raw,270,10115,8800,NaN,NaN,NaN,NaN,None,classification,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,Ensemble meta-model,test_roc_auc,roc_auc,higher,0.913043,-0.913043,5.0,True,0.031957,-0.061043,475,2,0.496842,0.496842,345,0.726316,"(279.999, 667.0]"
3,tdc_hia_hou,tdc,hia_hou,578,461,117,predefined,raw,86,10115,8681,NaN,NaN,NaN,NaN,None,classification,CFA (Combinatorial Fusion),CFA fusion,CFA combinatorial fusion,test_roc_auc,roc_auc,higher,0.989918,-0.989918,4.0,True,0.004082,-0.013918,578,2,0.865052,0.134948,403,0.697232,"(279.999, 667.0]"
4,tdc_bioavailability_ma,tdc,bioavailability_ma,640,512,128,predefined,raw,12,10115,8756,NaN,NaN,NaN,NaN,None,classification,CatBoost,conventional,Conventional ML,test_roc_auc,roc_auc,higher,0.776688,-0.776688,1.0,True,-0.028688,-0.136688,640,2,0.768750,0.231250,456,0.712500,"(279.999, 667.0]"
5,tdc_herg,tdc,herg,655,523,132,predefined,raw,131,10115,8665,NaN,NaN,NaN,NaN,None,classification,AdaBoost,conventional,Conventional ML,test_roc_auc,roc_auc,higher,0.860825,-0.860825,3.0,True,0.019175,-0.054825,651,2,0.686636,0.313364,410,0.632716,"(279.999, 667.0]"
6,tdc_cyp2d6_substrate_carbonmangels,tdc,cyp2d6_substrate_carbonmangels,667,532,135,predefined,raw,142,10115,8743,NaN,NaN,NaN,NaN,None,classification,"Chemprop v2 (AttentiveFP, ensemble=1)",Chemprop v2,Graph neural networks (Chemprop),test_auprc,auprc,higher,0.683938,-0.683938,5.0,True,0.082062,-0.113938,665,2,0.287218,0.287218,470,0.707831,"(279.999, 667.0]"
7,tdc_cyp2c9_substrate_carbonmangels,tdc,cyp2c9_substrate_carbonmangels,669,534,135,predefined,raw,8,10115,8739,NaN,NaN,NaN,NaN,None,classification,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,Ensemble meta-model,test_auprc,auprc,higher,0.427480,-0.427480,3.0,True,0.022520,-0.067480,666,2,0.211712,0.211712,471,0.707207,"(667.0, 1604.5]"
8,tdc_cyp3a4_substrate_carbonmangels,tdc,cyp3a4_substrate_carbonmangels,670,535,135,predefined,raw,176,10115,8755,NaN,NaN,NaN,NaN,None,classification,LogisticRegression,conventional,Conventional ML,test_auprc,auprc,higher,0.716619,-0.716619,7.0,True,0.033207,-0.007793,667,2,0.530735,0.469265,473,0.709145,"(667.0, 1604.5]"
9,tdc_pgp_broccatelli,tdc,pgp_broccatelli,1218,973,245,predefined,raw,274,10115,8703,NaN,NaN,NaN,NaN,None,classification,HistGradientBoosting,conventional,Conventional ML,test_roc_auc,roc_auc,higher,0.924020,-0.924020,5.0,True,0.069980,-0.013020,1212,2,0.533828,0.466172,700,0.577558,"(667.0, 1604.5]"


Descriptor/outcome correlations

,descriptor,outcome,datasets,spearman_correlation,pearson_correlation
0,post_dedup_feature_count,best_primary_metric_value,44,0.287104,0.040385
1,class_positive_fraction,best_primary_metric_value,22,0.262564,0.306791
2,selected_feature_count,best_primary_metric_value,44,0.194802,0.250419
3,class_minority_fraction,best_primary_metric_value,22,0.189159,0.214335
4,n_train,best_primary_metric_value,44,0.129753,-0.149356
5,n_molecules,best_primary_metric_value,44,0.125247,-0.149422
6,n_test,best_primary_metric_value,44,0.118020,-0.149683
7,scaffold_count,best_primary_metric_value,44,0.040806,-0.155472
8,scaffold_diversity,best_primary_metric_value,44,-0.233068,0.061572
9,selected_feature_count,estimated_rank_vs_top10,37,0.364556,0.300972


Winning-family profile by dataset size and task type

,winning_family,task_kind,dataset_size_quartile,datasets,median_n_molecules,median_best_metric,median_estimated_rank
0,Conventional ML,classification,"(279.999, 667.0]",3,640.0,0.860825,1.0
1,Ensemble meta-model,classification,"(279.999, 667.0]",2,439.5,0.843941,3.0
2,CFA combinatorial fusion,classification,"(279.999, 667.0]",1,578.0,0.989918,4.0
3,Graph neural networks (Chemprop),classification,"(279.999, 667.0]",1,667.0,0.683938,5.0
4,Graph transfer + tabular head,classification,"(279.999, 667.0]",0,NaN,NaN,NaN
5,TabPFN foundation model,classification,"(279.999, 667.0]",0,NaN,NaN,NaN
6,Conventional ML,classification,"(667.0, 1604.5]",3,1218.0,0.924020,5.0
7,Ensemble meta-model,classification,"(667.0, 1604.5]",1,669.0,0.427480,3.0
8,CFA combinatorial fusion,classification,"(667.0, 1604.5]",0,NaN,NaN,NaN
9,Graph neural networks (Chemprop),classification,"(667.0, 1604.5]",0,NaN,NaN,NaN


Bemis-Murcko scaffold diversity computed from de-duplicated prediction SMILES.

In [70]:
# NO_RERUN_PUBLICATION_REPRODUCIBILITY
# Publication analysis 4: reproducibility audit and archive manifest from existing artifacts.

required_dataset_files = [
    "metrics.csv",
    "predictions.csv",
    "selected_features.csv",
    "selector_coefficients.csv",
    "run_status.json",
    "step_runtime.csv",
]

manifest_rows = []
for dataset_dir in sorted([path for path in RUN_DIR.iterdir() if path.is_dir()]):
    metrics_path = dataset_dir / "metrics.csv"
    first_metric = {}
    if metrics_path.exists():
        try:
            metric_head = pd.read_csv(metrics_path, nrows=1)
            if not metric_head.empty:
                first_metric = metric_head.iloc[0].to_dict()
        except Exception:
            first_metric = {}
    row = {
        "dataset": dataset_dir.name,
        "dataset_dir": str(dataset_dir).replace("\\", "/"),
        "git_commit": git_value(["rev-parse", "HEAD"]),
        "git_is_dirty": bool(git_value(["status", "--porcelain"])),
        "configured_random_seed": (RUN_CONFIG or {}).get("random_seed", "") if isinstance(RUN_CONFIG, dict) else "",
        "configured_chemprop_seed": (RUN_CONFIG or {}).get("chemprop_random_seed", "") if isinstance(RUN_CONFIG, dict) else "",
        "configured_maplight_parity_seeds": (RUN_CONFIG or {}).get("maplight_parity_seeds", "") if isinstance(RUN_CONFIG, dict) else "",
        "split_strategy": first_metric.get("split_strategy", ""),
        "split_train_hash": first_metric.get("split_train_hash", ""),
        "split_test_hash": first_metric.get("split_test_hash", ""),
        "n_train": first_metric.get("n_train", np.nan),
        "n_test": first_metric.get("n_test", np.nan),
    }
    present_count = 0
    for filename in required_dataset_files:
        path = dataset_dir / filename
        present = path.exists()
        present_count += int(present)
        stem = filename.replace(".", "_")
        row[f"has_{stem}"] = bool(present)
        row[f"sha256_{stem}"] = sha256_file(path) if present else ""
        row[f"bytes_{stem}"] = int(path.stat().st_size) if present else 0
    row["required_dataset_artifacts_present"] = present_count
    row["required_dataset_artifacts_total"] = len(required_dataset_files)
    row["required_dataset_artifact_fraction"] = present_count / len(required_dataset_files)
    manifest_rows.append(row)

reproducibility_manifest = pd.DataFrame(manifest_rows)

repo_level_files = [
    "README.md",
    "publication_recommendations.md",
    "portable_colab_qsar_bundle/run_autoqsar_ga_benchmarks.py",
    "portable_colab_qsar_bundle/benchmark_results_summary.ipynb",
    "data/benchmark_dataset_catalog.csv",
    "data/benchmark_leaderboards/TDC_ADMET_Benchmark_Performance_by_Model.csv",
    "data/benchmark_leaderboards/ESOL_Lipophilicity_Current_Benchmarks_csv.csv",
]
repo_manifest_rows = []
for relative_path in repo_level_files:
    path = resolve_repo_file(relative_path)
    repo_manifest_rows.append({
        "relative_path": relative_path,
        "exists": bool(path.exists()),
        "bytes": int(path.stat().st_size) if path.exists() else 0,
        "sha256": sha256_file(path) if path.exists() else "",
    })
repo_reproducibility_manifest = pd.DataFrame(repo_manifest_rows)

repro_summary = pd.DataFrame([
    {
        "datasets_audited": int(len(reproducibility_manifest)),
        "datasets_with_metrics": int(reproducibility_manifest.get("has_metrics_csv", pd.Series(dtype=bool)).sum()) if not reproducibility_manifest.empty else 0,
        "datasets_with_predictions": int(reproducibility_manifest.get("has_predictions_csv", pd.Series(dtype=bool)).sum()) if not reproducibility_manifest.empty else 0,
        "datasets_with_selected_features": int(reproducibility_manifest.get("has_selected_features_csv", pd.Series(dtype=bool)).sum()) if not reproducibility_manifest.empty else 0,
        "datasets_with_split_hashes": int((reproducibility_manifest.get("split_train_hash", pd.Series(dtype=object)).astype(str).str.len() > 0).sum()) if not reproducibility_manifest.empty else 0,
        "median_required_artifact_fraction": float(reproducibility_manifest["required_dataset_artifact_fraction"].median()) if not reproducibility_manifest.empty else np.nan,
        "git_commit": git_value(["rev-parse", "HEAD"]),
        "git_is_dirty": bool(git_value(["status", "--porcelain"])),
        "run_config_present": bool((RUN_DIR / "run_config.json").exists()),
    }
])

write_publication_csv(reproducibility_manifest, "publication_reproducibility_manifest.csv")
write_publication_csv(repo_reproducibility_manifest, "publication_reproducibility_repo_files.csv")
write_publication_csv(repro_summary, "publication_reproducibility_summary.csv")
archive_file_list = reproducibility_manifest[["dataset", "dataset_dir"] + [f"has_{name.replace('.', '_')}" for name in required_dataset_files if f"has_{name.replace('.', '_')}" in reproducibility_manifest.columns]].copy() if not reproducibility_manifest.empty else pd.DataFrame()
write_publication_csv(archive_file_list, "publication_zenodo_archive_dataset_file_list.csv")

availability_statement = (
    "Benchmark artifacts include per-dataset metrics, predictions, selected features, selector coefficients, "
    "run configuration, split hashes, and checksum manifests. The generated publication_reproducibility_manifest.csv "
    "and publication_reproducibility_repo_files.csv files are intended as the Zenodo archive checklist."
)
(RUN_DIR / "publication_data_availability_statement.txt").write_text(availability_statement, encoding="utf-8")

if reproducibility_manifest.empty:
    display_note("No dataset directories were available for reproducibility auditing.")
else:
    display_note("### Publication Analysis: Reproducibility Audit and Archive Manifest")
    display(repro_summary)
    display(reproducibility_manifest.sort_values(["required_dataset_artifact_fraction", "dataset"], ascending=[True, True]).reset_index(drop=True))
    display_note("Repo-level files proposed for the archive")
    display(repo_reproducibility_manifest)


### Publication Analysis: Reproducibility Audit and Archive Manifest

,datasets_audited,datasets_with_metrics,datasets_with_predictions,datasets_with_selected_features,datasets_with_split_hashes,median_required_artifact_fraction,git_commit,git_is_dirty,run_config_present
0,45,44,44,44,44,1.0,b06baf5ef2ebdd2a4c6bc76824103dfbd35be861,True,True


,dataset,dataset_dir,git_commit,git_is_dirty,configured_random_seed,configured_chemprop_seed,configured_maplight_parity_seeds,split_strategy,split_train_hash,split_test_hash,n_train,n_test,has_metrics_csv,sha256_metrics_csv,bytes_metrics_csv,has_predictions_csv,sha256_predictions_csv,bytes_predictions_csv,has_selected_features_csv,sha256_selected_features_csv,bytes_selected_features_csv,has_selector_coefficients_csv,sha256_selector_coefficients_csv,bytes_selector_coefficients_csv,has_run_status_json,sha256_run_status_json,bytes_run_status_json,has_step_runtime_csv,sha256_step_runtime_csv,bytes_step_runtime_csv,required_dataset_artifacts_present,required_dataset_artifacts_total,required_dataset_artifact_fraction
0,tdc_herg_central,C:/Users/Scott.Coffin/OneDrive - California OE...,b06baf5ef2ebdd2a4c6bc76824103dfbd35be861,True,13,42,"1,2,3,4,5",,,,NaN,NaN,False,,0,False,,0,False,,0,False,,0,True,180607fda121b7a6ff290e5f6f2bd257c1bb31e0ba081a...,171,True,e669ea9361dc36cfab2f12639f6f1accf22e21c8a4e17c...,524,2,6,0.333333
1,chemml_cep_homo,C:/Users/Scott.Coffin/OneDrive - California OE...,b06baf5ef2ebdd2a4c6bc76824103dfbd35be861,True,13,42,"1,2,3,4,5",target_quartiles,7bca3443e9455134a9e500ae47d65ef8a9258d17fc844f...,2cf9245678766eb17714604f0f4651d32bd496fde0b205...,400.0,100.0,True,e04a6232c08c75528283bf7a88ed76e07ef94908dea820...,36106,True,a90dcc0367102ef25a58ef62f5baa3c7476db3bfdd07ae...,1328935,True,1a2325b2f5664359ec908c8c32f95626b2c7a07cc04e0f...,6025,True,0525eed288abdf77ff06ca78e5568958361d717ae6461e...,186011,True,64f2460642243538ba752c567ab89fe53d542d00b28cf1...,235,True,5bd48194623a1679af1a9793d23aa3d926ff8f6b926379...,4452,6,6,1.000000
2,chemml_organic_density,C:/Users/Scott.Coffin/OneDrive - California OE...,b06baf5ef2ebdd2a4c6bc76824103dfbd35be861,True,13,42,"1,2,3,4,5",target_quartiles,3a2203cf8e9934f0de0a2d6aca47be30df6b0507780a18...,e6718ea91d5848923d4910c905793d1e3e7d3ae27315e6...,400.0,100.0,True,c728980d4e62e0749c518cf766b9c787d3efeb22b4e2af...,36614,True,14b44b4d726142ae4e151c6acb200e5df9e839cd18a1c9...,1391952,True,90db7e9c311d72cc8f573bf54bc50695582e88d36e3105...,4467,True,da2c49f3482ad3e417d56aaa5028ccee714d9c183d7be5...,203256,True,825fe13239e0027ed53d796980c57579125daf55939656...,250,True,e076bb7b9be7aedab92803f2ee6838de1c876ac349c6ee...,4626,6,6,1.000000
3,esol_delaney,C:/Users/Scott.Coffin/OneDrive - California OE...,b06baf5ef2ebdd2a4c6bc76824103dfbd35be861,True,13,42,"1,2,3,4,5",scaffold,6f01da23b222c7d7cd5476ad9b5cb52d01f6d1626756e4...,9ee48128aa2c8388f7a9825139781be3d4fde955fb02d6...,874.0,254.0,True,4976bd39c50cc39eab183a8ba54c739812804fce8adabe...,47053,True,574a3066063aa5904ba4ed03049cb1f9ee02b5329b5611...,2087442,True,80670272430de20b26e31745207f17217486da85521d60...,9643,True,8469a3dd59e8144b6da780009da7bd370d560f920af556...,236693,True,d7c949f224c1573311a209460cdb4b6c6c29faf742e8d5...,246,True,c0e561fca906d8c612b91b7487e3bd0af8f7ab09e2bb47...,4411,6,6,1.000000
4,freesolv_sampl,C:/Users/Scott.Coffin/OneDrive - California OE...,b06baf5ef2ebdd2a4c6bc76824103dfbd35be861,True,13,42,"1,2,3,4,5",random,1a4d3541a62887d29772b7142a7571b65d0c668794fde3...,751e782db79be4730236c0e1232d599506645e8018bd2d...,513.0,129.0,True,2a0f93dfd48d7bb3eeae49e033b7eacf1cf206c2a4c907...,36964,True,275057d272c3f9bf8231c30f26ee99a5427c64bfb330cb...,1263870,True,a7c6c15f16e8fe3446bc6d585f9f9c3d84a4124ecd507d...,8411,True,f076f8a726cc0b455099eb2d355d19cc95f90c58c75566...,175529,True,828262081e423d0acb72f181a8aede5ba056accb8e9c08...,250,True,66092efd0a10ffef6320f7e9128d6e3f4fa399a0a36cf0...,4447,6,6,1.000000
5,lipophilicity,C:/Users/Scott.Coffin/OneDrive - California OE...,b06baf5ef2ebdd2a4c6bc76824103dfbd35be861,True,13,42,"1,2,3,4,5",scaffold,1c9600e79ad39c5f75209604a7171b2bd6a9c7fc733939...,26aaf8bb369bd22baa01dd6ed101857bdecaacd32b2d1b...,3357.0,843.0,True,89fb2824aaa7f8e29603de9e71a232d7a59c5ab968b218...,42759,True,6644890de41a431b0d0203dca1a7125937a884b331d972...,10306645,True,48687a44952790d70b15226a27f2cfcf19d9b45698642

Repo-level files proposed for the archive

,relative_path,exists,bytes,sha256
0,README.md,True,23482,6ef5e2f81b788c937ee84905faf151b18619e7b2229305...
1,publication_recommendations.md,True,23963,df0e7fd54208f05c6ff563b332a6cd8c59bd9f6f6de534...
2,portable_colab_qsar_bundle/run_autoqsar_ga_ben...,True,430846,9f904ca663e23b22781ca09afc98b0bcd793029c7d70c5...
3,portable_colab_qsar_bundle/benchmark_results_s...,True,1879987,54b44fbcb5ec2affadc89e29725a0de544795f193471f1...
4,data/benchmark_dataset_catalog.csv,True,150667,927ea321997d61382adfaef5c15b5ba032b73c00313f0a...
5,data/benchmark_leaderboards/TDC_ADMET_Benchmar...,True,28282,ac9a0ae65240e25434779711e2caca6be3d8aa8034b980...
6,data/benchmark_leaderboards/ESOL_Lipophilicity...,True,7320,332f21a3db5628aa50e4dd27e38324cb18ba781f710cf8...


In [71]:
# NO_RERUN_PUBLICATION_LEADERBOARD_REFERENCES
# Publication analysis 5: updated leaderboard-reference audit and regenerated comparison tables.


def load_current_physchem_reference(path="data/benchmark_leaderboards/ESOL_Lipophilicity_Current_Benchmarks_csv.csv"):
    path = resolve_repo_file(path)
    frame = read_optional_csv(path)
    if frame is None or frame.empty:
        return pd.DataFrame()
    required = {"Dataset", "Metric", "Direction", "Model", "Score_Mean"}
    if not required.issubset(set(frame.columns)):
        return pd.DataFrame()
    work = frame.copy()
    work["Score_Mean"] = pd.to_numeric(work["Score_Mean"], errors="coerce")
    work["Score_StdDev"] = pd.to_numeric(work.get("Score_StdDev", np.nan), errors="coerce")
    captured_at = pd.Timestamp(path.stat().st_mtime, unit="s").strftime("%Y-%m-%d") if path.exists() else ""
    rows = []
    for _, item in work.iterrows():
        score_mean = item.get("Score_Mean", np.nan)
        if pd.isna(score_mean):
            continue
        dataset_name = normalize_reference_dataset_name(item.get("Dataset"), "moleculenet")
        rank_value = pd.to_numeric(item.get("Rank", np.nan), errors="coerce")
        rows.append({
            "dataset": dataset_name,
            "dataset_source": f"Current MoleculeNet literature reference: {item.get('Dataset', '')}",
            "benchmark_suite": "moleculenet",
            "benchmark_id": clean_reference_text(item.get("Dataset")),
            "leaderboard_url": clean_reference_text(item.get("DOI_or_URL")),
            "leaderboard_dataset_split": clean_reference_text(item.get("Split")),
            "leaderboard_metric_name": clean_reference_text(item.get("Metric")),
            "leaderboard_metric_direction": clean_reference_text(item.get("Direction")),
            "rank": str(int(rank_value)) if pd.notna(rank_value) else "",
            "rank_numeric": float(rank_value) if pd.notna(rank_value) else np.nan,
            "model": clean_reference_text(item.get("Model")),
            "metric_value": format_reference_metric_value(score_mean, item.get("Score_StdDev", "") if pd.notna(item.get("Score_StdDev", np.nan)) else ""),
            "metric_value_numeric": float(score_mean),
            "is_top10_entry": True,
            "reference_source": "current_moleculenet_literature_reference",
            "captured_at": captured_at,
            "paper_short": clean_reference_text(item.get("Paper_Short")),
            "year": clean_reference_text(item.get("Year")),
            "reference": clean_reference_text(item.get("Reference")),
            "doi_or_url": clean_reference_text(item.get("DOI_or_URL")),
            "notes": clean_reference_text(item.get("Notes")),
            "manual_reference_source_file": str(path).replace("\\", "/"),
        })
    return pd.DataFrame(rows)


CURRENT_PHYSCHEM_REFERENCE = load_current_physchem_reference()
UPDATED_REFERENCE_ROWS = combine_leaderboard_reference_frames(
    LEADERBOARD_TOP10,
    MANUAL_LEADERBOARD_REFERENCE,
    CURRENT_PHYSCHEM_REFERENCE,
)

if UPDATED_REFERENCE_ROWS.empty:
    display_note("No reference rows were available for updated leaderboard comparison.")
else:
    UPDATED_REFERENCE_ROWS = UPDATED_REFERENCE_ROWS.copy()
    if "leaderboard_metric_name" in UPDATED_REFERENCE_ROWS.columns:
        UPDATED_REFERENCE_ROWS["metric_kind"] = UPDATED_REFERENCE_ROWS["leaderboard_metric_name"].astype(str).apply(leaderboard_metric_kind)
    UPDATED_REFERENCE_ROWS["metric_value_numeric"] = pd.to_numeric(UPDATED_REFERENCE_ROWS.get("metric_value_numeric", pd.Series(dtype=float)), errors="coerce")

    UPDATED_LEADERBOARD_COMPARISON = comparison_rows_from_reference(UPDATED_REFERENCE_ROWS)

    tdc_admet_group_ids = {
        "caco2_wang", "hia_hou", "pgp_broccatelli", "bioavailability_ma", "lipophilicity_astrazeneca",
        "solubility_aqsoldb", "bbb_martins", "ppbr_az", "vdss_lombardo", "cyp2d6_veith",
        "cyp3a4_veith", "cyp2c9_veith", "cyp2d6_substrate_carbonmangels",
        "cyp3a4_substrate_carbonmangels", "cyp2c9_substrate_carbonmangels", "half_life_obach",
        "clearance_microsome_az", "clearance_hepatocyte_az", "herg", "ames", "dili", "ld50_zhu",
    }
    ref_dataset_key = UPDATED_REFERENCE_ROWS.get("benchmark_id", pd.Series("", index=UPDATED_REFERENCE_ROWS.index)).astype(str).str.lower()
    ref_dataset_name = UPDATED_REFERENCE_ROWS.get("dataset", pd.Series("", index=UPDATED_REFERENCE_ROWS.index)).astype(str).str.lower().str.replace("^tdc_", "", regex=True)
    is_tdc22 = ref_dataset_key.isin(tdc_admet_group_ids) | ref_dataset_name.isin(tdc_admet_group_ids)
    has_maxqsaring = UPDATED_REFERENCE_ROWS.get("model", pd.Series("", index=UPDATED_REFERENCE_ROWS.index)).astype(str).str.contains("maxqsaring", case=False, na=False)
    maxq_datasets = sorted(set(ref_dataset_key.loc[is_tdc22 & has_maxqsaring].replace("", np.nan).dropna().tolist()) | set(ref_dataset_name.loc[is_tdc22 & has_maxqsaring].replace("", np.nan).dropna().tolist()))

    physchem_current_mask = UPDATED_REFERENCE_ROWS.get("dataset", pd.Series("", index=UPDATED_REFERENCE_ROWS.index)).astype(str).str.lower().isin({"esol_delaney", "lipophilicity"})
    physchem_current_rows = UPDATED_REFERENCE_ROWS.loc[physchem_current_mask].copy()

    reference_audit = pd.DataFrame([
        {
            "audit_item": "TDC-22 datasets with MaxQsaring reference rows",
            "count": len(maxq_datasets),
            "expected": len(tdc_admet_group_ids),
            "status": "complete" if len(maxq_datasets) >= len(tdc_admet_group_ids) else "partial",
            "details": "; ".join(maxq_datasets),
        },
        {
            "audit_item": "Current ESOL/Lipophilicity literature reference rows",
            "count": int(len(physchem_current_rows)),
            "expected": 2,
            "status": "complete" if physchem_current_rows["dataset"].nunique() >= 2 else "partial",
            "details": "; ".join(sorted(physchem_current_rows.get("model", pd.Series(dtype=object)).dropna().astype(str).unique().tolist())[:20]),
        },
        {
            "audit_item": "Updated reference rows total",
            "count": int(len(UPDATED_REFERENCE_ROWS)),
            "expected": np.nan,
            "status": "available",
            "details": f"datasets={UPDATED_REFERENCE_ROWS['dataset'].nunique() if 'dataset' in UPDATED_REFERENCE_ROWS.columns else 0}",
        },
        {
            "audit_item": "Updated comparison rows generated",
            "count": int(len(UPDATED_LEADERBOARD_COMPARISON)) if UPDATED_LEADERBOARD_COMPARISON is not None else 0,
            "expected": np.nan,
            "status": "available" if UPDATED_LEADERBOARD_COMPARISON is not None and not UPDATED_LEADERBOARD_COMPARISON.empty else "missing",
            "details": f"datasets={UPDATED_LEADERBOARD_COMPARISON['dataset'].nunique() if UPDATED_LEADERBOARD_COMPARISON is not None and not UPDATED_LEADERBOARD_COMPARISON.empty and 'dataset' in UPDATED_LEADERBOARD_COMPARISON.columns else 0}",
        },
    ])

    write_publication_csv(UPDATED_REFERENCE_ROWS, "publication_leaderboard_top10_reference.csv")
    write_publication_csv(UPDATED_LEADERBOARD_COMPARISON, "publication_leaderboard_comparison_by_dataset.csv")
    write_publication_csv(reference_audit, "publication_leaderboard_reference_audit.csv")

    display_note("### Publication Analysis: Updated Leaderboard References")
    display(reference_audit)
    if not UPDATED_LEADERBOARD_COMPARISON.empty:
        display(UPDATED_LEADERBOARD_COMPARISON.sort_values(["dataset", "estimated_rank_vs_top10"]).reset_index(drop=True))

        rank_counts = (
            UPDATED_LEADERBOARD_COMPARISON.assign(in_top10_estimate=UPDATED_LEADERBOARD_COMPARISON.get("in_top10_estimate", False).astype(bool))
            .groupby("in_top10_estimate", dropna=False)["dataset"]
            .nunique()
            .reset_index(name="datasets")
        )
        fig = px.bar(
            rank_counts,
            x="in_top10_estimate",
            y="datasets",
            title="AutoQSAR top-10 status using updated local reference tables",
        )
        fig.update_layout(xaxis_title="Estimated in top 10", yaxis_title="Datasets")
        fig.show()


### Publication Analysis: Updated Leaderboard References

,audit_item,count,expected,status,details
0,TDC-22 datasets with MaxQsaring reference rows,22,22.0,complete,ames; bbb_martins; bioavailability_ma; caco2_w...
1,Current ESOL/Lipophilicity literature referenc...,73,2.0,complete,3D INFOMAX; 3D PGT; AutoML; AutoML (xLSTM pape...
2,Updated reference rows total,407,NaN,available,datasets=37
3,Updated comparison rows generated,37,NaN,available,datasets=37


,dataset,benchmark_id,leaderboard_metric_name,metric_kind,direction,best_model,best_workflow,best_value,best_value_raw_model_metric,best_value_metric_column,leaderboard_top1,leaderboard_top10_cutoff,known_reference_count,top10_reference_count,reference_sources,reference_models,estimated_rank_vs_top10,in_top10_estimate,gap_to_top1,gap_to_top10_cutoff
0,esol_delaney,esol_delaney,RMSE,rmse,lower,TabPFNRegressor,TabPFN deep learning,0.621439,0.621439,test_rmse,0.558,0.615,38,10,current_moleculenet_literature_reference; lite...,PrismNet; HiGNN; DMPNN; MGPhyche; DeeperGCN; A...,11,False,0.063439,0.006439
1,lipophilicity,lipophilicity,RMSE,rmse,lower,Ensemble (Weighted average (inverse train RMSE)),ensemble,0.594788,0.594788,test_rmse,0.549,0.587,35,10,current_moleculenet_literature_reference; lite...,PrismNet; GRAPHMSL; MV-Mol; InstructMol; GEM +...,11,False,0.045788,0.007788
2,poduam_pod_nc_std,poduam_pod_nc_std,RMSE,rmse,lower,CFA (Combinatorial Fusion),CFA fusion,0.699850,0.699850,test_rmse,0.550,0.730,2,2,literature_static,PODUAM BNN (PODnc); PODUAM BNN (PODnc),2,True,0.149850,-0.030150
3,poduam_pod_rd_std,poduam_pod_rd_std,RMSE,rmse,lower,XGBoost,conventional,0.554603,0.554603,test_rmse,0.410,0.630,2,2,literature_static,PODUAM BNN (PODrd); PODUAM BNN (PODrd),2,True,0.144603,-0.075397
4,polaris_adme_fang_hppb_1,polaris/adme-fang-hppb-1,mean_squared_error,mse,lower,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,0.199498,0.446652,test_rmse,0.143,0.383,10,10,polaris,1B_MPNN_LargeMix-and-Phenomics; 1B_MPNN_MolGPS...,3,True,0.056498,-0.183502
5,polaris_adme_fang_perm_1,polaris/adme-fang-perm-1,mean_squared_error,mse,lower,XGBoost,conventional,0.185158,0.430300,test_rmse,0.113,0.257,10,10,polaris,1B_MPNN_MolGPS-ens_LargeMix; 1B_MPNN_LargeMix-...,6,True,0.072158,-0.071842
6,polaris_adme_fang_rclint_1,polaris/adme-fang-rclint-1,mean_squared_error,mse,lower,CatBoost,conventional,0.285363,0.534193,test_rmse,0.216,0.403,10,10,polaris,1B_MPNN_MolGPS-ens_LargeMix; 1B_MPNN_LargeMix-...,4,True,0.069363,-0.117637
7,polaris_adme_fang_rppb_1,polaris/adme-fang-rppb-1,mean_squared_error,mse,lower,"MapLight + GNN (CatBoost, Strict Parity)",MapLight + GNN,0.243599,0.493558,test_rmse,0.230,0.634,10,10,polaris,1B_MPNN_LargeMix-and-Phenomics; 1B_MPNN_MolGPS...,3,True,0.013599,-0.390401
8,polaris_adme_fang_solu_1,polaris/adme-fang-solu-1,mean_squared_error,mse,lower,XGBoost,conventional,0.343380,0.585986,test_rmse,0.222,0.329,10,10,polaris,1B_MPNN_MolGPS-ens_LargeMix; 1B_MPNN_LargeMix-...,11,False,0.121380,0.014380
9,tdc_ames,ames,AUROC,roc_auc,higher,"Ensemble (OOF Stacking (RidgeCV, 5-fold))",ensemble,0.876017,0.876017,test_roc_auc,0.912,0.834,7,7,manual_tdc_admet_reference,QW-MTL; Meta-model (NIST); ADMETboost (XGBoost...,2,True,0.035983,-0.042017
